> The monitoring data and building-specific configuration are confidential and are not included. See `README.md` for execution requirements. Clear any generated outputs before sharing an executed notebook.


In [ ]:
# Private workspace and configuration
from copy import deepcopy as _privacy_deepcopy
from functools import lru_cache as _privacy_lru_cache
import json
import os
from pathlib import Path


def _private_config_path(variable):
    value = os.environ.get(variable)
    if not value:
        raise RuntimeError(f"Set {variable} before running this notebook; see README.md.")
    return Path(value).expanduser().resolve()


def private_workspace():
    """Return the configured private input/output directory."""
    path = _private_config_path("HVAC_PRIVATE_WORKSPACE")
    if not path.is_dir():
        raise FileNotFoundError("The configured private workspace does not exist.")
    return path


def _decode_private_values(value):
    if isinstance(value, list):
        return [_decode_private_values(item) for item in value]
    if isinstance(value, dict):
        if set(value) == {"__tuple_key_dict__"}:
            return {tuple(key): _decode_private_values(item)
                    for key, item in value["__tuple_key_dict__"]}
        return {key: _decode_private_values(item) for key, item in value.items()}
    return value


@_privacy_lru_cache(maxsize=1)
def _load_private_values():
    with _private_config_path("HVAC_PRIVATE_SETTINGS").open(encoding="utf-8") as handle:
        values = json.load(handle).get("private_values")
    if not isinstance(values, dict):
        raise ValueError("Private settings must contain a private_values object.")
    return _decode_private_values(values)


def private_value(key):
    """Read a private setting without embedding its value in the public source."""
    values = _load_private_values()
    if key not in values or values[key] is None:
        raise KeyError(f"Required private setting is missing: {key!r}.")
    return _privacy_deepcopy(values[key])


# Notebook 01: Event detection and dataset preparation

This notebook prepares heating and cooling recovery events, temperature-forecast targets, and remaining-duration targets from private monitoring data. The morning-recovery criteria follow the reference study by Dadras Javan and colleagues, with sign-reversed criteria for heating. No external reference dataset is loaded or compared.

The workflow cleans and aligns observations, detects event boundaries, builds five-minute feature trajectories, and creates a chronological event split. Subsequent notebooks reuse the saved split and its SHA-256 fingerprint.

Stored tables use `phase` for operating mode. A duration **anchor** is a prediction time identified by `anchor_key`; it is distinct from the future temperature targets associated with that time.


## Observation times and feature times

Event boundaries are detected on cleaned observations at their original timestamps. Model features and temperature targets use a reconstructed five-minute grid. With `CAUSAL_GRID_FILL_FOR_INPUTS=False`, continuous signals can use later observations during interpolation. Backward derivative stencils on that grid therefore do not establish real-time input availability.

Remaining-duration rows are generated directly from events and current feature rows. They do not require a valid future temperature target, which preserves duration predictions for short events. The trajectory-regularity score `q` summarizes directional progress from a prediction time to the observed endpoint; it is training information, not a predictor.


In [ ]:

# Mount Drive, imports, paths, global settings

from google.colab import drive
drive.mount('/content/drive')

import os
import re
import json
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

# Private workspace

BASE_DIR = str(private_workspace())

# Raw input files

# All input files are resolved exclusively from the private raw-data directory.
RAW_DATA_DIR = os.path.join(BASE_DIR, 'Datasets')

EXACT_INPUT_CANDIDATES = {
    private_value('inputs.literal_002'): [os.path.join(RAW_DATA_DIR, private_value('inputs.literal_002'))],
    private_value('inputs.literal_003'): [os.path.join(RAW_DATA_DIR, private_value('inputs.literal_003'))],
    private_value('inputs.literal_004'): [os.path.join(RAW_DATA_DIR, private_value('inputs.literal_004'))],
    private_value('inputs.literal_005'): [os.path.join(RAW_DATA_DIR, private_value('inputs.literal_005'))],
}

def _choose_existing_exact_file(name, required):
    candidates = EXACT_INPUT_CANDIDATES[name]
    for p in candidates:
        if os.path.exists(p):
            return p
    if required:
        msg = [f'Required input file not found: {name}', 'Checked exact known paths:']
        msg += [f'  - {p}' for p in candidates]
        msg += [
            '',
            'This is not a notebook logic issue: the file is absent from the known raw-data locations.',
            'Put the raw CSV in one of the checked paths or update EXACT_INPUT_CANDIDATES in Cell 1.'
        ]
        raise FileNotFoundError('\n'.join(msg))
    return candidates[0]

Dataset_DIR = RAW_DATA_DIR
AHU_FILE    = _choose_existing_exact_file(private_value('inputs.literal_002'), required=True)
CAL_FILE    = _choose_existing_exact_file(private_value('inputs.literal_003'), required=True)
VALVE_FILE  = _choose_existing_exact_file(private_value('inputs.literal_004'), required=False)
LOCALI_FILE = _choose_existing_exact_file(private_value('inputs.literal_005'), required=True)

COOLING_TARGET_FILE = os.path.join(BASE_DIR, 'cooling_room_targets.csv')

OUT_DIR = os.path.join(BASE_DIR, 'full_rampup_5min_feature_dataset_UNIFIED_DERIVATIVE_DURATION_AWARE')

# Build duration prediction rows from event boundaries and current features.
# Future temperature-target availability does not restrict duration coverage.
SAVE_REMAINING_DURATION_ANCHORS = True
REQUIRE_DURATION_ANCHOR_FOR_EACH_EVENT = True
ALWAYS_ADD_EVENT_START_ASOF_ANCHOR = True
ADD_INSIDE_GRID_DURATION_ANCHORS = True
DURATION_ANCHOR_INCLUDE_EVENT_END_GRID_ROW = False
DURATION_ANCHOR_MAX_ASOF_GAP_MIN = 5.0

print('\nResolved exact input files:')
for _label, _path in [
    (private_value('inputs.literal_002'), AHU_FILE),
    (private_value('inputs.literal_003'), CAL_FILE),
    (private_value('inputs.literal_004'), VALVE_FILE),
    (private_value('inputs.literal_005'), LOCALI_FILE),
]:
    print(f'  {_label}: {_path} | exists={os.path.exists(_path)}')

# Remove existing extraction outputs when a clean run is requested.
CLEAN_OUTPUT_DIR_BEFORE_RUN = True
if CLEAN_OUTPUT_DIR_BEFORE_RUN and os.path.exists(OUT_DIR):
    shutil.rmtree(OUT_DIR)
os.makedirs(OUT_DIR, exist_ok=True)

# Selected sensor prefixes only

SELECTED_ROOM_PREFIXES = private_value('nb01.selected_room_prefixes')

# Phases to run

PHASES_TO_RUN = ['heating', 'cooling']

# Prepare original room observations using UTC-to-local time conversion, first-record
# deduplication, invalid-reading replacement, forward fill and rounding.
# Event boundaries use these observations; model features use the reconstructed grid.
DATA_PREPARATION_MODE = 'localiin_dadras_style_only_for_event_detection'
# Event detection uses LocaliIN observation-based rows; feature extraction uses the
# regular feature grid.
COOLING_USE_LOCALIIN_DADRAS_STYLE_FOR_EVENT_DETECTION = True
HEATING_USE_LOCALIIN_DADRAS_STYLE_FOR_EVENT_DETECTION = True
COOLING_USE_OUR_FEATURE_GRID_FOR_SAMPLES = True
HEATING_USE_OUR_FEATURE_GRID_FOR_SAMPLES = True

# Apply the morning-recovery criteria and configured time windows.
# Cooling requires a downward response toward the target; heating uses the corresponding
# upward response.
# Duration labels are rounded differences between observed event boundaries.
COOLING_EVENT_EXTRACTION_MODE = 'dadras_exact'
HEATING_EVENT_EXTRACTION_MODE = 'dadras_symmetric_exact'

# Extraction settings

GRID_FREQ = '5min'
STEP_MIN = 5

# Extend future temperature targets after the event only while consecutive changes
# retain the expected operating-mode sign.
POST_END_EXPORT_MODE = 'trend_sign_until_first_break'
POST_END_BUFFER_MIN = 0  # legacy compatibility only; fixed buffer is disabled in this version
POST_END_SLOPE_TOL_C_PER_STEP = 0.0
POST_END_MAX_CONTINUATION_MIN = None  # set to a number to cap continuation; None means stop only when trend sign breaks

MIN_RAMPUP_MINUTES = 5
MAX_HEATING_RAMPUP_MINUTES = 180
MAX_COOLING_RAMPUP_MINUTES = 60
# The duration ceiling defines the retained analysis cohort.

# Enforce one saved operational ramp-up event per room per calendar day.
ONE_EVENT_PER_ROOM_DAY = True

# Exact grid lags. Targets are generated dynamically, not with fixed 5--60 min only.
ROOM_LAGS_MINUTES = list(range(5, 65, 5))
BOILER_LAGS_MINUTES = list(range(5, 30, 5))
COOLING_SUPPLY_LAGS_MINUTES = list(range(5, 30, 5))

# Use central derivatives at past grid centers and backward estimates at the current
# grid time.
# These stencils are past-facing relative to the reconstructed grid.
DERIVATIVE_HISTORY_LAGS_MINUTES = list(range(5, 65, 5))
SOURCE_DERIVATIVE_HISTORY_LAGS_MINUTES = list(range(5, 30, 5))
END_BAND_C = 0.4
REMAINING_SCORE_EPS = 1e-12
NEW_FRAMEWORK_NAME = 'unified_heating_cooling_derivative_rate_duration_aware'

ADD_EXTERNAL_TEMP_LAGS = False
EXT_LAGS_MINUTES = list(range(5, 65, 5))

ADD_SOLAR_RADIATION_FEATURES_IF_AVAILABLE = False
SOLAR_LAGS_MINUTES = [5, 10, 15, 20, 25]

BUILDING_LATITUDE_DEG = private_value('nb01.building_latitude_deg')
BUILDING_LONGITUDE_DEG = private_value('nb01.building_longitude_deg')

# Match preceding source observations before reconstructing the regular grid.
TIME_TOLERANCE = '3min'
MERGE_DIRECTION_FOR_INPUTS = 'backward'

# Reconstruct the five-minute grid offline. Continuous interpolation can use later
# observations, so real-time input availability requires separate evaluation.
CAUSAL_GRID_FILL_FOR_INPUTS = False

DROP_ONLY_REQUIRED_NA = True

# Setpoint and offset fields are diagnostic outputs, not required predictors.
# Their absence must not remove otherwise valid temperature-target rows.
REQUIRE_SELECTED_ROOMS_WITH_EVENTS_IN_LONG = True
REQUIRE_ALL_SELECTED_ROOMS_IN_LONG = False  # require only rooms that actually have detected events
USE_POSITIONAL_GRID_TARGET_LOOKUP_FOR_LONG_SAMPLES = True
SAVE_FULL_COVERAGE_AUDITS = True

# Replace invalid sentinel readings before synchronization and interpolation to avoid
# propagating invalid values.
BAD_SENTINEL_THRESHOLD = -9999.0
SENTINEL_CLEANING_AUDIT = []
ALL_MISSING_FEATURE_AUDIT = []
SOURCE_TIME_ALIGNMENT_AUDIT = []
HEATING_DETECTION_GRID_AUDIT = []

# Export optional source features only when enabled and supported by non-missing
# observations.
EXPORT_OPTIONAL_COOLING_SOURCE_FEATURES = False
EXPORT_OPTIONAL_SOLAR_WEATHER_FEATURES = False

# Detect both modes on cleaned original observations, using fixed heating targets.
# Use the regular grid for feature and target construction.
HEATING_EVENT_DETECTION_ON_FINAL_FEATURE_GRID = False
SAVE_HEATING_DETECTION_FRAME_VS_FEATURE_GRID_AUDIT = False

# Interpolate continuous temperature signals and forward-fill control, setpoint and
# status signals.
SIGNAL_AWARE_GRID_FILL = True

# Paper-defined heating event settings

HEATING_START_BELOW_SETPOINT = 0.5
HEATING_END_BELOW_SETPOINT   = 0.4
# Use fixed room-specific heating targets for event detection; recorded setpoints remain
# diagnostic.
HEATING_USE_ROOM_SPECIFIC_TARGET_FOR_DETECTION = True
HEATING_DYNAMIC_SAME_TIMESTAMP_SETPOINT = False
HEATING_SETPOINT_CHANGE_IS_DIAGNOSTIC_ONLY = True  # actual Room_S(t) remains a diagnostic/feature only

HEATING_ROOM_TARGET_C = private_value('nb01.heating_room_target_c')

missing_heating_targets_for_selected = sorted(set(SELECTED_ROOM_PREFIXES) - set(HEATING_ROOM_TARGET_C))
if missing_heating_targets_for_selected:
    raise ValueError(f'Missing heating target for selected rooms: {missing_heating_targets_for_selected}')
HEATING_MIN_SUPPLY_TEMP      = 40.0
HEATING_MIN_ROOM_FAN_SIGNAL  = 0.4
HEATING_MORNING_START        = '06:00'
HEATING_MORNING_END          = '10:00'
HEATING_LATEST_START_HOUR    = 10

# Optional noise-adjusted start-confirmation settings; disabled by the active extraction
# mode.
USE_ROOM_SPECIFIC_HEATING_START_CONFIRMATION = False  # not used in symmetric observation-based heating
HEATING_LOCAL_SLOPE_LOOKBACK_MIN = 5
HEATING_LOCAL_SLOPE_MIN_RISE_FLOOR_C = 0.03
HEATING_LOCAL_SLOPE_NOISE_SIGMA_MULTIPLIER = 1.0
HEATING_FUTURE_CONFIRMATION_LOOKAHEAD_MIN = 20
HEATING_FUTURE_CONFIRMATION_MIN_RISE_FLOOR_C = 0.15
HEATING_FUTURE_CONFIRMATION_NOISE_SIGMA_MULTIPLIER = 2.0
HEATING_CONFIRMATION_MIN_STABLE_SAMPLES = 30

# Paper-defined cooling event settings

TARGET_MODE = 'manual_constant'   # options: manual_constant, manual_file, room_setpoint, auto
FIXED_COOLING_TARGET_C = private_value('nb01.fixed_cooling_target_c')     # used only for retrospective cooling event labelling
COOLING_START_ABOVE_TARGET = 0.5
COOLING_END_ABOVE_TARGET   = 0.4
COOLING_MIN_ROOM_FAN_SIGNAL = 0.4
MIN_COOLING_DELTA_T = 0.5
MAX_REASONABLE_COOLING_SUPPLY_TEMP = 26.0
COOLING_MORNING_START = '05:00'
COOLING_MORNING_END   = '09:30'
# Apply the configured strict latest-start clock limit.
COOLING_LATEST_START_TIME = '08:00'
COOLING_LATEST_START_HOUR = None  # deprecated; kept only for backward compatibility
COOLING_START_STRICTLY_BEFORE_LATEST = True  # Reference-rule: idx.hthe < 8, so 08:00 is excluded
COOLING_END_SEARCH_END_TIME = '09:30'       # Reference-rule end-search limit
USE_COOLING_SEASON_DATE_FILTER = True
COOLING_SEASON_START = private_value('nb01.cooling_season_start')
COOLING_SEASON_END   = private_value('nb01.cooling_season_end')
USE_COOLING_SEASON_MONTH_FILTER = True
COOLING_MONTHS = [5, 6, 7, 8, 9]

# Retain optional start-confirmation parameters for compatibility; the active extraction
# mode does not use them.
USE_ROOM_SPECIFIC_COOLING_START_CONFIRMATION = False
COOLING_LOCAL_SLOPE_LOOKBACK_MIN = 5
COOLING_LOCAL_SLOPE_MIN_DROP_FLOOR_C = 0.03
COOLING_LOCAL_SLOPE_NOISE_SIGMA_MULTIPLIER = 1.0
COOLING_FUTURE_CONFIRMATION_LOOKAHEAD_MIN = 20
COOLING_FUTURE_CONFIRMATION_MIN_DROP_FLOOR_C = 0.15
COOLING_FUTURE_CONFIRMATION_NOISE_SIGMA_MULTIPLIER = 2.0
COOLING_CONFIRMATION_MIN_STABLE_SAMPLES = 30

# Backward-compatible aliases used by older diagnostic code, if any.
COOLING_CONFIRMATION_LOOKAHEAD_MIN = COOLING_FUTURE_CONFIRMATION_LOOKAHEAD_MIN
COOLING_CONFIRMATION_MIN_DROP_FLOOR_C = COOLING_FUTURE_CONFIRMATION_MIN_DROP_FLOOR_C
COOLING_CONFIRMATION_NOISE_SIGMA_MULTIPLIER = COOLING_FUTURE_CONFIRMATION_NOISE_SIGMA_MULTIPLIER

print('BASE_DIR:', BASE_DIR)
print('OUT_DIR:', OUT_DIR)
print('Counterfactual/scoring disabled: using observed extracted durations only')
print('Cooling event extraction mode:', COOLING_EVENT_EXTRACTION_MODE)
print('Heating event extraction mode:', HEATING_EVENT_EXTRACTION_MODE)
print('Selected prefixes:', SELECTED_ROOM_PREFIXES)
print('Phases:', PHASES_TO_RUN)
print('Grid:', GRID_FREQ, '| post-end export mode:', POST_END_EXPORT_MODE, '| fixed post-end trend-continuation disabled')
print('Clean output before run:', CLEAN_OUTPUT_DIR_BEFORE_RUN)


In [ ]:

# General helper functions

def read_csv_safe(path):
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()
    return df

# Sentinel / impossible-value cleaning helpers

def clean_physical_sentinels(df, label, threshold=None, protected_cols=None, verbose=True):
    """Replace invalid numeric sentinel readings with NaN and record the affected
    counts before interpolation.
    """
    if threshold is None:
        threshold = globals().get('BAD_SENTINEL_THRESHOLD', -9999.0)
    if protected_cols is None:
        protected_cols = {
            'timestamp', 'room', 'room_label', 'phase', 'prefixes', 'selected_prefixes',
            'event_detection_source', 'feature_extraction_source', 'duration_definition',
            'target_context', 'horizon_support_category',
        }

    out = df.copy()
    audit_rows = []

    for c in out.columns:
        if c in protected_cols:
            continue

        s = pd.to_numeric(out[c], errors='coerce')
        # Only act on columns where numeric conversion is meaningful.
        if s.notna().sum() == 0:
            continue

        bad = s <= float(threshold)
        if bad.any():
            audit_rows.append({
                'label': label,
                'column': c,
                'n_bad_values_replaced': int(bad.sum()),
                'threshold': float(threshold),
                'min_bad_value': float(s[bad].min()),
                'max_bad_value': float(s[bad].max()),
                'n_rows': int(len(out)),
            })
            out.loc[bad, c] = np.nan

    if audit_rows:
        if 'SENTINEL_CLEANING_AUDIT' not in globals():
            globals()['SENTINEL_CLEANING_AUDIT'] = []
        globals()['SENTINEL_CLEANING_AUDIT'].extend(audit_rows)
        if verbose:
            print(f'[sentinel-clean] {label}: replaced {sum(r["n_bad_values_replaced"] for r in audit_rows)} values in {len(audit_rows)} columns.')

    return out

def assert_no_bad_sentinels(df, label, threshold=None):
    """Raise an error if invalid sentinel readings remain in the exported data."""
    if threshold is None:
        threshold = globals().get('BAD_SENTINEL_THRESHOLD', -9999.0)
    if df is None or len(df) == 0:
        return

    bad_rows = []
    for c in df.columns:
        if c == 'timestamp':
            continue
        s = pd.to_numeric(df[c], errors='coerce')
        if s.notna().sum() == 0:
            continue
        bad = s <= float(threshold)
        if bad.any():
            bad_rows.append({
                'label': label,
                'column': c,
                'n_bad_values': int(bad.sum()),
                'min_bad_value': float(s[bad].min()),
            })

    if bad_rows:
        display(pd.DataFrame(bad_rows))
        raise RuntimeError(f'{label}: sentinel-like values remain in exported data. Fix raw-source cleaning before saving.')

def audit_all_missing_columns(df, label, candidate_cols=None):
    """Record columns that are present but contain no non-missing values."""
    if df is None or len(df) == 0:
        return pd.DataFrame()
    cols = list(candidate_cols) if candidate_cols is not None else list(df.columns)
    rows = []
    for c in cols:
        if c in df.columns and df[c].isna().all():
            rows.append({'label': label, 'column': c, 'n_rows': int(len(df)), 'issue': 'all_missing'})
    if rows:
        if 'ALL_MISSING_FEATURE_AUDIT' not in globals():
            globals()['ALL_MISSING_FEATURE_AUDIT'] = []
        globals()['ALL_MISSING_FEATURE_AUDIT'].extend(rows)
    return pd.DataFrame(rows)

def save_data_cleaning_audits(out_dir):
    audit_dir = os.path.join(out_dir, 'data_cleaning_audits')
    os.makedirs(audit_dir, exist_ok=True)

    sentinel_df = pd.DataFrame(globals().get('SENTINEL_CLEANING_AUDIT', []))
    sentinel_path = os.path.join(audit_dir, 'sentinel_cleaning_audit.csv')
    sentinel_df.to_csv(sentinel_path, index=False)

    all_missing_df = pd.DataFrame(globals().get('ALL_MISSING_FEATURE_AUDIT', []))
    all_missing_path = os.path.join(audit_dir, 'all_missing_feature_audit.csv')
    all_missing_df.to_csv(all_missing_path, index=False)

    source_time_df = pd.DataFrame(globals().get('SOURCE_TIME_ALIGNMENT_AUDIT', []))
    source_time_path = os.path.join(audit_dir, 'source_time_alignment_audit.csv')
    source_time_df.to_csv(source_time_path, index=False)

    heating_detection_grid_df = pd.DataFrame(globals().get('HEATING_DETECTION_GRID_AUDIT', []))
    heating_detection_grid_path = os.path.join(audit_dir, 'heating_detection_frame_vs_feature_grid_audit.csv')
    heating_detection_grid_df.to_csv(heating_detection_grid_path, index=False)

    print('Saved sentinel cleaning audit:', sentinel_path, '| rows:', len(sentinel_df))
    print('Saved all-missing feature audit:', all_missing_path, '| rows:', len(all_missing_df))
    print('Saved source time alignment audit:', source_time_path, '| rows:', len(source_time_df))
    print('Saved heating detection-frame vs feature-grid audit:', heating_detection_grid_path, '| rows:', len(heating_detection_grid_df))
    return {
        'sentinel_cleaning_audit': sentinel_path,
        'all_missing_feature_audit': all_missing_path,
        'source_time_alignment_audit': source_time_path,
        'heating_detection_frame_vs_feature_grid_audit': heating_detection_grid_path,
    }

def feature_grid_value_at_time(df, value_col, target_time, time_col='timestamp'):
    target_time = pd.to_datetime(target_time, errors='coerce')
    if pd.isna(target_time) or df is None or len(df) == 0 or value_col not in df.columns:
        return np.nan
    tmp = df[[time_col, value_col]].copy()
    tmp[time_col] = pd.to_datetime(tmp[time_col], errors='coerce')
    tmp[value_col] = pd.to_numeric(tmp[value_col], errors='coerce')
    tmp = tmp.dropna(subset=[time_col, value_col]).drop_duplicates(subset=[time_col]).sort_values(time_col)
    if tmp.empty:
        return np.nan
    s = tmp.set_index(time_col)[value_col].sort_index()
    new_index = s.index.union(pd.DatetimeIndex([target_time]))
    s2 = s.reindex(new_index).sort_index().interpolate(method='time')
    return float(s2.loc[target_time]) if target_time in s2.index and pd.notna(s2.loc[target_time]) else np.nan

def audit_heating_detection_frame_vs_feature_grid(preliminary_events, final_grid_events, feature_grid, room_id, room_label=None):
    """Compare observation-based heating candidates with feature-grid conditions for
    diagnostic purposes. This optional audit does not define event boundaries.
    """
    rows = []
    final_dates = set()
    for ev in final_grid_events:
        try:
            final_dates.add(pd.to_datetime(ev.get('start_time')).date())
        except Exception:
            pass

    for ev in preliminary_events:
        start = pd.to_datetime(ev.get('start_time'), errors='coerce')
        end = pd.to_datetime(ev.get('end_time'), errors='coerce')
        event_date = pd.to_datetime(start).date() if pd.notna(start) else ev.get('event_date', None)

        T_start_g = feature_grid_value_at_time(feature_grid, 'Room_T', start)
        S_start_g = feature_grid_value_at_time(feature_grid, 'Heating_Target_for_detection', start)
        T_end_g = feature_grid_value_at_time(feature_grid, 'Room_T', end)
        S_end_g = feature_grid_value_at_time(feature_grid, 'Heating_Target_for_detection', end)

        start_thr_g = S_start_g - HEATING_START_BELOW_SETPOINT if pd.notna(S_start_g) else np.nan
        end_thr_g = S_end_g - HEATING_END_BELOW_SETPOINT if pd.notna(S_end_g) else np.nan
        start_valid_g = pd.notna(T_start_g) and pd.notna(start_thr_g) and T_start_g < start_thr_g
        end_valid_g = pd.notna(T_end_g) and pd.notna(end_thr_g) and T_end_g >= end_thr_g

        rows.append({
            'phase': 'heating',
            'room': room_id,
            'room_label': room_label,
            'preliminary_event_id_detection_frame': ev.get('event_id', np.nan),
            'event_date': event_date,
            'preliminary_start_time': start,
            'preliminary_end_time': end,
            'preliminary_duration_min': ev.get('duration_min', np.nan),

            'detection_frame_T_start': ev.get('room_T_at_start', np.nan),
            'detection_frame_heating_target_start': ev.get('target_at_start', np.nan),
            'detection_frame_T_end': ev.get('room_T_at_end', np.nan),
            'detection_frame_heating_target_end': ev.get('target_at_end', np.nan),

            'feature_grid_T_start_at_prelim_start': T_start_g,
            'feature_grid_heating_target_start_at_prelim_start': S_start_g,
            'feature_grid_start_threshold_at_prelim_start_C': start_thr_g,
            'feature_grid_start_valid_at_prelim_start': bool(start_valid_g),

            'feature_grid_T_end_at_prelim_end': T_end_g,
            'feature_grid_heating_target_end_at_prelim_end': S_end_g,
            'feature_grid_end_threshold_at_prelim_end_C': end_thr_g,
            'feature_grid_end_margin_at_prelim_end_C': (T_end_g - end_thr_g) if pd.notna(T_end_g) and pd.notna(end_thr_g) else np.nan,
            'feature_grid_end_valid_at_prelim_end': bool(end_valid_g),

            'has_final_grid_event_same_date': event_date in final_dates,
            'root_cause_if_not_kept': (
                'feature_grid_end_below_fixed_heating_target_threshold'
                if not bool(end_valid_g) else
                'feature_grid_start_not_below_start_threshold'
                if not bool(start_valid_g) else
                'kept_or_removed_by_other_filter'
            ),
        })

    out = pd.DataFrame(rows)
    if 'HEATING_DETECTION_GRID_AUDIT' not in globals():
        globals()['HEATING_DETECTION_GRID_AUDIT'] = []
    if len(out):
        globals()['HEATING_DETECTION_GRID_AUDIT'].extend(out.to_dict(orient='records'))
    return out

def parse_time(df, file_label='file'):
    df = df.copy()
    if 'timestamp' not in df.columns:
        raise ValueError(f'{file_label} has no timestamp column.')

    raw = df['timestamp'].astype(str).str.strip()
    try:
        parsed = pd.to_datetime(raw, errors='coerce', format='mixed', dayfirst=False)
    except TypeError:
        parsed = pd.to_datetime(raw, errors='coerce', dayfirst=False, infer_datetime_format=True)

    failed = parsed.isna()
    if failed.any():
        fallback = pd.to_datetime(raw[failed], errors='coerce', dayfirst=True, infer_datetime_format=True)
        parsed.loc[failed] = fallback

    df['timestamp'] = parsed

    print('\n' + '=' * 80)
    print(file_label)
    print('Rows before timestamp parsing:', len(df))
    print('Parsed timestamps:', df['timestamp'].notna().sum())
    print('Failed timestamps:', df['timestamp'].isna().sum())

    if df['timestamp'].isna().sum() > 0:
        print('Failed timestamp examples:')
        print(raw[df['timestamp'].isna()].head(10).tolist())

    df = df.dropna(subset=['timestamp'])
    df = df.sort_values('timestamp')
    df = df.drop_duplicates(subset=['timestamp'], keep='first')
    df = df.reset_index(drop=True)

    print('Final rows:', len(df))
    print('Time range:', df['timestamp'].min(), 'to', df['timestamp'].max())
    return df

def clean_filename(name):
    name = str(name).replace(' ', '_')
    name = re.sub(r'[^A-Za-z0-9_\-]', '', name)
    return name

def prefix_columns(df, prefix):
    df = df.copy()
    return df.rename(columns={c: prefix + c for c in df.columns if c != 'timestamp'})

def active_signal(series):
    s = series.copy()
    if s.dtype == 'O':
        return s.astype(str).str.lower().str.strip().isin(
            ['on', 'true', '1', 'active', 'comfort', 'acceso', 'yes', 'automatic', 'auto']
        )
    return pd.to_numeric(s, errors='coerce').fillna(0) > 0

def get_room_columns(df, prefixes):
    cols = ['timestamp']
    for p in prefixes:
        for suffix in ['O', 'T', 'S', 'V', 'M', 'deltaT']:
            col = f'{p}_{suffix}'
            if col in df.columns:
                cols.append(col)
    # Retain available shared environmental and plant fields when constructing
    # room-level dataframes.
    for c in [
        'EX_1.1_T', 'EX_1_1_T', 'External_T', 'T_ext',
        'T_est', 'Stato', 'GF_PATT', 'MandGF', 'MandPuffer', 'RitFCFreddo', 'LuxMillivolt',
        'AHU_T_est', 'AHU_Stato', private_value('inputs.literal_010'), private_value('inputs.literal_011'),
        private_value('inputs.literal_012'), 'SONDE_GF_PATT', private_value('inputs.literal_013')
    ]:
        if c in df.columns:
            cols.append(c)
    return list(dict.fromkeys(cols))

def get_valve_columns(df, prefixes):
    cols = ['timestamp']
    for p in prefixes:
        for suffix in ['V', 'VA']:
            col = f'{p}_{suffix}'
            if col in df.columns:
                cols.append(col)
    return list(dict.fromkeys(cols))

def pick_first_existing(df, candidates, default=np.nan):
    for c in candidates:
        if c in df.columns:
            return pd.to_numeric(df[c], errors='coerce')
    return pd.Series(default, index=df.index)

def merge_asof_with_offset(left, right, source_label, tolerance=TIME_TOLERANCE, direction=MERGE_DIRECTION_FOR_INPUTS):
    """Align each room timestamp with the latest eligible source observation and
    retain the matching time offset.
    """
    left = left.copy().sort_values('timestamp')
    right = right.copy().sort_values('timestamp')

    source_time_col = f'{source_label}_matched_timestamp'
    offset_col = f'{source_label}_sync_offset_min'

    right[source_time_col] = right['timestamp']

    merged = pd.merge_asof(
        left,
        right,
        on='timestamp',
        direction=direction,
        tolerance=pd.Timedelta(tolerance)
    )

    merged[offset_col] = (
        pd.to_datetime(merged[source_time_col], errors='coerce') -
        pd.to_datetime(merged['timestamp'], errors='coerce')
    ).dt.total_seconds() / 60.0

    return merged

def summarize_offsets(df, room_name, source_label, phase):
    offset_col = f'{source_label}_sync_offset_min'
    if offset_col not in df.columns:
        return None
    s = pd.to_numeric(df[offset_col], errors='coerce')
    valid = s.dropna()
    if len(valid) == 0:
        return {
            'phase': phase, 'room': room_name, 'source': source_label, 'n_rows': len(df),
            'n_matched': 0, 'match_rate_pct': 0.0, 'future_match_pct': np.nan,
            'exact_match_pct': np.nan, 'offset_min_mean': np.nan, 'offset_min_median': np.nan,
            'offset_min_min': np.nan, 'offset_min_max': np.nan,
        }
    return {
        'phase': phase,
        'room': room_name,
        'source': source_label,
        'n_rows': len(df),
        'n_matched': int(valid.shape[0]),
        'match_rate_pct': 100.0 * valid.shape[0] / max(len(df), 1),
        'future_match_pct': 100.0 * (valid > 1e-9).mean(),
        'exact_match_pct': 100.0 * (valid.abs() < 1e-9).mean(),
        'offset_min_mean': float(valid.mean()),
        'offset_min_median': float(valid.median()),
        'offset_min_min': float(valid.min()),
        'offset_min_max': float(valid.max()),
    }

def ceil_timestamp(ts, freq=GRID_FREQ):
    return pd.Timestamp(ts).ceil(freq)

def floor_timestamp(ts, freq=GRID_FREQ):
    return pd.Timestamp(ts).floor(freq)

def is_step_like_grid_column(col):
    """Identify setpoint, control and status signals that should be forward-filled
    rather than interpolated.
    """
    c = str(col)
    cl = c.lower()

    # Room-level BEMS setpoint/control columns.
    if re.search(r'(^|_)s$', c, flags=re.IGNORECASE):  
        return True
    if re.search(r'(^|_)v$', c, flags=re.IGNORECASE):  
        return True
    if re.search(r'(^|_)o$', c, flags=re.IGNORECASE):  
        return True
    if c in {'Room_S', 'Room_V', 'Room_O', 'Offset', 'Heating_Target_for_detection'}:
        return True

    # Status/command/control-like features.
    keywords = [
        'stato', 'status', 'signal', 'valve', 'fan', 'planton', 'plant_on',
        'mod_', 'out_v', 'in_pm', 'gf_patt', 'patt', 'command', 'cmd',
        'apertura', 'open', 'switch',
    ]
    if any(k in cl for k in keywords):
        return True

    # Avoid treating temperatures as step-like just because they come from a prefixed
    # source.
    return False

def interpolate_to_exact_grid(df, freq=GRID_FREQ):
    """Reconstruct signals on the configured regular grid.

    When causal filling is enabled, use the latest preceding values. Otherwise,
    interpolate continuous signals in time and forward-fill step-like signals when
    signal-aware filling is enabled. Interpolation can use later observations and
    therefore represents offline reconstruction.
    """
    df = df.copy().sort_values('timestamp')
    df = df.drop_duplicates(subset=['timestamp'], keep='first')
    df = df.set_index('timestamp').sort_index()

    start = ceil_timestamp(df.index.min(), freq)
    end = floor_timestamp(df.index.max(), freq)
    grid = pd.date_range(start=start, end=end, freq=freq)

    work = df.copy()
    for c in work.columns:
        if work[c].dtype == 'O':
            converted = pd.to_numeric(work[c], errors='coerce')
            if converted.notna().sum() > 0:
                work[c] = converted

    numeric_cols = work.select_dtypes(include=[np.number, 'bool']).columns.tolist()
    non_numeric_cols = [c for c in work.columns if c not in numeric_cols]
    combined_index = work.index.union(grid).sort_values()

    if globals().get('CAUSAL_GRID_FILL_FOR_INPUTS', True):
        # Causal: each grid row uses the most recent value available at or before that
        # grid time.
        num_grid = work[numeric_cols].reindex(combined_index).ffill().loc[grid]
    else:
        if globals().get('SIGNAL_AWARE_GRID_FILL', True):
            step_cols = [c for c in numeric_cols if is_step_like_grid_column(c)]
            continuous_cols = [c for c in numeric_cols if c not in step_cols]

            pieces_num = []
            if continuous_cols:
                cont_grid = (
                    work[continuous_cols]
                    .reindex(combined_index)
                    .interpolate(method='time', limit_area='inside')
                    .loc[grid]
                )
                pieces_num.append(cont_grid)
            if step_cols:
                step_grid = work[step_cols].reindex(combined_index).ffill().loc[grid]
                pieces_num.append(step_grid)

            num_grid = pd.concat(pieces_num, axis=1) if pieces_num else pd.DataFrame(index=grid)
            # Preserve original column order.
            num_grid = num_grid[[c for c in numeric_cols if c in num_grid.columns]]
        else:
            # Compatibility branch: interpolate numeric fields when signal-aware filling
            # is disabled.
            num_grid = (
                work[numeric_cols]
                .reindex(combined_index)
                .interpolate(method='time', limit_area='inside')
                .loc[grid]
            )

    pieces = [num_grid]
    if non_numeric_cols:
        non_grid = work[non_numeric_cols].reindex(combined_index).ffill().loc[grid]
        pieces.append(non_grid)

    out = pd.concat(pieces, axis=1).reset_index().rename(columns={'index': 'timestamp'})
    out = out.sort_values('timestamp').reset_index(drop=True)
    out['is_exact_5min_grid'] = True
    return out

def add_exact_shift(df, value_col, minutes, out_col, direction='past'):
    steps = int(minutes // STEP_MIN)
    if minutes % STEP_MIN != 0:
        raise ValueError('Only multiples of STEP_MIN are supported.')
    if direction == 'past':
        df[out_col] = df[value_col].shift(steps)
    elif direction == 'future':
        df[out_col] = df[value_col].shift(-steps)
    else:
        raise ValueError("direction must be 'past' or 'future'")
    return df

# observation-based raw CSV reading and preparation helpers

def read_csv_indexed_dadras_style(path, label):
    """Read a private CSV with mixed timestamp formats, parse times as UTC, and
    convert them to timezone-naive local timestamps. Remove duplicate timestamps
    using the configured first-record convention.
    """
    if not os.path.exists(path):
        print(f'[DrDadras-style prep] Missing optional file: {label}: {path}')
        return None

    d = pd.read_csv(path)
    d.columns = d.columns.astype(str).str.strip()

    if 'timestamp' not in d.columns:
        raise ValueError(f'{label} has no timestamp column: {path}')

    raw = d['timestamp'].astype(str).str.strip()

    # Robust parser for mixed timestamp strings, while keeping reference-study timezone
    # spirit.
    try:
        ts = pd.to_datetime(raw, errors='coerce', utc=True, format='mixed', dayfirst=False)
    except TypeError:
        ts = pd.to_datetime(raw, errors='coerce', utc=True, dayfirst=False, infer_datetime_format=True)

    # Fallback for any remaining failures.
    failed = ts.isna()
    if failed.any():
        try:
            fallback = pd.to_datetime(raw[failed], errors='coerce', utc=True, format='mixed', dayfirst=True)
        except TypeError:
            fallback = pd.to_datetime(raw[failed], errors='coerce', utc=True, dayfirst=True, infer_datetime_format=True)
        ts.loc[failed] = fallback

    n_failed = int(ts.isna().sum())

    # Reference-study timezone conversion.
    ts = ts.dt.tz_convert('Europe/Rome').dt.tz_localize(None)

    d = d.drop(columns=['timestamp'])
    d.index = ts
    d = d[~d.index.isna()]
    d = d.sort_index()
    before_dup = len(d)
    d = d[~d.index.duplicated(keep='first')]

    print(f'\n{label} — DrDadras-style read')
    print('Rows before timestamp drop:', len(raw))
    print('Failed timestamps:', n_failed)
    print('Rows after parsing:', before_dup)
    print('Rows after duplicate removal:', len(d))
    print('Time range:', d.index.min(), 'to', d.index.max())
    if n_failed > 0:
        print('Failed timestamp examples:', raw[pd.Series(ts).isna()].head(5).tolist() if hasattr(raw, 'head') else '')
    return d

def reset_index_timestamp(d):
    out = d.copy()
    out = out.reset_index().rename(columns={'index': 'timestamp'})
    return out

def keep_existing_columns(d, cols):
    return [c for c in cols if c in d.columns]

def read_feature_source_dadras_time(path, label):
    """Read an auxiliary feature stream using the same UTC-to-local timestamp
    convention as the room observations.
    """
    d = read_csv_indexed_dadras_style(path, label)
    if d is None:
        return pd.DataFrame(columns=['timestamp'])

    d = clean_physical_sentinels(d, f'raw_{label}_before_reset_timestamp')
    d = reset_index_timestamp(d)
    d = d.sort_values('timestamp').drop_duplicates(subset=['timestamp'], keep='first').reset_index(drop=True)
    return d

def audit_source_time_alignment(source_frames, out_dir=None):
    """Summarize source timestamp ranges and flag unusual offsets without requiring
    identical observation coverage.
    """
    rows = []
    for label, df in source_frames.items():
        if df is None or len(df) == 0 or 'timestamp' not in df.columns:
            rows.append({
                'source': label,
                'n_rows': 0,
                'start_time': pd.NaT,
                'end_time': pd.NaT,
                'median_step_min': np.nan,
                'status': 'missing_or_empty',
            })
            continue

        ts = pd.to_datetime(df['timestamp'], errors='coerce').dropna().sort_values()
        if len(ts) == 0:
            rows.append({
                'source': label,
                'n_rows': int(len(df)),
                'start_time': pd.NaT,
                'end_time': pd.NaT,
                'median_step_min': np.nan,
                'status': 'no_parseable_timestamps',
            })
            continue

        diffs = ts.diff().dropna().dt.total_seconds() / 60.0
        rows.append({
            'source': label,
            'n_rows': int(len(df)),
            'n_parseable_timestamps': int(len(ts)),
            'start_time': ts.min(),
            'end_time': ts.max(),
            'median_step_min': float(diffs.median()) if len(diffs) else np.nan,
            'status': 'ok',
        })

    audit = pd.DataFrame(rows)

    if len(audit) > 0 and 'locali_features' in set(audit['source']):
        ref = audit.loc[audit['source'] == 'locali_features'].iloc[0]
        ref_start = pd.to_datetime(ref['start_time'], errors='coerce')
        ref_end = pd.to_datetime(ref['end_time'], errors='coerce')
        audit['start_offset_from_locali_min'] = (
            pd.to_datetime(audit['start_time'], errors='coerce') - ref_start
        ).dt.total_seconds() / 60.0
        audit['end_offset_from_locali_min'] = (
            pd.to_datetime(audit['end_time'], errors='coerce') - ref_end
        ).dt.total_seconds() / 60.0

    if 'SOURCE_TIME_ALIGNMENT_AUDIT' not in globals():
        globals()['SOURCE_TIME_ALIGNMENT_AUDIT'] = []
    globals()['SOURCE_TIME_ALIGNMENT_AUDIT'].extend(audit.to_dict(orient='records'))

    print('\nSOURCE TIME ALIGNMENT AUDIT')
    display(audit)

    if out_dir is not None:
        audit_dir = os.path.join(out_dir, 'data_cleaning_audits')
        os.makedirs(audit_dir, exist_ok=True)
        path = os.path.join(audit_dir, 'source_time_alignment_audit.csv')
        audit.to_csv(path, index=False)
        print('Saved source time alignment audit:', path)

    # Source streams may have different observation periods. Report offsets for
    # inspection rather than requiring identical coverage.
    if 'start_offset_from_locali_min' in audit.columns:
        suspicious = audit[
            audit['source'].isin(['cal_features', 'valve_features']) &
            audit['start_offset_from_locali_min'].abs().between(30, 180, inclusive='both')
        ].copy()
        if len(suspicious):
            print('\nWARNING: possible source-time mismatch detected.')
            display(suspicious)

    return audit


In [ ]:

# Calendar/daylight proxy features

def add_calendar_daylight_proxy_features(df, latitude_deg=BUILDING_LATITUDE_DEG):
    df = df.copy()
    ts = pd.to_datetime(df['timestamp'])

    df['hour_decimal'] = ts.dt.hour + ts.dt.minute / 60.0 + ts.dt.second / 3600.0
    df['day_of_year'] = ts.dt.dayofyear
    df['day_of_week'] = ts.dt.dayofweek
    df['month'] = ts.dt.month
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

    df['doy_sin'] = np.sin(2 * np.pi * df['day_of_year'] / 365.25)
    df['doy_cos'] = np.cos(2 * np.pi * df['day_of_year'] / 365.25)
    df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7.0)
    df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7.0)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12.0)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12.0)

    gamma = 2 * np.pi * (df['day_of_year'] - 1) / 365.25
    decl = (
        0.006918
        - 0.399912 * np.cos(gamma)
        + 0.070257 * np.sin(gamma)
        - 0.006758 * np.cos(2 * gamma)
        + 0.000907 * np.sin(2 * gamma)
        - 0.002697 * np.cos(3 * gamma)
        + 0.00148 * np.sin(3 * gamma)
    )
    lat = np.deg2rad(latitude_deg)
    cos_omega = -np.tan(lat) * np.tan(decl)
    cos_omega = np.clip(cos_omega, -1, 1)
    omega = np.arccos(cos_omega)
    daylight_hours = 24.0 * omega / np.pi

    df['solar_declination_proxy_deg'] = np.rad2deg(decl)
    df['daylight_hours_proxy'] = daylight_hours
    df['sunrise_hour_proxy'] = 12.0 - daylight_hours / 2.0
    df['sunset_hour_proxy'] = 12.0 + daylight_hours / 2.0

    df['daylight_fraction_proxy'] = (
        (df['hour_decimal'] - df['sunrise_hour_proxy']) /
        (df['daylight_hours_proxy'].replace(0, np.nan))
    ).clip(0, 1)
    df['time_since_sunrise_h_proxy'] = (df['hour_decimal'] - df['sunrise_hour_proxy']).clip(lower=0)
    df['time_to_sunset_h_proxy'] = (df['sunset_hour_proxy'] - df['hour_decimal']).clip(lower=0)
    df['solar_elevation_proxy'] = np.sin(np.pi * df['daylight_fraction_proxy']).fillna(0)
    return df

In [ ]:

# Cooling target helpers

def load_manual_cooling_targets(path=None):
    targets = {}
    if path is None or str(path).strip() == '':
        print(f'No manual cooling target file provided. Using fixed fallback = {FIXED_COOLING_TARGET_C} °C.')
        return targets
    if not os.path.exists(path):
        print(f'Manual cooling target file not found: {path}')
        print(f'Using fixed fallback = {FIXED_COOLING_TARGET_C} °C for rooms not found in a target table.')
        return targets

    try:
        tmp = pd.read_excel(path) if str(path).lower().endswith(('.xlsx', '.xls')) else pd.read_csv(path)
    except Exception as e:
        print(f'Could not read manual cooling target file: {path}')
        print('Error:', repr(e))
        return targets

    tmp.columns = [str(c).strip() for c in tmp.columns]
    room_col_candidates = ['room', 'Room', 'ROOM', 'room_name', 'Room_Name', 'RoomName', 'zone', 'Zone', 'prefix', 'Prefix', 'sensor', 'Sensor', 'ID', 'id']
    target_col_candidates = ['Target', 'target', 'Cooling_Target', 'cooling_target', 'CoolingTarget', 'T_target', 'target_temperature', 'Target_Temperature', 'Manual_Target']
    room_col = next((c for c in room_col_candidates if c in tmp.columns), None)
    target_col = next((c for c in target_col_candidates if c in tmp.columns), None)
    if room_col is None or target_col is None:
        print('Manual target file was found but required columns were not found.')
        print('Available columns:', list(tmp.columns))
        return targets

    for _, row in tmp.iterrows():
        key = str(row[room_col]).strip()
        val = pd.to_numeric(row[target_col], errors='coerce')
        if key and pd.notna(val):
            targets[key] = float(val)

    print(f'Loaded {len(targets)} manual cooling targets from: {path}')
    return targets

def resolve_cooling_target(room_name, prefixes, measured_setpoint_series=None, manual_targets=None):
    manual_targets = manual_targets or {}

    def constant_target(value, index_like):
        idx = measured_setpoint_series.index if measured_setpoint_series is not None else index_like
        return pd.Series(float(value), index=idx)

    manual_value = None
    lookup_keys = [room_name] + list(prefixes)
    lookup_keys += [clean_filename(k) for k in lookup_keys]
    for key in lookup_keys:
        if key in manual_targets:
            manual_value = manual_targets[key]
            break

    mode = str(TARGET_MODE).lower().strip()

    if mode == 'manual_constant':
        return constant_target(FIXED_COOLING_TARGET_C, None), f'fixed_manual_{FIXED_COOLING_TARGET_C:g}C'

    if mode == 'manual_file':
        if manual_value is not None:
            return constant_target(manual_value, None), f'manual_file_{manual_value:g}C'
        return constant_target(FIXED_COOLING_TARGET_C, None), f'fixed_fallback_{FIXED_COOLING_TARGET_C:g}C'

    if mode == 'room_setpoint':
        if measured_setpoint_series is not None:
            s = pd.to_numeric(measured_setpoint_series, errors='coerce')
            if s.notna().sum() > 0:
                return s.ffill().bfill().fillna(FIXED_COOLING_TARGET_C), 'measured_Room_S'
        return constant_target(FIXED_COOLING_TARGET_C, None), f'fixed_fallback_{FIXED_COOLING_TARGET_C:g}C'

    if mode == 'auto':
        if manual_value is not None:
            return constant_target(manual_value, None), f'manual_file_{manual_value:g}C'
        if measured_setpoint_series is not None:
            s = pd.to_numeric(measured_setpoint_series, errors='coerce')
            if s.notna().sum() > 0:
                return s.ffill().bfill().fillna(FIXED_COOLING_TARGET_C), 'measured_Room_S'
        return constant_target(FIXED_COOLING_TARGET_C, None), f'fixed_fallback_{FIXED_COOLING_TARGET_C:g}C'

    raise ValueError(f'Unknown TARGET_MODE={TARGET_MODE!r}.')

In [ ]:

# Feature creation on exact 5-minute grid

OPTIONAL_COOLING_SOURCE_FEATURES = [
    'MandGF', 'MandPuffer', 'RitFCFreddo', 'GF_PATT', 'LuxMillivolt',
    'T_ReturnAir', 'CoolingCoilSignal', 'AHUPlantOnSignal',
]

OPTIONAL_SOLAR_WEATHER_FEATURES = [
    'SolarRadiation', 'RelativeHumidity', 'WindSpeed',
] + [f'SolarRadiation -{lag}min' for lag in SOLAR_LAGS_MINUTES]

def create_heating_features_on_grid(full_room_df):
    df = full_room_df.copy().sort_values('timestamp').reset_index(drop=True)

    # Prefer the cleaned air-handling-unit outdoor temperature when available.
    df['T_ext'] = pick_first_existing(df, ['T_est', 'AHU_T_est', 'CAL_T_est', 'EX_1.1_T', 'EX_1_1_T', 'External_T', 'T_ext'])
    df['T_Boiler'] = pick_first_existing(df, ['CAL_T_caldaia'])
    df['MandBoiler'] = pick_first_existing(df, ['CAL_T_mand', 'AHU_T_mand', 'T_mand2'])
    df['Offset'] = pd.to_numeric(df['Room_O'], errors='coerce')

    df['Room_T_slope_prev'] = (pd.to_numeric(df['Room_T'], errors='coerce') - pd.to_numeric(df['Room_T'], errors='coerce').shift(1)) / STEP_MIN
    slope_10 = (pd.to_numeric(df['Room_T'], errors='coerce') - pd.to_numeric(df['Room_T'], errors='coerce').shift(2)) / (2 * STEP_MIN)
    df['Room_T_slope_prev'] = df['Room_T_slope_prev'].fillna(slope_10)

    for lag in ROOM_LAGS_MINUTES:
        add_exact_shift(df, 'Room_T', lag, f'Room_T -{lag}min', direction='past')

    for lag in BOILER_LAGS_MINUTES:
        add_exact_shift(df, 'T_Boiler', lag, f'T_Boiler -{lag}min', direction='past')

    if ADD_EXTERNAL_TEMP_LAGS:
        for lag in EXT_LAGS_MINUTES:
            add_exact_shift(df, 'T_ext', lag, f'T_ext -{lag}min', direction='past')

    df = add_calendar_daylight_proxy_features(df, latitude_deg=BUILDING_LATITUDE_DEG)
    df = clean_physical_sentinels(df, 'heating_features_after_feature_creation', verbose=False)
    return df

def create_cooling_features_on_grid(full_room_df):
    df = full_room_df.copy().sort_values('timestamp').reset_index(drop=True)

    # Prefer cleaned outdoor-temperature observations from the air-handling-unit stream.
    df['T_ext'] = pick_first_existing(df, [
        'T_est', 'AHU_T_est', 'AHU_CTA_T_EST', 'AHU_TEMP', 'AHU_ARPA_TEMP',
        'EX_1.1_T', 'EX_1_1_T', 'External_T', 'T_ext',
        'CAL_T_est', 'CAL_PAN_IN_TE1', 'CAL_PAN_IN_TE2',
    ])

    if ADD_SOLAR_RADIATION_FEATURES_IF_AVAILABLE:
        df['SolarRadiation'] = pick_first_existing(df, ['RADSOL', 'CAL_RADSOL', 'AHU_RADSOL', 'ARPA_RADSOL', 'SolarRadiation', 'solar_radiation', private_value('inputs.literal_013'), 'LuxMillivolt'])
        df['RelativeHumidity'] = pick_first_existing(df, ['UMID', 'CAL_UMID', 'AHU_UMID', 'ARPA_UMID', 'RelativeHumidity', 'relative_humidity'])
        df['WindSpeed'] = pick_first_existing(df, ['VVENTO', 'CAL_VVENTO', 'AHU_VVENTO', 'ARPA_VVENTO', 'WindSpeed', 'wind_speed'])

    df['T_CoolSupply'] = pick_first_existing(df, [
        private_value('inputs.literal_011'), 'MandGF', private_value('inputs.literal_012'), 'MandPuffer',
        'AHU_CTA_IN_TM', 'AHU_IN_TM', 'AHU_T_MAND', 'AHU_T_mand', 'AHU_Tmand',
        'CTA_IN_TM', 'T_mand', 'T_MAND', 'GFC_T_mand', 'VALVE_GFC_T_mand',
        'CAL_GFC_T_mand', 'CAL_T_mand_fancoil', 'CAL_T_mand_FC', 'CAL_T_mand'
    ])

    # Optional cooling-source columns are created only for audit/availability checking.
    # They are exported only if EXPORT_OPTIONAL_COOLING_SOURCE_FEATURES=True.
    df['MandGF'] = pick_first_existing(df, [private_value('inputs.literal_011'), 'MandGF'])
    df['MandPuffer'] = pick_first_existing(df, [private_value('inputs.literal_012'), 'MandPuffer'])
    df['RitFCFreddo'] = pick_first_existing(df, [private_value('inputs.literal_010'), 'RitFCFreddo'])
    df['GF_PATT'] = pick_first_existing(df, ['SONDE_GF_PATT', private_value('inputs.literal_014'), 'GF_PATT'])
    df['LuxMillivolt'] = pick_first_existing(df, [private_value('inputs.literal_013'), 'LuxMillivolt'])
    df['T_ReturnAir'] = pick_first_existing(df, ['AHU_CTA_IN_TR', 'AHU_IN_TR', 'AHU_T_RIPR', 'AHU_T_rit', 'CTA_IN_TR'])
    df['CoolingCoilSignal'] = pick_first_existing(df, ['AHU_CTA_OUT_VF', 'AHU_OUT_VF', 'CTA_OUT_VF', 'AHU_BATT_FREDDA', 'AHU_Batteria_Fredda'])
    df['AHUPlantOnSignal'] = pick_first_existing(df, ['AHU_CTA_MOD_IM', 'AHU_MOD_IM', 'CTA_MOD_IM', 'AHU_CTA_OUT_VM', 'AHU_OUT_VM', 'CTA_OUT_VM'])

    audit_all_missing_columns(df, 'cooling_features_after_feature_creation', OPTIONAL_COOLING_SOURCE_FEATURES + OPTIONAL_SOLAR_WEATHER_FEATURES)

    df['Offset'] = pd.to_numeric(df['Room_O'], errors='coerce')

    df['Room_T_slope_prev'] = (pd.to_numeric(df['Room_T'], errors='coerce') - pd.to_numeric(df['Room_T'], errors='coerce').shift(1)) / STEP_MIN
    slope_10 = (pd.to_numeric(df['Room_T'], errors='coerce') - pd.to_numeric(df['Room_T'], errors='coerce').shift(2)) / (2 * STEP_MIN)
    df['Room_T_slope_prev'] = df['Room_T_slope_prev'].fillna(slope_10)

    df['Cooling_deltaT_supply_minus_room'] = pd.to_numeric(df['T_CoolSupply'], errors='coerce') - pd.to_numeric(df['Room_T'], errors='coerce')
    df['Cooling_input_proxy'] = pd.to_numeric(df['Room_V'], errors='coerce') * pd.to_numeric(df['Cooling_deltaT_supply_minus_room'], errors='coerce')

    for lag in ROOM_LAGS_MINUTES:
        add_exact_shift(df, 'Room_T', lag, f'Room_T -{lag}min', direction='past')

    for lag in COOLING_SUPPLY_LAGS_MINUTES:
        add_exact_shift(df, 'T_CoolSupply', lag, f'T_CoolSupply -{lag}min', direction='past')

    if ADD_EXTERNAL_TEMP_LAGS:
        for lag in EXT_LAGS_MINUTES:
            add_exact_shift(df, 'T_ext', lag, f'T_ext -{lag}min', direction='past')

    if ADD_SOLAR_RADIATION_FEATURES_IF_AVAILABLE and 'SolarRadiation' in df.columns:
        for lag in SOLAR_LAGS_MINUTES:
            add_exact_shift(df, 'SolarRadiation', lag, f'SolarRadiation -{lag}min', direction='past')

    df = add_calendar_daylight_proxy_features(df, latitude_deg=BUILDING_LATITUDE_DEG)
    df = clean_physical_sentinels(df, 'cooling_features_after_feature_creation', verbose=False)
    return df


In [ ]:

# Cooling and heating use sign-consistent morning-recovery criteria on original
# observation timestamps.

def _parse_clock_to_minutes(clock_str):
    if clock_str is None:
        return None
    s = str(clock_str).strip()
    if s == "":
        return None
    parts = s.split(":")
    h = int(parts[0])
    m = int(parts[1]) if len(parts) > 1 else 0
    return 60 * h + m

def time_leq_clock(ts_series, latest_clock):
    """Return whether each timestamp is at or before the specified clock time, using
    hour and minute.
    """
    latest_min = _parse_clock_to_minutes(latest_clock)
    if latest_min is None:
        return pd.Series(True, index=ts_series.index)
    ts = pd.to_datetime(ts_series, errors='coerce')
    tod_min = ts.dt.hour * 60 + ts.dt.minute
    return tod_min <= latest_min

def time_lt_clock(ts_series, latest_clock):
    """Return whether each timestamp is strictly before the specified clock time,
    using hour and minute.
    """
    latest_min = _parse_clock_to_minutes(latest_clock)
    if latest_min is None:
        return pd.Series(True, index=ts_series.index)
    ts = pd.to_datetime(ts_series, errors='coerce')
    tod_min = ts.dt.hour * 60 + ts.dt.minute
    return tod_min < latest_min

def time_between_clock(ts_series, start_clock=None, end_clock=None, include_start=True, include_end=True):
    """Return a time-of-day mask for the specified clock limits and endpoint
    conventions.
    """
    ts = pd.to_datetime(ts_series, errors='coerce')
    tod_min = ts.dt.hour * 60 + ts.dt.minute
    ok = pd.Series(True, index=ts_series.index)
    start_min = _parse_clock_to_minutes(start_clock)
    end_min = _parse_clock_to_minutes(end_clock)
    if start_min is not None:
        ok &= (tod_min >= start_min) if include_start else (tod_min > start_min)
    if end_min is not None:
        ok &= (tod_min <= end_min) if include_end else (tod_min < end_min)
    return ok

def interpolate_target_crossing_time(
    t_prev,
    t_curr,
    T_prev,
    T_curr,
    threshold_prev,
    threshold_curr=None,
    phase=None,
):
    """Estimate a crossing between adjacent points by linearly interpolating the zero
    of temperature minus threshold. This also supports thresholds that differ
    between the two points.
    """
    t_prev = pd.Timestamp(t_prev)
    t_curr = pd.Timestamp(t_curr)

    if threshold_curr is None:
        threshold_curr = threshold_prev

    vals = [T_prev, T_curr, threshold_prev, threshold_curr]
    if any(pd.isna(v) for v in vals):
        return t_curr

    T_prev = float(T_prev)
    T_curr = float(T_curr)
    th_prev = float(threshold_prev)
    th_curr = float(threshold_curr)

    g_prev = T_prev - th_prev
    g_curr = T_curr - th_curr

    if pd.isna(g_prev) or pd.isna(g_curr):
        return t_curr

    if g_curr == g_prev:
        return t_curr

    frac = -g_prev / (g_curr - g_prev)
    frac = float(np.clip(frac, 0.0, 1.0))
    return t_prev + frac * (t_curr - t_prev)

def compute_continuous_end_time(df, end_idx, threshold_series, phase):
    """Return the detected grid endpoint and a diagnostic interpolated crossing time
    between it and the preceding point.
    """
    grid_end_time = pd.Timestamp(df.loc[end_idx, 'timestamp'])

    if end_idx <= 0:
        return grid_end_time, grid_end_time

    t_prev = pd.Timestamp(df.loc[end_idx - 1, 'timestamp'])
    t_curr = pd.Timestamp(df.loc[end_idx, 'timestamp'])
    T_prev = pd.to_numeric(df.loc[end_idx - 1, 'Room_T'], errors='coerce')
    T_curr = pd.to_numeric(df.loc[end_idx, 'Room_T'], errors='coerce')
    threshold_prev = pd.to_numeric(threshold_series.iloc[end_idx - 1], errors='coerce')
    threshold_curr = pd.to_numeric(threshold_series.iloc[end_idx], errors='coerce')

    continuous_end_time = interpolate_target_crossing_time(
        t_prev=t_prev,
        t_curr=t_curr,
        T_prev=T_prev,
        T_curr=T_curr,
        threshold_prev=threshold_prev,
        threshold_curr=threshold_curr,
        phase=phase,
    )

    return grid_end_time, continuous_end_time

def estimate_room_5min_noise_sigma(room_T, stable_mask=None, min_samples=None):
    """Estimate room temperature-change noise from stable or inactive periods, with
    the configured fallback when support is limited.
    """
    if min_samples is None:
        min_samples = globals().get('COOLING_CONFIRMATION_MIN_STABLE_SAMPLES', 30)

    T = pd.to_numeric(room_T, errors='coerce')
    dT = T.diff()

    if stable_mask is not None:
        m = pd.Series(stable_mask, index=T.index).fillna(False) & dT.notna()
        vals = dT[m].dropna()
    else:
        vals = dT.dropna()

    # If the stable pool is too small, fall back to all small-to-moderate deltas.
    if len(vals) < min_samples:
        vals = dT.dropna()

    if len(vals) == 0:
        return np.nan

    med = float(vals.median())
    mad = float(np.median(np.abs(vals - med)))
    sigma = 1.4826 * mad

    # Robust fallback if MAD collapses to zero.
    if not np.isfinite(sigma) or sigma <= 1e-9:
        sigma = float(vals.quantile(0.75) - vals.quantile(0.25)) / 1.349 if len(vals) >= 4 else 0.0

    return float(max(sigma, 0.0))

def compute_directional_start_confirmation_signal(
    df,
    room_T,
    active_signal_series,
    fan_active_series,
    phase,
):
    """Calculate optional noise-adjusted local and future temperature-change checks.
    Heating requires positive changes and cooling negative changes. The active
    event-extraction configuration disables this additional start-confirmation
    procedure.
    """
    T = pd.to_numeric(room_T, errors='coerce').reset_index(drop=True)
    active = pd.Series(active_signal_series, index=df.index).reset_index(drop=True).fillna(False)
    fan = pd.Series(fan_active_series, index=df.index).reset_index(drop=True).fillna(False)

    phase = str(phase).lower().strip()
    if phase not in ['heating', 'cooling']:
        raise ValueError("phase must be 'heating' or 'cooling'")

    if phase == 'heating':
        local_lookback_min = globals().get('HEATING_LOCAL_SLOPE_LOOKBACK_MIN', 5)
        future_lookahead_min = globals().get('HEATING_FUTURE_CONFIRMATION_LOOKAHEAD_MIN', 20)
        local_floor = globals().get('HEATING_LOCAL_SLOPE_MIN_RISE_FLOOR_C', 0.03)
        future_floor = globals().get('HEATING_FUTURE_CONFIRMATION_MIN_RISE_FLOOR_C', 0.15)
        local_mult = globals().get('HEATING_LOCAL_SLOPE_NOISE_SIGMA_MULTIPLIER', 1.0)
        future_mult = globals().get('HEATING_FUTURE_CONFIRMATION_NOISE_SIGMA_MULTIPLIER', 2.0)
        min_samples = globals().get('HEATING_CONFIRMATION_MIN_STABLE_SAMPLES', 30)
    else:
        local_lookback_min = globals().get('COOLING_LOCAL_SLOPE_LOOKBACK_MIN', 5)
        future_lookahead_min = globals().get('COOLING_FUTURE_CONFIRMATION_LOOKAHEAD_MIN', 20)
        local_floor = globals().get('COOLING_LOCAL_SLOPE_MIN_DROP_FLOOR_C', 0.03)
        future_floor = globals().get('COOLING_FUTURE_CONFIRMATION_MIN_DROP_FLOOR_C', 0.15)
        local_mult = globals().get('COOLING_LOCAL_SLOPE_NOISE_SIGMA_MULTIPLIER', 1.0)
        future_mult = globals().get('COOLING_FUTURE_CONFIRMATION_NOISE_SIGMA_MULTIPLIER', 2.0)
        min_samples = globals().get('COOLING_CONFIRMATION_MIN_STABLE_SAMPLES', 30)

    local_steps = max(1, int(round(float(local_lookback_min) / float(STEP_MIN))))
    future_steps = max(1, int(round(float(future_lookahead_min) / float(STEP_MIN))))

    # Noise is estimated mostly from non-active or fan-off periods. If too few points
    # exist,
    # estimate_room_5min_noise_sigma falls back to all available 5-min differences.
    stable_mask = (~active) | (~fan)
    sigma_5min = estimate_room_5min_noise_sigma(T, stable_mask=stable_mask, min_samples=min_samples)
    if not np.isfinite(sigma_5min):
        sigma_5min = 0.0

    # A lookback/future delta spans several 5-min increments, so noise grows
    # approximately as sqrt(n_steps).
    local_threshold = max(float(local_floor), float(local_mult) * float(sigma_5min) * np.sqrt(local_steps))
    future_threshold = max(float(future_floor), float(future_mult) * float(sigma_5min) * np.sqrt(future_steps))

    local_delta = T - T.shift(local_steps)
    future_delta = T.shift(-future_steps) - T

    if phase == 'heating':
        local_confirmed = local_delta >= local_threshold
        future_confirmed = future_delta >= future_threshold
    else:
        local_confirmed = local_delta <= -local_threshold
        future_confirmed = future_delta <= -future_threshold

    combined_confirmed = local_confirmed.fillna(False) & future_confirmed.fillna(False)

    return (
        local_confirmed.fillna(False),
        future_confirmed.fillna(False),
        combined_confirmed.fillna(False),
        local_delta,
        future_delta,
        local_threshold,
        future_threshold,
        sigma_5min,
        local_steps,
        future_steps,
    )

def compute_heating_confirmation_signal(df, room_T, heating_active, fan_active):
    return compute_directional_start_confirmation_signal(
        df=df,
        room_T=room_T,
        active_signal_series=heating_active,
        fan_active_series=fan_active,
        phase='heating',
    )

def compute_cooling_confirmation_signal(df, room_T, cooling_active, fan_active):
    return compute_directional_start_confirmation_signal(
        df=df,
        room_T=room_T,
        active_signal_series=cooling_active,
        fan_active_series=fan_active,
        phase='cooling',
    )

def resolve_heating_target(room_name, room_prefix_list):
    """Return the configured fixed heating target for a room independently of its
    recorded thermostat setpoint.
    """
    prefix_list = normalize_prefix_list(room_prefix_list)
    values = []
    missing = []
    for p in prefix_list:
        if p in HEATING_ROOM_TARGET_C:
            values.append(float(HEATING_ROOM_TARGET_C[p]))
        else:
            missing.append(p)
    if missing:
        raise ValueError(f'Missing fixed heating target(s) for {room_name}: {missing}')
    if len(values) == 0:
        raise ValueError(f'No fixed heating target resolved for {room_name} with prefixes {prefix_list}')
    if len(set(np.round(values, 6))) > 1:
        raise ValueError(f'Conflicting fixed heating targets for {room_name}: {dict(zip(prefix_list, values))}')
    return float(values[0]), 'fixed_room_specific_heating_target_from_HEATING_ROOM_TARGET_C'

def get_heating_target_series_for_detection(df):
    """Return the heating target series used by the event detector."""
    if 'Heating_Target_for_detection' not in df.columns:
        raise ValueError(
            "Heating_Target_for_detection is missing. "
            "Build it from HEATING_ROOM_TARGET_C before extracting heating events."
        )
    target = pd.to_numeric(df['Heating_Target_for_detection'], errors='coerce')
    if target.isna().all():
        raise ValueError('Heating_Target_for_detection is entirely NaN.')
    return target

def extract_heating_rampup_periods(df):
    """Detect heating recovery events from cleaned observations at their original
    timestamps.

    Select the first eligible morning row with sufficient fan signal and
    temperature below the fixed target allowance. Select the first subsequent row
    with a positive observed temperature increment and temperature within the
    endpoint allowance. Apply the configured search windows and duration bounds;
    no start-slope or future-confirmation condition is added.
    """
    mode = globals().get('HEATING_EVENT_EXTRACTION_MODE', 'dadras_symmetric_exact')
    if mode != 'dadras_symmetric_exact':
        raise ValueError(
            "This notebook version is configured for exact symmetric Dadras-style heating extraction only. "
            "Set HEATING_EVENT_EXTRACTION_MODE='dadras_symmetric_exact'."
        )

    df = df.copy().sort_values('timestamp').reset_index(drop=True)
    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')

    room_T = pd.to_numeric(df['Room_T'], errors='coerce')
    heating_target = get_heating_target_series_for_detection(df)
    room_V = pd.to_numeric(df['Room_V'], errors='coerce')

    heating_start_threshold = heating_target - HEATING_START_BELOW_SETPOINT
    heating_end_threshold = heating_target - HEATING_END_BELOW_SETPOINT

    t_lag = room_T.shift(1)
    delta_from_lag = room_T - t_lag
    ts = pd.to_datetime(df['timestamp'], errors='coerce')

    in_heating_daily_window = time_between_clock(
        ts,
        HEATING_MORNING_START,
        HEATING_MORNING_END,
        include_start=True,
        include_end=True,
    )

    latest_hour = int(globals().get('HEATING_LATEST_START_HOUR', 10))
    before_heating_latest_start = ts.dt.hour < latest_hour

    fan_active = room_V > HEATING_MIN_ROOM_FAN_SIGNAL
    below_start = room_T < heating_start_threshold

    # Symmetric reference-study start: fan + wrong-side-of-target + latest-start rule.
    # No start slope and no future confirmation.
    start_condition = (
        in_heating_daily_window
        & before_heating_latest_start
        & fan_active
        & below_start
    ).fillna(False)

    # Symmetric reference-study end: positive local temperature change + heating
    # target-band reached.
    end_condition = (
        in_heating_daily_window
        & (delta_from_lag > 0)
        & (room_T >= heating_end_threshold)
    ).fillna(False)

    # Diagnostic columns, with names paired to the cooling reference-study diagnostics.
    df['heating_candidate_in_window'] = in_heating_daily_window
    df['heating_candidate_before_latest_start'] = before_heating_latest_start
    df['heating_candidate_in_end_search_window'] = in_heating_daily_window
    df['heating_candidate_below_start_threshold'] = below_start
    df['heating_candidate_active'] = True
    df['heating_candidate_fan_active'] = fan_active
    df['heating_candidate_local_delta_C'] = delta_from_lag
    df['heating_candidate_local_slope_required_C'] = 0.0
    df['heating_candidate_local_slope_confirmed'] = delta_from_lag > 0
    df['heating_candidate_future_rise_C'] = np.nan
    df['heating_candidate_required_future_rise_C'] = np.nan
    df['heating_candidate_noise_sigma_5min_C'] = np.nan
    df['heating_candidate_confirmed_future_warming'] = True
    df['heating_candidate_confirmed_start'] = start_condition
    df['heating_latest_start_rule'] = f'dadras_symmetric_idx_hour_lt_{latest_hour}'
    df['heating_start_condition'] = start_condition
    df['heating_end_condition_dadras_symmetric'] = end_condition
    df['heating_target_used_for_detection'] = heating_target
    df['heating_start_threshold'] = heating_start_threshold
    df['heating_end_threshold'] = heating_end_threshold
    df['heating_T_minus_T_1lag'] = delta_from_lag

    rampup_indices = []
    events = []
    event_id = 0
    saved_event_dates = set()

    tmp = df.copy()
    tmp['_row_pos'] = np.arange(len(tmp))
    tmp = tmp[in_heating_daily_window.fillna(False)].copy()

    for day, day_df in tmp.groupby(ts.loc[tmp.index].dt.date):
        if pd.isna(day):
            continue
        if ONE_EVENT_PER_ROOM_DAY and day in saved_event_dates:
            continue

        start_rows = day_df[start_condition.loc[day_df.index].values]
        if len(start_rows) == 0:
            continue

        start_idx = int(start_rows['_row_pos'].iloc[0])
        start_time = pd.Timestamp(df.loc[start_idx, 'timestamp'])

        after_start = day_df[day_df['_row_pos'] >= start_idx].copy()
        end_mask = end_condition.loc[after_start.index].values & (after_start['_row_pos'].values > start_idx)
        end_rows = after_start[end_mask]
        if len(end_rows) == 0:
            continue

        end_idx = int(end_rows['_row_pos'].iloc[0])
        end_time = pd.Timestamp(df.loc[end_idx, 'timestamp'])
        duration_min = round((end_time - start_time).seconds / 60.0, 0)

        if not (MIN_RAMPUP_MINUTES <= duration_min <= MAX_HEATING_RAMPUP_MINUTES):
            continue

        event_id += 1
        idx_range = list(range(start_idx, end_idx + 1))
        rampup_indices.extend(idx_range)

        start_temp = float(room_T.iloc[start_idx]) if pd.notna(room_T.iloc[start_idx]) else np.nan
        end_temp = float(room_T.iloc[end_idx]) if pd.notna(room_T.iloc[end_idx]) else np.nan
        target_start = float(heating_target.iloc[start_idx]) if pd.notna(heating_target.iloc[start_idx]) else np.nan
        target_end = float(heating_target.iloc[end_idx]) if pd.notna(heating_target.iloc[end_idx]) else np.nan
        temp_slope_5min_at_start = float(delta_from_lag.iloc[start_idx]) if pd.notna(delta_from_lag.iloc[start_idx]) else np.nan
        temp_slope_5min_at_end = float(delta_from_lag.iloc[end_idx]) if pd.notna(delta_from_lag.iloc[end_idx]) else np.nan

        temp30_time = start_time + pd.Timedelta(minutes=30)
        future_rows = df[pd.to_datetime(df['timestamp']) >= temp30_time]
        if len(future_rows) > 0:
            temp30_idx = int(future_rows.index[0])
            temp30 = float(room_T.iloc[temp30_idx]) if pd.notna(room_T.iloc[temp30_idx]) else np.nan
            slope30 = (temp30 - start_temp) / 30.0 if pd.notna(temp30) and pd.notna(start_temp) else np.nan
        else:
            temp30 = np.nan
            slope30 = np.nan

        slope_all = (end_temp - start_temp) / duration_min if duration_min not in [0, np.nan] and duration_min != 0 else np.nan

        events.append({
            'event_id': event_id,
            'phase': 'heating',
            'event_date': day,
            'start_time': start_time,
            'end_time': end_time,
            'duration_min': duration_min,
            'duration_target_min': duration_min,
            'duration_continuous_min': duration_min,
            'duration_grid_min': duration_min,
            'end_grid_time': end_time,
            'end_crossing_time': end_time,
            'end_idx_grid': end_idx,
            'duration_estimation': 'dadras_symmetric_exact_discrete_timestamp_no_interpolation',
            'observed_end_time': end_time,
            'actual_end_time': end_time,
            'observed_duration_min': duration_min,
            'actual_duration_min': duration_min,
            'indices': idx_range,
            'n_rows': len(idx_range),
            'availability_source': 'dadras_symmetric_exact_fan_signal_only',
            'event_detection_source': 'localiin_dadras_symmetric_heating_TV_fixed_target',
            'target_source_for_event_detection': 'fixed_room_specific_heating_target_from_HEATING_ROOM_TARGET_C',
            'heating_target_used_for_detection_at_start': target_start,
            'heating_start_threshold_at_start': float(heating_start_threshold.iloc[start_idx]) if pd.notna(heating_start_threshold.iloc[start_idx]) else np.nan,
            'heating_end_threshold_at_end': float(heating_end_threshold.iloc[end_idx]) if pd.notna(heating_end_threshold.iloc[end_idx]) else np.nan,
            'heating_start_local_slope_lookback_min': np.nan,
            'heating_start_local_delta_C': temp_slope_5min_at_start,
            'heating_start_required_local_delta_C': np.nan,
            'heating_start_future_confirmation_lookahead_min': np.nan,
            'heating_start_future_rise_C': np.nan,
            'heating_start_required_future_rise_C': np.nan,
            'heating_noise_sigma_5min_C': np.nan,
            'dadras_symmetric_T_minus_T_1lag_at_end_C': temp_slope_5min_at_end,
            'dadras_spirit_T_minus_T_1lag_at_end_C': temp_slope_5min_at_end,  # backward-compatible column name
            'dadras_symmetric_Temp30min': temp30,
            'dadras_symmetric_Slope30min_C_per_min': slope30,
            'dadras_symmetric_SlopeALL_C_per_min': slope_all,
            'room_T_at_start': start_temp,
            'room_T_at_end': end_temp,
            'target_at_start': target_start,
            'target_at_end': target_end,
            'target_crossing_threshold_at_end': float(heating_end_threshold.iloc[end_idx]) if pd.notna(heating_end_threshold.iloc[end_idx]) else np.nan,
            'threshold_at_start': float(heating_start_threshold.iloc[start_idx]) if pd.notna(heating_start_threshold.iloc[start_idx]) else np.nan,
            'threshold_at_end_grid': float(heating_end_threshold.iloc[end_idx]) if pd.notna(heating_end_threshold.iloc[end_idx]) else np.nan,
            'start_rule': f'symmetric Dadras: first idx in {HEATING_MORNING_START}-{HEATING_MORNING_END} with V>0.4, T<Heating_Target-0.5, idx.hour<{latest_hour}',
            'end_rule': 'symmetric Dadras: first idx after start with T-T_1lag>0 and T>=Heating_Target-0.4',
        })
        saved_event_dates.add(day)

    return attach_event_labels(df, events, phase='heating')

def build_cooling_availability_signal(df):
    cooling_specific_parts = []
    generic_plant_parts = []

    cooling_command_candidates = [
        'AHU_CTA_OUT_VF', 'AHU_OUT_VF', 'AHU_BATT_FREDDA', 'AHU_Batteria_Fredda',
        'CTA_OUT_VF', 'OUT_VF', 'CAL_GFC_SET_EI', 'GFC_SET_EI'
    ]
    for c in cooling_command_candidates:
        if c in df.columns:
            cooling_specific_parts.append(active_signal(df[c]))

    if 'T_CoolSupply' in df.columns:
        t_supply = pd.to_numeric(df['T_CoolSupply'], errors='coerce')
        t_room = pd.to_numeric(df['Room_T'], errors='coerce')
        cold_supply = (
            t_supply.notna() & t_room.notna() &
            (t_supply < (t_room - MIN_COOLING_DELTA_T)) &
            (t_supply <= MAX_REASONABLE_COOLING_SUPPLY_TEMP)
        )
        cooling_specific_parts.append(cold_supply)

    generic_plant_candidates = [
        'AHU_Stato', 'Stato',
        'AHU_CTA_MOD_IM', 'AHU_MOD_IM', 'CTA_MOD_IM',
        'AHU_CTA_OUT_VM', 'AHU_OUT_VM', 'CTA_OUT_VM',
        'AHU_CTA_IN_PM2', 'AHU_IN_PM2', 'CTA_IN_PM2',
        'AHU_CTA_OUT_VR', 'AHU_OUT_VR', 'CTA_OUT_VR',
        'GF_PATT', 'SONDE_GF_PATT', private_value('inputs.literal_014'),
    ]
    for c in generic_plant_candidates:
        if c in df.columns:
            generic_plant_parts.append(active_signal(df[c]))

    def combine(parts):
        out = pd.Series(False, index=df.index)
        for sig in parts:
            out = out | sig.fillna(False)
        return out

    if len(cooling_specific_parts) > 0:
        return combine(cooling_specific_parts), 'cooling_specific_or_cold_supply'
    if len(generic_plant_parts) > 0:
        return combine(generic_plant_parts), 'generic_plant_on_fallback_no_specific_cooling_signal'
    return pd.Series(True, index=df.index), 'no_explicit_signal_fallback'

def extract_cooling_rampup_periods(df, max_duration_minutes=None):
    """Detect cooling recovery events from cleaned observations at their original
    timestamps.

    Select the first eligible morning row with sufficient fan signal and
    temperature above the target allowance. Select the first subsequent row with a
    negative observed temperature increment and temperature within the endpoint
    allowance. Apply the configured date, clock and duration filters. Duration
    labels use discrete observed boundaries; interpolation and future-confirmation
    checks are not applied to those labels.
    """
    mode = globals().get('COOLING_EVENT_EXTRACTION_MODE', 'dadras_exact')
    if mode != 'dadras_exact':
        raise ValueError(
            "This notebook version is configured for exact DrDadras cooling extraction only. "
            "Set COOLING_EVENT_EXTRACTION_MODE='dadras_exact'."
        )
    if max_duration_minutes is None:
        max_duration_minutes = float(MAX_COOLING_RAMPUP_MINUTES)
    max_duration_minutes = float(max_duration_minutes)
    if not np.isfinite(max_duration_minutes) or max_duration_minutes < MIN_RAMPUP_MINUTES:
        raise ValueError(f'Invalid cooling maximum duration: {max_duration_minutes}')

    df = df.copy().sort_values('timestamp').reset_index(drop=True)
    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')

    room_T = pd.to_numeric(df['Room_T'], errors='coerce')
    room_V = pd.to_numeric(df['Room_V'], errors='coerce')

    if 'Cooling_Target_for_detection' in df.columns:
        cooling_target = pd.to_numeric(df['Cooling_Target_for_detection'], errors='coerce')
    else:
        cooling_target = pd.Series(float(FIXED_COOLING_TARGET_C), index=df.index)

    cooling_start_threshold = cooling_target + COOLING_START_ABOVE_TARGET
    cooling_end_threshold = cooling_target + COOLING_END_ABOVE_TARGET
    t_lag = room_T.shift(1)
    delta_from_lag = room_T - t_lag

    ts = pd.to_datetime(df['timestamp'], errors='coerce')

    # Exact reference-study daily window: between_time('05:00', '9:30').
    in_dadras_daily_window = time_between_clock(
        ts,
        COOLING_MORNING_START,
        COOLING_MORNING_END,
        include_start=True,
        include_end=True,
    )

    # Exact reference-study start-time rule: idx.hthe < 8.
    # With 5-min data this is equivalent to 05:00 <= t < 08:00.
    before_dadras_latest_start = ts.dt.hour < 8

    if USE_COOLING_SEASON_DATE_FILTER:
        season_start = pd.to_datetime(COOLING_SEASON_START)
        season_end = pd.to_datetime(COOLING_SEASON_END)
        in_cooling_date_season = (ts >= season_start) & (ts <= season_end)
    else:
        in_cooling_date_season = pd.Series(True, index=df.index)

    if USE_COOLING_SEASON_MONTH_FILTER:
        in_cooling_month = ts.dt.month.isin(COOLING_MONTHS)
    else:
        in_cooling_month = pd.Series(True, index=df.index)

    fan_active = room_V > COOLING_MIN_ROOM_FAN_SIGNAL
    above_start = room_T > cooling_start_threshold

    # Exact reference-study start candidate. No cooling plant signal, no start slope, no
    # future confirmation.
    start_condition = (
        in_dadras_daily_window
        & before_dadras_latest_start
        & in_cooling_date_season
        & in_cooling_month
        & fan_active
        & above_start
    ).fillna(False)

    # Exact reference-study end condition. Slope is used only here.
    end_condition = (
        in_dadras_daily_window
        & in_cooling_date_season
        & in_cooling_month
        & (delta_from_lag < 0)
        & (room_T <= cooling_end_threshold)
    ).fillna(False)

    # Keep diagnostic columns used by later plotting / missed-event checks.
    df['cooling_candidate_in_window'] = in_dadras_daily_window
    df['cooling_candidate_before_latest_start'] = before_dadras_latest_start
    df['cooling_candidate_in_end_search_window'] = in_dadras_daily_window
    df['cooling_candidate_above_start_threshold'] = above_start
    df['cooling_candidate_active'] = True  # Reference-study exact method does not use plant/cooling-active evidence.
    df['cooling_candidate_fan_active'] = fan_active
    df['cooling_candidate_local_delta_C'] = delta_from_lag
    df['cooling_candidate_local_slope_required_C'] = 0.0
    df['cooling_candidate_local_slope_confirmed'] = delta_from_lag < 0
    df['cooling_candidate_future_drop_C'] = np.nan
    df['cooling_candidate_required_drop_C'] = np.nan
    df['cooling_candidate_noise_sigma_5min_C'] = np.nan
    df['cooling_candidate_confirmed_future_cooling'] = True
    df['cooling_candidate_confirmed_start'] = start_condition
    df['cooling_latest_start_rule'] = 'dadras_exact_idx_hour_lt_8'
    df['cooling_start_condition'] = start_condition
    df['cooling_end_condition_dadras_exact'] = end_condition
    df['cooling_target_used_for_detection'] = cooling_target
    df['cooling_start_threshold'] = cooling_start_threshold
    df['cooling_end_threshold'] = cooling_end_threshold
    df['cooling_T_minus_T_1lag'] = delta_from_lag

    rampup_indices = []
    events = []
    excluded_duration_candidates = []
    event_id = 0
    saved_event_dates = set()

    # Exact reference-study groups by calendar day within the 05:00-09:30 window.
    # We use index positions so attach_event_labels remains compatible.
    tmp = df.copy()
    tmp['_row_pos'] = np.arange(len(tmp))
    tmp = tmp[in_dadras_daily_window.fillna(False)].copy()

    for day, day_df in tmp.groupby(ts.loc[tmp.index].dt.date):
        if pd.isna(day):
            continue
        if ONE_EVENT_PER_ROOM_DAY and day in saved_event_dates:
            continue

        # Start: first idx where V>0.4, T>Target+0.5, idx.hour<8.
        start_rows = day_df[start_condition.loc[day_df.index].values]
        if len(start_rows) == 0:
            continue

        start_idx = int(start_rows['_row_pos'].iloc[0])
        start_time = pd.Timestamp(df.loc[start_idx, 'timestamp'])

        # End: first idx2 in this_df.loc[this_Start:] satisfying reference-study end
        # rule and idx2 > this_Start.
        after_start = day_df[day_df['_row_pos'] >= start_idx].copy()
        end_mask = end_condition.loc[after_start.index].values & (after_start['_row_pos'].values > start_idx)
        end_rows = after_start[end_mask]
        if len(end_rows) == 0:
            # Exact reference-study effectively skips failed days by try/except pass.
            continue

        end_idx = int(end_rows['_row_pos'].iloc[0])
        end_time = pd.Timestamp(df.loc[end_idx, 'timestamp'])
        duration_min = round((end_time - start_time).seconds / 60.0, 0)

        if duration_min < MIN_RAMPUP_MINUTES:
            continue
        if duration_min > max_duration_minutes:
            excluded_duration_candidates.append({
                'phase': 'cooling',
                'event_date': day,
                'start_time': start_time,
                'end_time': end_time,
                'duration_min': duration_min,
                'maximum_included_duration_min': max_duration_minutes,
                'exclusion_reason': 'duration_above_prespecified_operational_cooling_scope',
            })
            continue

        event_id += 1
        idx_range = list(range(start_idx, end_idx + 1))
        rampup_indices.extend(idx_range)

        start_temp = float(room_T.iloc[start_idx]) if pd.notna(room_T.iloc[start_idx]) else np.nan
        end_temp = float(room_T.iloc[end_idx]) if pd.notna(room_T.iloc[end_idx]) else np.nan
        target_start = float(cooling_target.iloc[start_idx]) if pd.notna(cooling_target.iloc[start_idx]) else np.nan
        target_end = float(cooling_target.iloc[end_idx]) if pd.notna(cooling_target.iloc[end_idx]) else np.nan

        # Record the consecutive-observation temperature increment used in the endpoint
        # rule as a diagnostic.
        temp_slope_5min_at_start = float(delta_from_lag.iloc[start_idx]) if pd.notna(delta_from_lag.iloc[start_idx]) else np.nan
        temp_slope_5min_at_end = float(delta_from_lag.iloc[end_idx]) if pd.notna(delta_from_lag.iloc[end_idx]) else np.nan

        end_slope_time = start_time + pd.Timedelta(minutes=30)
        future_rows = df[pd.to_datetime(df['timestamp']) >= end_slope_time]
        if len(future_rows) > 0:
            end_slope_idx = int(future_rows.index[0])
            temp30 = float(room_T.iloc[end_slope_idx]) if pd.notna(room_T.iloc[end_slope_idx]) else np.nan
            slope30 = (temp30 - start_temp) / 30.0 if pd.notna(temp30) and pd.notna(start_temp) else np.nan
        else:
            temp30 = np.nan
            slope30 = np.nan

        slope_all = (end_temp - start_temp) / duration_min if duration_min not in [0, np.nan] and duration_min != 0 else np.nan

        events.append({
            'event_id': event_id,
            'phase': 'cooling',
            'event_date': day,
            'start_time': start_time,
            'end_time': end_time,
            'duration_min': duration_min,
            'duration_target_min': duration_min,
            'duration_continuous_min': duration_min,
            'duration_grid_min': duration_min,
            'end_grid_time': end_time,
            'end_crossing_time': end_time,
            'end_idx_grid': end_idx,
            'duration_estimation': 'dadras_exact_discrete_timestamp_no_interpolation',
            'observed_end_time': end_time,
            'actual_end_time': end_time,
            'observed_duration_min': duration_min,
            'actual_duration_min': duration_min,
            'indices': idx_range,
            'n_rows': len(idx_range),
            'availability_source': 'dadras_exact_fan_signal_only',
            'event_detection_source': 'localiin_dadras_exact_cooling_TV_fixed_target',
            'target_source_for_event_detection': 'fixed_or_resolved_cooling_target_used_by_dadras_exact_logic',
            'cooling_target_used_for_detection_at_start': target_start,
            'cooling_start_threshold_at_start': float(cooling_start_threshold.iloc[start_idx]) if pd.notna(cooling_start_threshold.iloc[start_idx]) else np.nan,
            'cooling_end_threshold_at_end': float(cooling_end_threshold.iloc[end_idx]) if pd.notna(cooling_end_threshold.iloc[end_idx]) else np.nan,
            'cooling_start_local_slope_lookback_min': np.nan,
            'cooling_start_local_delta_C': temp_slope_5min_at_start,
            'cooling_start_required_local_drop_C': np.nan,
            'cooling_start_future_confirmation_lookahead_min': np.nan,
            'cooling_start_future_drop_C': np.nan,
            'cooling_start_required_future_drop_C': np.nan,
            'cooling_noise_sigma_5min_C': np.nan,
            'dadras_exact_T_minus_T_1lag_at_end_C': temp_slope_5min_at_end,
            'dadras_exact_Temp30min': temp30,
            'dadras_exact_Slope30min_C_per_min': slope30,
            'dadras_exact_SlopeALL_C_per_min': slope_all,
            'room_T_at_start': start_temp,
            'room_T_at_end': end_temp,
            'target_at_start': target_start,
            'target_at_end': target_end,
            'target_crossing_threshold_at_end': float(cooling_end_threshold.iloc[end_idx]) if pd.notna(cooling_end_threshold.iloc[end_idx]) else np.nan,
            'start_rule': 'first idx in 05:00-09:30 with V>0.4, T>Target+0.5, idx.hour<8',
            'end_rule': 'first idx after start with T-T_1lag<0 and T<=Target+0.4',
        })
        saved_event_dates.add(day)

    rampup_df, labeled_events = attach_event_labels(df, events, phase='cooling')
    return rampup_df, labeled_events, excluded_duration_candidates

def attach_event_labels(df, events, phase):
    df = df.copy().sort_values('timestamp').reset_index(drop=True)
    event_col = f'{phase}_rampup_event_id'
    start_col = f'{phase}_rampup_start_time'
    end_col = f'{phase}_rampup_end_time'

    df[event_col] = pd.Series(pd.NA, index=df.index, dtype='Int64')
    df['Duration'] = np.nan
    df['Duration_continuous_min'] = np.nan
    df['Duration_grid_min'] = np.nan
    df[start_col] = pd.NaT
    df[end_col] = pd.NaT
    df[f'{phase}_rampup_end_grid_time'] = pd.NaT
    df['inside_detected_event'] = False

    for ev in events:
        for idx in ev['indices']:
            df.loc[idx, event_col] = ev['event_id']
            df.loc[idx, 'Duration'] = ev['duration_min']
            df.loc[idx, 'Duration_continuous_min'] = ev.get('duration_continuous_min', ev['duration_min'])
            df.loc[idx, 'Duration_grid_min'] = ev.get('duration_grid_min', np.nan)
            df.loc[idx, start_col] = ev['start_time']
            df.loc[idx, end_col] = ev['end_time']
            df.loc[idx, f'{phase}_rampup_end_grid_time'] = ev.get('end_grid_time', ev['end_time'])
            df.loc[idx, 'inside_detected_event'] = True

    rampup_df = df[df['inside_detected_event']].copy().reset_index(drop=True)
    return rampup_df, events


In [ ]:

# Dynamic full-event sample extraction

def common_calendar_cols():
    return [
        'hour_decimal', 'day_of_year', 'day_of_week', 'month', 'is_weekend',
        'doy_sin', 'doy_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos',
        'solar_declination_proxy_deg', 'daylight_hours_proxy', 'sunrise_hour_proxy', 'sunset_hour_proxy',
        'daylight_fraction_proxy', 'time_since_sunrise_h_proxy', 'time_to_sunset_h_proxy', 'solar_elevation_proxy',
    ]

def phase_feature_columns(phase):
    base = [
        'timestamp', 'target_timestamp', 'horizon_min', 'actual_target_gap_min',
        'room', 'room_label', 'phase', 'prefixes', 'selected_prefixes', 'event_id',
        'Room_T', 'target_Room_T', 'Room_S', 'Room_V', 'Offset', 'T_ext',
        'T_ave_other_rooms', 'Room_T_slope_prev',
        'rampup_start_time', 'rampup_end_time', 'rampup_end_grid_time', 'extraction_end_time',
        'post_end_export_rule', 'post_end_trend_continuation_min',
        'is_post_end_buffer_target', 'target_context', 'horizon_support_category',
        'event_detection_source', 'feature_extraction_source', 'duration_definition',
    ]

    if phase == 'heating':
        base += ['Heating_Target_for_detection', 'T_Boiler', 'MandBoiler']
        base += common_calendar_cols()
        base += [f'Room_T -{lag}min' for lag in ROOM_LAGS_MINUTES]
        base += [f'T_Boiler -{lag}min' for lag in BOILER_LAGS_MINUTES]
    else:
        base += [
            'T_CoolSupply',
            'Cooling_deltaT_supply_minus_room', 'Cooling_input_proxy',
        ]
        if globals().get('EXPORT_OPTIONAL_COOLING_SOURCE_FEATURES', False):
            base += OPTIONAL_COOLING_SOURCE_FEATURES
        if globals().get('EXPORT_OPTIONAL_SOLAR_WEATHER_FEATURES', False):
            base += ['SolarRadiation', 'RelativeHumidity', 'WindSpeed']
        base += common_calendar_cols()
        base += [f'Room_T -{lag}min' for lag in ROOM_LAGS_MINUTES]
        base += [f'T_CoolSupply -{lag}min' for lag in COOLING_SUPPLY_LAGS_MINUTES]
        if ADD_SOLAR_RADIATION_FEATURES_IF_AVAILABLE and globals().get('EXPORT_OPTIONAL_SOLAR_WEATHER_FEATURES', False):
            base += [f'SolarRadiation -{lag}min' for lag in SOLAR_LAGS_MINUTES]

    if ADD_EXTERNAL_TEMP_LAGS:
        base += [f'T_ext -{lag}min' for lag in EXT_LAGS_MINUTES]
    return base

def required_columns_for_phase(phase):
    """Return the variables required to define a supervised temperature row. Optional
    diagnostic and model-input fields are handled separately so their absence does
    not discard otherwise valid targets.
    """
    cols = [
        'timestamp', 'target_timestamp', 'horizon_min', 'target_Room_T',
        'Room_T', 'Room_V', 'event_id', 'rampup_start_time', 'rampup_end_time',
    ]
    if phase == 'heating':
        cols += ['Heating_Target_for_detection']
    elif phase == 'cooling':
        cols += ['Cooling_Target_for_detection'] if 'Cooling_Target_for_detection' in globals() else []
    return cols

def normalize_prefix_list(prefixes):
    """Return the selected sensor prefixes as a normalized list."""
    if isinstance(prefixes, (list, tuple, set)):
        out = [str(p).strip() for p in prefixes]
    else:
        out = [str(prefixes).strip()]
    out = [p for p in out if p and p != 'nan']
    return out

def primary_room_prefix(prefixes):
    """Return the primary prefix used for exported room identifiers and filenames."""
    pref = normalize_prefix_list(prefixes)
    if len(pref) == 0:
        raise ValueError('Empty prefix list for selected room.')
    return pref[0] if len(pref) == 1 else '__'.join(pref)

def assert_selected_prefixes_only(df, label):
    """Raise an error if an exported record contains a non-selected sensor prefix."""
    if df is None or len(df) == 0:
        return
    selected = set(SELECTED_ROOM_PREFIXES)
    observed = set()

    if 'prefixes' in df.columns:
        for val in df['prefixes'].dropna().astype(str):
            parts = [p.strip() for p in val.replace(';', ',').split(',') if p.strip()]
            observed.update(parts)
    elif 'selected_prefixes' in df.columns:
        for val in df['selected_prefixes'].dropna().astype(str):
            parts = [p.strip() for p in val.replace(';', ',').split(',') if p.strip()]
            observed.update(parts)
    elif 'room' in df.columns:
        observed.update(df['room'].dropna().astype(str).unique())

    bad = sorted(observed - selected)
    if bad:
        raise ValueError(f'{label} contains non-selected prefixes/rooms: {bad}')

def extract_full_event_samples(room_grid, events, room_name, prefixes, phase):
    """Create long-format targets at successive future horizons for eligible event
    prediction times.

    Keep the detected event endpoint as the duration label. Post-end temperature
    targets may continue while consecutive changes retain the expected sign,
    stopping at a sign break, missing value or configured continuation limit.
    """
    if len(events) == 0:
        return pd.DataFrame()

    df = room_grid.copy().sort_values('timestamp').reset_index(drop=True)
    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
    df = df.dropna(subset=['timestamp']).copy()
    df = df.drop_duplicates(subset=['timestamp'], keep='first').sort_values('timestamp').reset_index(drop=True)

    if df.empty:
        return pd.DataFrame()

    if 'Room_T' not in df.columns:
        raise RuntimeError(f'{phase} {room_name}: Room_T missing from room_grid.')
    df['Room_T'] = pd.to_numeric(df['Room_T'], errors='coerce')

    ts_values = pd.to_datetime(df['timestamp']).to_numpy(dtype='datetime64[ns]')
    sample_frames = []

    def _ceil_grid_time(t):
        """Return the first grid timestamp at or after the requested time, or NaT
        when unavailable.
        """
        if pd.isna(t):
            return pd.NaT
        t64 = np.datetime64(pd.Timestamp(t).to_datetime64(), 'ns')
        pos = int(np.searchsorted(ts_values, t64, side='left'))
        if pos >= len(ts_values):
            return pd.NaT
        return pd.Timestamp(ts_values[pos])

    def _floor_grid_time(t):
        """Return the last grid timestamp at or before the requested time, or NaT
        when unavailable.
        """
        if pd.isna(t):
            return pd.NaT
        t64 = np.datetime64(pd.Timestamp(t).to_datetime64(), 'ns')
        pos = int(np.searchsorted(ts_values, t64, side='right')) - 1
        if pos < 0:
            return pd.NaT
        return pd.Timestamp(ts_values[pos])

    def _pos_of_exact_grid_time(t, side='left'):
        if pd.isna(t):
            return None
        t64 = np.datetime64(pd.Timestamp(t).to_datetime64(), 'ns')
        if side == 'right_minus_one':
            return int(np.searchsorted(ts_values, t64, side='right')) - 1
        return int(np.searchsorted(ts_values, t64, side='left'))

    def _trend_sign_keep(dT, phase_name):
        tol = float(globals().get('POST_END_SLOPE_TOL_C_PER_STEP', 0.0))
        if not np.isfinite(dT):
            return False
        if phase_name == 'heating':
            return dT > tol
        if phase_name == 'cooling':
            return dT < -tol
        return False

    def _trend_based_extraction_end_pos(input_end_pos, input_end_grid_time):
        """Return the final eligible target position under the post-end
        directional-continuation rule.
        """
        mode = globals().get('POST_END_EXPORT_MODE', 'trend_sign_until_first_break')
        if mode != 'trend_sign_until_first_break':
            # Legacy fallback, kept only for reproducibility if user changes the mode.
            fixed_min = float(globals().get('POST_END_BUFFER_MIN', 0.0))
            fixed_end = pd.Timestamp(input_end_grid_time) + pd.Timedelta(minutes=fixed_min)
            fixed_grid = _floor_grid_time(fixed_end)
            if pd.isna(fixed_grid):
                return input_end_pos
            return max(input_end_pos, min(len(df) - 1, _pos_of_exact_grid_time(fixed_grid, side='right_minus_one')))

        max_extra = globals().get('POST_END_MAX_CONTINUATION_MIN', None)
        extraction_pos = int(input_end_pos)
        input_end_time = pd.Timestamp(input_end_grid_time)

        for k in range(input_end_pos + 1, len(df)):
            t_k = pd.Timestamp(df.loc[k, 'timestamp'])
            if max_extra is not None:
                extra_min = (t_k - input_end_time).total_seconds() / 60.0
                if extra_min > float(max_extra) + 1e-9:
                    break

            prev_T = pd.to_numeric(df.loc[k - 1, 'Room_T'], errors='coerce')
            curr_T = pd.to_numeric(df.loc[k, 'Room_T'], errors='coerce')
            if pd.isna(prev_T) or pd.isna(curr_T):
                break
            dT = float(curr_T) - float(prev_T)
            if _trend_sign_keep(dT, phase):
                extraction_pos = k
            else:
                break
        return extraction_pos

    for ev in events:
        start = pd.Timestamp(ev['start_time'])
        end = pd.Timestamp(ev.get('end_time', ev.get('observed_end_time')))
        end_grid = pd.Timestamp(ev.get('end_grid_time', end))
        if pd.isna(start) or pd.isna(end):
            continue

        # Retain observed event boundaries. Grid alignment determines eligible feature
        # rows only.
        input_end = max(end, end_grid)

        input_start_grid = _ceil_grid_time(start)
        input_end_grid = _floor_grid_time(input_end)

        if pd.isna(input_start_grid) or pd.isna(input_end_grid):
            continue

        input_start_pos = _pos_of_exact_grid_time(input_start_grid, side='left')
        input_end_pos = _pos_of_exact_grid_time(input_end_grid, side='right_minus_one')

        if input_start_pos is None or input_end_pos is None or input_start_pos > input_end_pos:
            continue

        extraction_end_pos = _trend_based_extraction_end_pos(input_end_pos, input_end_grid)
        extraction_end_grid = pd.Timestamp(df.loc[extraction_end_pos, 'timestamp'])
        extraction_end = extraction_end_grid
        post_end_trend_continuation_min = max(0.0, (extraction_end_grid - input_end_grid).total_seconds() / 60.0)

        rows = []
        for i in range(input_start_pos, input_end_pos + 1):
            row = df.iloc[i]
            t = pd.Timestamp(row['timestamp'])

            # Candidate target rows are future exact grid rows up to the trend-based
            # extraction_end.
            for j in range(i + 1, extraction_end_pos + 1):
                target_time = pd.Timestamp(df.loc[j, 'timestamp'])
                h_float = (target_time - t).total_seconds() / 60.0
                if h_float < STEP_MIN - 1e-9:
                    continue
                h_round = int(round(h_float))
                if abs(h_float - h_round) > 1e-6:
                    continue
                if h_round % STEP_MIN != 0:
                    continue

                target_val = df.loc[j, 'Room_T']
                if pd.isna(target_val):
                    continue

                out = row.to_dict()
                room_prefix_list = normalize_prefix_list(prefixes)
                room_prefix_id = primary_room_prefix(room_prefix_list)
                if not set(room_prefix_list).issubset(set(SELECTED_ROOM_PREFIXES)):
                    raise ValueError(f'Non-selected prefix attempted in sample extraction: {room_prefix_list}')

                is_post = bool(target_time > end)
                out['phase'] = phase
                out['room'] = room_prefix_id
                out['room_label'] = room_name
                out['prefixes'] = ','.join(room_prefix_list)
                out['selected_prefixes'] = ','.join(room_prefix_list)
                out['event_id'] = ev['event_id']
                out['event_duration_min'] = ev.get('duration_min', np.nan)
                out['target_timestamp'] = target_time
                out['horizon_min'] = h_round
                out['actual_target_gap_min'] = h_round
                out['target_Room_T'] = target_val
                out[f'Room_T +{h_round}min'] = target_val
                out[f'actual_target_gap_min +{h_round}min'] = h_round
                out['rampup_start_time'] = start
                out['rampup_end_time'] = end
                out['rampup_end_grid_time'] = end_grid
                out['extraction_end_time'] = extraction_end
                out['post_end_export_rule'] = globals().get('POST_END_EXPORT_MODE', 'trend_sign_until_first_break')
                out['post_end_trend_continuation_min'] = post_end_trend_continuation_min
                out['is_post_end_buffer_target'] = is_post  # legacy column name kept for training compatibility
                out['target_context'] = 'post_end_trend_continuation' if is_post else 'inside_event'
                out['event_detection_source'] = ev.get('event_detection_source', np.nan)
                out['feature_extraction_source'] = 'our_full_5min_feature_grid_positional_target_lookup_trend_post_end'
                out['duration_definition'] = ev.get('duration_estimation', 'discrete_event_duration')
                rows.append(out)

        if rows:
            sample_frames.append(pd.DataFrame(rows))

    if not sample_frames:
        return pd.DataFrame()

    samples = pd.concat(sample_frames, ignore_index=True)

    if 'target_context' not in samples.columns and 'is_post_end_buffer_target' in samples.columns:
        samples['target_context'] = np.where(samples['is_post_end_buffer_target'].astype(bool), 'post_end_trend_continuation', 'inside_event')

    if 'horizon_min' in samples.columns:
        h = pd.to_numeric(samples['horizon_min'], errors='coerce')
        if phase == 'cooling':
            samples['horizon_support_category'] = np.select(
                [h <= 30, (h > 30) & (h <= 60), h > 60],
                ['high_support_5_30min', 'medium_support_35_60min', 'low_support_over_60min'],
                default='unknown'
            )
        else:
            samples['horizon_support_category'] = np.select(
                [h <= 60, (h > 60) & (h <= 120), h > 120],
                ['high_support_5_60min', 'medium_support_65_120min', 'low_support_over_120min'],
                default='unknown'
            )

    forbidden_weight_cols = ['event_weight', 'room_event_weight', 'n_rows_in_event_samples', 'n_events_in_room_phase']
    forbidden_present = [c for c in forbidden_weight_cols if c in samples.columns]
    if forbidden_present:
        samples = samples.drop(columns=forbidden_present)

    cols = [c for c in phase_feature_columns(phase) if c in samples.columns]
    extra_cols = [
        c for c in samples.columns
        if c not in cols and (
            re.match(r'Room_T \+\d+min', c) or
            re.match(r'actual_target_gap_min \+\d+min', c)
        )
    ]
    samples = samples[cols + sorted(extra_cols)].copy()

    non_numeric = {
        'timestamp', 'target_timestamp', 'room', 'room_label', 'phase', 'prefixes', 'selected_prefixes',
        'rampup_start_time', 'rampup_end_time', 'rampup_end_grid_time', 'extraction_end_time',
        'target_context', 'horizon_support_category', 'event_detection_source', 'feature_extraction_source',
        'duration_definition', 'post_end_export_rule'
    }
    for c in samples.columns:
        if c not in non_numeric:
            samples[c] = pd.to_numeric(samples[c], errors='ignore')

    required = [c for c in required_columns_for_phase(phase) if c in samples.columns]
    if DROP_ONLY_REQUIRED_NA and len(required) > 0:
        before = len(samples)
        samples = samples.dropna(subset=required).reset_index(drop=True)
        dropped = before - len(samples)
        if dropped > 0:
            print(f'[{phase} {primary_room_prefix(prefixes)}] Required-only dropna removed {dropped} rows using {required}')
    else:
        samples = samples.reset_index(drop=True)

    assert_selected_prefixes_only(samples, f'{phase} samples for {room_name}')
    return samples

def _empty_schema_for_phase(phase):
    cols = list(dict.fromkeys(phase_feature_columns(phase)))
    return cols

def _events_schema():
    return [
        'phase', 'room', 'room_label', 'prefixes', 'event_id', 'event_date',
        'start_time', 'end_time', 'duration_min', 'duration_target_min',
        'duration_continuous_min', 'duration_grid_min', 'end_grid_time',
        'end_crossing_time', 'end_idx_grid', 'duration_estimation',
        'target_crossing_threshold_at_end', 'actual_end_time', 'actual_duration_min',
        'observed_end_time', 'observed_duration_min', 'extraction_end_time',
        'post_end_buffer_min', 'n_rows_inside_event_detection_frame',
        'availability_source', 'target_source_for_event_detection',
        'room_T_at_start', 'room_T_at_end', 'target_at_start', 'target_at_end',
        'event_detection_source', 'feature_extraction_source',
        'cooling_start_threshold_at_start', 'cooling_end_threshold_at_end',
        'dadras_exact_T_minus_T_1lag_at_end_C', 'dadras_spirit_T_minus_T_1lag_at_end_C',
        'start_rule', 'end_rule'
    ]

def _save_csv_with_columns(df, path, columns):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    if df is None or len(getattr(df, 'columns', [])) == 0:
        df = pd.DataFrame(columns=columns)
    else:
        df = df.copy()
        for c in columns:
            if c not in df.columns:
                df[c] = np.nan
    df.to_csv(path, index=False)

def validate_selected_room_long_coverage(phase, all_samples, events_df, phase_dir):
    """Record and optionally enforce temperature-target coverage for selected rooms
    with detected events.
    """
    os.makedirs(phase_dir, exist_ok=True)
    expected_selected = set(SELECTED_ROOM_PREFIXES)

    event_rooms = set()
    if events_df is not None and len(events_df) and 'room' in events_df.columns:
        event_rooms = set(events_df['room'].dropna().astype(str).unique())

    long_rooms = set()
    if all_samples is not None and len(all_samples) and 'room' in all_samples.columns:
        long_rooms = set(all_samples['room'].dropna().astype(str).unique())

    expected_with_events = sorted(event_rooms & expected_selected)
    missing_from_long = sorted(set(expected_with_events) - long_rooms)
    long_without_events = sorted(long_rooms - set(expected_with_events))
    selected_without_events = sorted(expected_selected - event_rooms)

    by_room_dir = os.path.join(phase_dir, 'by_room')
    expected_files = {
        r: os.path.join(by_room_dir, f'{clean_filename(r)}_{phase}_full_rampup_5min_LONG.csv')
        for r in expected_with_events
    }
    missing_files = sorted([r for r, p in expected_files.items() if not os.path.exists(p)])

    audit = pd.DataFrame([{
        'phase': phase,
        'n_selected_rooms_total': len(expected_selected),
        'n_event_rooms_selected': len(expected_with_events),
        'n_long_rooms': len(long_rooms),
        'event_rooms_selected': ','.join(expected_with_events),
        'long_rooms': ','.join(sorted(long_rooms)),
        'selected_rooms_without_detected_events': ','.join(selected_without_events),
        'event_rooms_missing_from_LONG': ','.join(missing_from_long),
        'event_rooms_missing_by_room_file': ','.join(missing_files),
        'long_rooms_without_events': ','.join(long_without_events),
        'require_selected_rooms_with_events_in_long': bool(globals().get('REQUIRE_SELECTED_ROOMS_WITH_EVENTS_IN_LONG', True)),
    }])
    audit_path = os.path.join(phase_dir, f'{phase}_selected_room_LONG_coverage_audit.csv')
    audit.to_csv(audit_path, index=False)

    # Room-level event/sample counts.
    rows = []
    all_rooms = sorted((event_rooms | long_rooms | expected_selected))
    for r in all_rooms:
        n_events = int((events_df['room'].astype(str) == r).sum()) if events_df is not None and len(events_df) and 'room' in events_df.columns else 0
        n_long = int((all_samples['room'].astype(str) == r).sum()) if all_samples is not None and len(all_samples) and 'room' in all_samples.columns else 0
        rows.append({
            'phase': phase,
            'room': r,
            'is_selected': r in expected_selected,
            'n_events': n_events,
            'n_LONG_rows': n_long,
            'has_LONG_rows': n_long > 0,
            'expected_by_room_file': expected_files.get(r, ''),
            'by_room_file_exists': os.path.exists(expected_files.get(r, '')) if r in expected_files else False,
        })
    room_audit = pd.DataFrame(rows)
    room_audit_path = os.path.join(phase_dir, f'{phase}_selected_room_LONG_coverage_by_room.csv')
    room_audit.to_csv(room_audit_path, index=False)

    print(f'{phase}: selected-room LONG coverage audit saved:', audit_path)
    try:
        display(audit)
        display(room_audit)
    except Exception:
        print(audit.to_string(index=False))
        print(room_audit.to_string(index=False))

    if globals().get('REQUIRE_SELECTED_ROOMS_WITH_EVENTS_IN_LONG', True):
        if missing_from_long:
            raise RuntimeError(
                f'{phase}: detected events exist for selected rooms, but no LONG samples were exported for: {missing_from_long}. '
                f'Coverage audit saved: {audit_path}. This must be fixed before training/duration prediction.'
            )
    return audit

# Remaining-duration anchor export

def remaining_duration_anchor_columns(phase):
    """Return the duration-row schema. These rows pair event boundaries with current
    features independently of future temperature-target availability.
    """
    base = [
        'phase', 'room', 'room_label', 'prefixes', 'selected_prefixes', 'event_id',
        'event_date',
        'timestamp', 'anchor_timestamp', 'feature_source_timestamp', 'anchor_kind',
        'asof_gap_min', 'is_event_start_anchor', 'is_inside_grid_anchor',
        'remaining_duration_min', 'true_remaining_duration_min', 'true_total_duration_min',
        'elapsed_since_start_min',
        'rampup_start_time', 'rampup_end_time', 'rampup_end_grid_time', 'extraction_end_time',
        'duration_definition', 'event_detection_source', 'feature_extraction_source',
        'target_source_for_event_detection',
        'Room_T', 'Room_S', 'Room_V', 'Offset', 'T_ext', 'T_ave_other_rooms', 'Room_T_slope_prev',
    ]
    if phase == 'heating':
        base += ['Heating_Target_for_detection', 'T_Boiler', 'MandBoiler']
        base += common_calendar_cols()
        base += [f'Room_T -{lag}min' for lag in ROOM_LAGS_MINUTES]
        base += [f'T_Boiler -{lag}min' for lag in BOILER_LAGS_MINUTES]
    else:
        base += [
            'Cooling_Target_for_detection', 'T_CoolSupply',
            'Cooling_deltaT_supply_minus_room', 'Cooling_input_proxy',
        ]
        if globals().get('EXPORT_OPTIONAL_COOLING_SOURCE_FEATURES', False):
            base += OPTIONAL_COOLING_SOURCE_FEATURES
        if globals().get('EXPORT_OPTIONAL_SOLAR_WEATHER_FEATURES', False):
            base += ['SolarRadiation', 'RelativeHumidity', 'WindSpeed']
        base += common_calendar_cols()
        base += [f'Room_T -{lag}min' for lag in ROOM_LAGS_MINUTES]
        base += [f'T_CoolSupply -{lag}min' for lag in COOLING_SUPPLY_LAGS_MINUTES]
    if globals().get('ADD_EXTERNAL_TEMP_LAGS', False):
        base += [f'T_ext -{lag}min' for lag in EXT_LAGS_MINUTES]
    return list(dict.fromkeys(base))

def _empty_duration_anchor_schema(phase):
    return remaining_duration_anchor_columns(phase)

def _event_time_value(ev, key, fallback=None):
    val = ev.get(key, fallback)
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return pd.NaT
    return pd.to_datetime(val, errors='coerce')

def _copy_anchor_feature_row(grid_row, phase, room_name, room_prefix_list):
    row = grid_row.to_dict()
    row['room'] = primary_room_prefix(room_prefix_list)
    row['room_label'] = room_name
    row['phase'] = phase
    row['prefixes'] = ', '.join(room_prefix_list)
    row['selected_prefixes'] = ', '.join(room_prefix_list)
    return row

def build_remaining_duration_anchors_from_events_and_grid(room_grid, events, room_name, room_prefix_list, phase, target_source):
    """Create startup and within-event duration prediction rows from event boundaries
    and current feature rows. No valid future temperature target is required,
    preserving coverage for short events.
    """
    if not globals().get('SAVE_REMAINING_DURATION_ANCHORS', True):
        return pd.DataFrame(columns=_empty_duration_anchor_schema(phase))
    if events is None or len(events) == 0:
        return pd.DataFrame(columns=_empty_duration_anchor_schema(phase))

    df = room_grid.copy().sort_values('timestamp').drop_duplicates(subset=['timestamp'], keep='first').reset_index(drop=True)
    if 'timestamp' not in df.columns or len(df) == 0:
        return pd.DataFrame(columns=_empty_duration_anchor_schema(phase))
    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
    df = df.dropna(subset=['timestamp']).sort_values('timestamp').reset_index(drop=True)
    if len(df) == 0:
        return pd.DataFrame(columns=_empty_duration_anchor_schema(phase))

    ts = df['timestamp'].values.astype('datetime64[ns]')
    max_asof_gap = float(globals().get('DURATION_ANCHOR_MAX_ASOF_GAP_MIN', 5.0))
    rows = []

    def floor_pos(t):
        if pd.isna(t):
            return None
        pos = int(np.searchsorted(ts, np.datetime64(pd.Timestamp(t).to_datetime64(), 'ns'), side='right')) - 1
        if pos < 0 or pos >= len(df):
            return None
        return pos

    def add_anchor(ev, anchor_time, source_pos, anchor_kind):
        if source_pos is None or source_pos < 0 or source_pos >= len(df):
            return False
        start = _event_time_value(ev, 'start_time')
        end = _event_time_value(ev, 'end_time')
        end_grid = _event_time_value(ev, 'end_grid_time', end)
        if pd.isna(start) or pd.isna(end) or pd.isna(anchor_time):
            return False
        remaining = (end - pd.Timestamp(anchor_time)).total_seconds() / 60.0
        elapsed = (pd.Timestamp(anchor_time) - start).total_seconds() / 60.0
        total = (end - start).total_seconds() / 60.0
        if not np.isfinite(remaining) or remaining < -1e-9:
            return False
        src_time = pd.Timestamp(df.iloc[source_pos]['timestamp'])
        asof_gap = abs((pd.Timestamp(anchor_time) - src_time).total_seconds()) / 60.0
        if anchor_kind == 'event_start_asof' and asof_gap > max_asof_gap + 1e-9:
            return False

        out = _copy_anchor_feature_row(df.iloc[source_pos], phase, room_name, room_prefix_list)
        out.update({
            'event_id': ev.get('event_id', np.nan),
            'event_date': ev.get('event_date', start.date() if not pd.isna(start) else None),
            # Keep timestamp as anchor timestamp for duration modeling;
            # feature_source_timestamp records where features came from.
            'timestamp': pd.Timestamp(anchor_time),
            'anchor_timestamp': pd.Timestamp(anchor_time),
            'feature_source_timestamp': src_time,
            'anchor_kind': anchor_kind,
            'asof_gap_min': float(asof_gap),
            'is_event_start_anchor': bool(anchor_kind == 'event_start_asof'),
            'is_inside_grid_anchor': bool(anchor_kind == 'inside_grid'),
            'remaining_duration_min': float(max(0.0, remaining)),
            'true_remaining_duration_min': float(max(0.0, remaining)),
            'true_total_duration_min': float(total),
            'elapsed_since_start_min': float(elapsed),
            'rampup_start_time': start,
            'rampup_end_time': end,
            'rampup_end_grid_time': end_grid,
            'extraction_end_time': (end_grid + pd.Timedelta(minutes=POST_END_BUFFER_MIN)) if not pd.isna(end_grid) else (end + pd.Timedelta(minutes=POST_END_BUFFER_MIN)),
            'duration_definition': 'remaining_duration_from_event_table_to_current_anchor',
            'event_detection_source': (ev.get('event_detection_source', 'final_feature_grid_fixed_room_target_heating_TV') if phase == 'heating' else ev.get('event_detection_source', 'localiin_dadras_style_TSV_only')),
            'feature_extraction_source': 'event_table_plus_current_5min_feature_grid_duration_anchor',
            'target_source_for_event_detection': target_source,
        })
        if phase == 'heating' and 'Heating_Target_for_detection' not in out:
            heating_target_value, _ = resolve_heating_target(room_name, room_prefix_list)
            out['Heating_Target_for_detection'] = float(heating_target_value)
        rows.append(out)
        return True

    for ev in events:
        start = _event_time_value(ev, 'start_time')
        end = _event_time_value(ev, 'end_time')
        if pd.isna(start) or pd.isna(end) or end <= start:
            continue

        # Always add one event-start anchor using the latest available 5-min feature row
        # at or before start.
        if globals().get('ALWAYS_ADD_EVENT_START_ASOF_ANCHOR', True):
            add_anchor(ev, start, floor_pos(start), 'event_start_asof')

        if globals().get('ADD_INSIDE_GRID_DURATION_ANCHORS', True):
            start_np = np.datetime64(start.to_datetime64(), 'ns')
            end_np = np.datetime64(end.to_datetime64(), 'ns')
            # Strictly after event start prevents duplicate exact-grid event-start
            # anchors.
            left = int(np.searchsorted(ts, start_np, side='right'))
            if globals().get('DURATION_ANCHOR_INCLUDE_EVENT_END_GRID_ROW', False):
                right = int(np.searchsorted(ts, end_np, side='right'))
            else:
                right = int(np.searchsorted(ts, end_np, side='left'))
            for pos in range(max(0, left), min(len(df), right)):
                anchor_time = pd.Timestamp(df.iloc[pos]['timestamp'])
                add_anchor(ev, anchor_time, pos, 'inside_grid')

    if not rows:
        return pd.DataFrame(columns=_empty_duration_anchor_schema(phase))

    anchors = pd.DataFrame(rows)
    # Deduplicate exact duplicates while preserving event-start anchors.
    dedup_cols = [c for c in ['phase', 'room', 'event_id', 'anchor_timestamp', 'anchor_kind'] if c in anchors.columns]
    anchors = anchors.drop_duplicates(subset=dedup_cols, keep='first').reset_index(drop=True)

    # Keep reference columns first, then any extra current feature/audit columns.
    canonical = [c for c in remaining_duration_anchor_columns(phase) if c in anchors.columns]
    extra = [c for c in anchors.columns if c not in canonical]
    anchors = anchors[canonical + sorted(extra)].copy()

    for c in ['timestamp', 'anchor_timestamp', 'feature_source_timestamp', 'rampup_start_time', 'rampup_end_time', 'rampup_end_grid_time', 'extraction_end_time']:
        if c in anchors.columns:
            anchors[c] = pd.to_datetime(anchors[c], errors='coerce')
    for c in ['remaining_duration_min', 'true_remaining_duration_min', 'true_total_duration_min', 'elapsed_since_start_min', 'asof_gap_min']:
        if c in anchors.columns:
            anchors[c] = pd.to_numeric(anchors[c], errors='coerce')

    assert_selected_prefixes_only(anchors, f'{phase} remaining-duration anchors for {room_name}')
    return anchors

def save_remaining_duration_anchor_outputs(phase, duration_anchors_df, events_df, phase_dir):
    """Save duration rows by operating mode and room, together with event-coverage
    diagnostics.
    """
    os.makedirs(phase_dir, exist_ok=True)
    by_room_dir = os.path.join(phase_dir, 'by_room')
    os.makedirs(by_room_dir, exist_ok=True)

    anchor_path = os.path.join(phase_dir, f'{phase}_remaining_duration_anchors_5min.csv')
    _save_csv_with_columns(duration_anchors_df, anchor_path, _empty_duration_anchor_schema(phase))

    room_paths = []
    if duration_anchors_df is not None and len(duration_anchors_df) > 0 and 'room' in duration_anchors_df.columns:
        for room_prefix, r_df in duration_anchors_df.groupby('room'):
            if str(room_prefix) not in set(SELECTED_ROOM_PREFIXES):
                raise ValueError(f'Attempted to save non-selected duration-anchor by-room file: {room_prefix}')
            r_path = os.path.join(by_room_dir, f'{clean_filename(room_prefix)}_{phase}_remaining_duration_anchors_5min.csv')
            r_df.to_csv(r_path, index=False)
            room_paths.append(r_path)

    event_keys = set()
    anchor_event_keys = set()
    event_work = events_df.copy() if events_df is not None else pd.DataFrame()
    anchor_work = duration_anchors_df.copy() if duration_anchors_df is not None else pd.DataFrame()
    if len(event_work) and {'room', 'event_id'}.issubset(event_work.columns):
        event_keys = set((event_work['room'].astype(str) + '|' + event_work['event_id'].astype(str)).tolist())
    if len(anchor_work) and {'room', 'event_id'}.issubset(anchor_work.columns):
        anchor_event_keys = set((anchor_work['room'].astype(str) + '|' + anchor_work['event_id'].astype(str)).tolist())

    missing_event_keys = sorted(event_keys - anchor_event_keys)
    extra_anchor_keys = sorted(anchor_event_keys - event_keys)

    coverage = pd.DataFrame([{
        'phase': phase,
        'n_event_table_events': len(event_keys),
        'n_duration_anchor_events': len(anchor_event_keys),
        'event_coverage_fraction': (len(anchor_event_keys & event_keys) / len(event_keys)) if len(event_keys) else np.nan,
        'n_events_missing_duration_anchor': len(missing_event_keys),
        'n_extra_anchor_events_not_in_event_table': len(extra_anchor_keys),
        'n_duration_anchor_rows': int(len(duration_anchors_df)) if duration_anchors_df is not None else 0,
        'n_duration_anchor_rooms': int(duration_anchors_df['room'].nunique()) if duration_anchors_df is not None and len(duration_anchors_df) and 'room' in duration_anchors_df.columns else 0,
        'require_duration_anchor_for_each_event': bool(globals().get('REQUIRE_DURATION_ANCHOR_FOR_EACH_EVENT', True)),
    }])
    coverage_path = os.path.join(phase_dir, f'{phase}_remaining_duration_anchor_event_coverage_audit.csv')
    coverage.to_csv(coverage_path, index=False)

    missing_path = os.path.join(phase_dir, f'{phase}_events_missing_remaining_duration_anchor.csv')
    if missing_event_keys and len(event_work):
        event_work['_event_key_tmp'] = event_work['room'].astype(str) + '|' + event_work['event_id'].astype(str)
        event_work[event_work['_event_key_tmp'].isin(missing_event_keys)].drop(columns=['_event_key_tmp']).to_csv(missing_path, index=False)
    else:
        pd.DataFrame(columns=list(event_work.columns) if len(event_work) else ['phase', 'room', 'event_id']).to_csv(missing_path, index=False)

    room_rows = []
    rooms = sorted(set(event_work['room'].astype(str).unique()) if len(event_work) and 'room' in event_work.columns else set())
    rooms = sorted(set(rooms) | (set(anchor_work['room'].astype(str).unique()) if len(anchor_work) and 'room' in anchor_work.columns else set()))
    for r in rooms:
        ev_r = event_work[event_work['room'].astype(str) == r] if len(event_work) and 'room' in event_work.columns else pd.DataFrame()
        an_r = anchor_work[anchor_work['room'].astype(str) == r] if len(anchor_work) and 'room' in anchor_work.columns else pd.DataFrame()
        ev_keys_r = set((ev_r['room'].astype(str) + '|' + ev_r['event_id'].astype(str)).tolist()) if len(ev_r) and 'event_id' in ev_r.columns else set()
        an_keys_r = set((an_r['room'].astype(str) + '|' + an_r['event_id'].astype(str)).tolist()) if len(an_r) and 'event_id' in an_r.columns else set()
        room_rows.append({
            'phase': phase,
            'room': r,
            'n_event_table_events': len(ev_keys_r),
            'n_duration_anchor_events': len(an_keys_r),
            'event_coverage_fraction': (len(ev_keys_r & an_keys_r) / len(ev_keys_r)) if len(ev_keys_r) else np.nan,
            'n_duration_anchor_rows': len(an_r),
            'has_event_start_anchors': bool((an_r.get('anchor_kind', pd.Series(dtype=str)).astype(str) == 'event_start_asof').any()) if len(an_r) else False,
            'has_inside_grid_anchors': bool((an_r.get('anchor_kind', pd.Series(dtype=str)).astype(str) == 'inside_grid').any()) if len(an_r) else False,
        })
    room_cov = pd.DataFrame(room_rows)
    room_cov_path = os.path.join(phase_dir, f'{phase}_remaining_duration_anchor_coverage_by_room.csv')
    room_cov.to_csv(room_cov_path, index=False)

    print(f'{phase}: remaining-duration anchors saved:', anchor_path)
    print(f'{phase}: duration-anchor event coverage audit saved:', coverage_path)
    try:
        display(coverage)
        display(room_cov)
    except Exception:
        print(coverage.to_string(index=False))
        print(room_cov.to_string(index=False))

    if globals().get('REQUIRE_DURATION_ANCHOR_FOR_EACH_EVENT', True) and missing_event_keys:
        raise RuntimeError(
            f'{phase}: {len(missing_event_keys)} detected events have no remaining-duration anchor. '
            f'Missing-event audit saved: {missing_path}'
        )

    return {
        'remaining_duration_anchors': anchor_path,
        'remaining_duration_anchor_event_coverage_audit': coverage_path,
        'remaining_duration_anchor_coverage_by_room': room_cov_path,
        'remaining_duration_anchor_missing_events': missing_path,
        'n_remaining_duration_anchor_by_room_files': len(room_paths),
    }

def save_phase_outputs(phase, all_samples, events_df, summary_df, sync_df, duration_anchors_df=None):
    phase_dir = os.path.join(OUT_DIR, phase)
    os.makedirs(phase_dir, exist_ok=True)
    os.makedirs(os.path.join(phase_dir, 'by_horizon'), exist_ok=True)
    os.makedirs(os.path.join(phase_dir, 'by_room'), exist_ok=True)

    # Validate selected-room membership before writing outputs.
    assert_no_bad_sentinels(all_samples, f'{phase} all_samples before save')
    assert_no_bad_sentinels(events_df, f'{phase} events before save')
    assert_selected_prefixes_only(all_samples, f'{phase} all_samples')
    assert_selected_prefixes_only(events_df, f'{phase} events_df')
    assert_selected_prefixes_only(summary_df, f'{phase} summary_df')
    assert_selected_prefixes_only(sync_df, f'{phase} sync_df')
    forbidden_export_cols = ['event_weight', 'room_event_weight', 'n_rows_in_event_samples', 'n_events_in_room_phase']
    forbidden_export_present = [c for c in forbidden_export_cols if c in all_samples.columns]
    if forbidden_export_present:
        raise RuntimeError(f'{phase}: forbidden sample-weight columns still present before saving: {forbidden_export_present}')

    long_path = os.path.join(phase_dir, f'{phase}_full_rampup_5min_samples_LONG.csv')
    events_path = os.path.join(phase_dir, f'{phase}_events_5min_grid.csv')
    summary_path = os.path.join(phase_dir, f'{phase}_extraction_summary.csv')
    sync_path = os.path.join(phase_dir, f'{phase}_sync_diagnostics.csv')

    _save_csv_with_columns(all_samples, long_path, _empty_schema_for_phase(phase))
    _save_csv_with_columns(events_df, events_path, _events_schema())
    _save_csv_with_columns(summary_df, summary_path, ['phase', 'room', 'room_label', 'status', 'horizon_min', 'rows_saved', 'events_detected', 'sample_rows_generated', 'error'])
    _save_csv_with_columns(sync_df, sync_path, ['phase', 'room', 'source', 'n_matched', 'mean_abs_offset_min', 'max_abs_offset_min'])

    horizon_paths = []
    room_paths = []
    if all_samples is not None and len(all_samples) > 0:
        for h, h_df in all_samples.groupby('horizon_min'):
            assert_selected_prefixes_only(h_df, f'{phase} horizon {h}')
            h = int(h)
            h_dir = os.path.join(phase_dir, 'by_horizon', f'horizon_{h:03d}min')
            os.makedirs(h_dir, exist_ok=True)
            h_path = os.path.join(h_dir, f'{phase}_horizon_{h:03d}min_all_selected_prefixes.csv')
            h_df.to_csv(h_path, index=False)
            horizon_paths.append(h_path)

        for room_prefix, r_df in all_samples.groupby('room'):
            if str(room_prefix) not in set(SELECTED_ROOM_PREFIXES):
                raise ValueError(f'Attempted to save non-selected by-room file: {room_prefix}')
            assert_selected_prefixes_only(r_df, f'{phase} by-room {room_prefix}')
            r_path = os.path.join(phase_dir, 'by_room', f'{clean_filename(room_prefix)}_{phase}_full_rampup_5min_LONG.csv')
            r_df.to_csv(r_path, index=False)
            room_paths.append(r_path)

    # Final coverage audit after by-room files are written.
    coverage_audit = None
    if globals().get('SAVE_FULL_COVERAGE_AUDITS', True):
        coverage_audit = validate_selected_room_long_coverage(phase, all_samples, events_df, phase_dir)

    duration_anchor_paths = {}
    if globals().get('SAVE_REMAINING_DURATION_ANCHORS', True):
        if duration_anchors_df is None:
            duration_anchors_df = pd.DataFrame(columns=_empty_duration_anchor_schema(phase))
        duration_anchor_paths = save_remaining_duration_anchor_outputs(phase, duration_anchors_df, events_df, phase_dir)

    out_paths = {
        'long_samples': long_path,
        'events': events_path,
        'summary': summary_path,
        'sync_diagnostics': sync_path,
        'selected_room_LONG_coverage_audit': os.path.join(phase_dir, f'{phase}_selected_room_LONG_coverage_audit.csv'),
        'selected_room_LONG_coverage_by_room': os.path.join(phase_dir, f'{phase}_selected_room_LONG_coverage_by_room.csv'),
        'n_horizon_files': len(horizon_paths),
        'n_by_room_files': len(room_paths),
    }
    out_paths.update(duration_anchor_paths)
    return out_paths


In [ ]:
# Wrap the existing feature/export functions with the shared derivative representation;
# retain event-detection functions.

# Keep the base implementations for use by the wrappers below.
_legacy_create_heating_features_on_grid = create_heating_features_on_grid
_legacy_create_cooling_features_on_grid = create_cooling_features_on_grid
_legacy_phase_feature_columns = phase_feature_columns
_legacy_remaining_duration_anchor_columns = remaining_duration_anchor_columns
_legacy_extract_full_event_samples = extract_full_event_samples
_legacy_build_remaining_duration_anchors = build_remaining_duration_anchors_from_events_and_grid

def _numeric_series(df, col):
    if col not in df.columns:
        return pd.Series(np.nan, index=df.index, dtype=float)
    return pd.to_numeric(df[col], errors='coerce').astype(float)

def _central_derivatives(values: pd.Series, dt_min: float):
    """Calculate central first and second derivatives for offline rate targets and
    lagged derivative features.
    """
    y = pd.to_numeric(values, errors='coerce').astype(float)
    d1 = (y.shift(-1) - y.shift(1)) / (2.0 * dt_min)
    d2 = (y.shift(-1) - 2.0 * y + y.shift(1)) / (dt_min ** 2)
    return d1, d2

def _backward_second_order_derivatives(values: pd.Series, dt_min: float):
    """Estimate current first and second derivatives using only the current and
    preceding grid values. Availability of the underlying observations depends on
    grid reconstruction.
    """
    y = pd.to_numeric(values, errors='coerce').astype(float)
    d1 = (3.0*y - 4.0*y.shift(1) + y.shift(2)) / (2.0 * dt_min)
    d2 = (2.0*y - 5.0*y.shift(1) + 4.0*y.shift(2) - y.shift(3)) / (dt_min ** 2)
    # Use backward estimates where sufficient preceding grid values are available.
    d1_first = (y - y.shift(1)) / dt_min
    d1 = d1.where(d1.notna(), d1_first)
    d2_first = (y - 2.0*y.shift(1) + y.shift(2)) / (dt_min ** 2)
    d2 = d2.where(d2.notna(), d2_first)
    return d1, d2

def add_unified_derivative_state_features(df: pd.DataFrame, phase: str):
    """Add mode-aligned state variables and past-facing derivative features to the
    reconstructed grid.
    """
    out = df.copy().sort_values('timestamp').reset_index(drop=True)
    phase = str(phase).lower().strip()
    if phase not in {'heating', 'cooling'}:
        raise ValueError(phase)
    sgn = 1.0 if phase == 'heating' else -1.0
    out['thermal_mode_sign'] = sgn

    room_T = _numeric_series(out, 'Room_T')
    d1_c, d2_c = _central_derivatives(room_T, float(STEP_MIN))
    d1_now, d2_now = _backward_second_order_derivatives(room_T, float(STEP_MIN))

    # Retain central derivatives for future labels and lagged features; exclude the
    # current central derivative from model inputs.
    out['_Room_dTdt_central_label_C_per_min'] = d1_c
    out['_Room_d2Tdt2_central_label_C_per_min2'] = d2_c
    out['Room_dTdt_now_bwd2_C_per_min'] = d1_now
    out['Room_d2Tdt2_now_bwd2_C_per_min2'] = d2_now

    for lag in DERIVATIVE_HISTORY_LAGS_MINUTES:
        steps = int(round(float(lag) / float(STEP_MIN)))
        if steps < 1:
            raise ValueError(f'Derivative-history lag must be >= one grid step: {lag}')
        # For a five-minute lag, the central stencil ends at the current grid time.
        # This does not change the offline availability of the underlying interpolated
        # observations.
        out[f'Room_dTdt_central_-{lag}min_C_per_min'] = d1_c.shift(steps)
        out[f'Room_d2Tdt2_central_-{lag}min_C_per_min2'] = d2_c.shift(steps)

    if phase == 'heating':
        target = _numeric_series(out, 'Heating_Target_for_detection')
        source = _numeric_series(out, 'T_Boiler')
    else:
        target = _numeric_series(out, 'Cooling_Target_for_detection')
        if target.isna().all():
            target = pd.Series(float(FIXED_COOLING_TARGET_C), index=out.index, dtype=float)
        else:
            target = target.fillna(float(FIXED_COOLING_TARGET_C))
        source = _numeric_series(out, 'T_CoolSupply')

    out['Phase_Target_T_C'] = target
    out['Directed_Distance_to_Target_C'] = sgn * (target - room_T)
    out['Thermal_Source_T_C'] = source
    out['Directed_Source_Room_DeltaT_C'] = sgn * (source - room_T)
    out['Thermal_Input_Proxy'] = _numeric_series(out, 'Room_V') * out['Directed_Source_Room_DeltaT_C']

    # Source derivative history gives the shared NN recent forcing dynamics without
    # feeding raw source lags.
    src_d1_c, src_d2_c = _central_derivatives(source, float(STEP_MIN))
    for lag in SOURCE_DERIVATIVE_HISTORY_LAGS_MINUTES:
        steps = int(round(float(lag) / float(STEP_MIN)))
        out[f'ThermalSource_dTdt_central_-{lag}min_C_per_min'] = src_d1_c.shift(steps)
        out[f'ThermalSource_d2Tdt2_central_-{lag}min_C_per_min2'] = src_d2_c.shift(steps)

    return out

def _drop_legacy_raw_history_columns(df):
    # Export derivative histories rather than raw temperature-history lags.
    pats = [
        re.compile(r'^Room_T -\d+min$'),
        re.compile(r'^T_Boiler -\d+min$'),
        re.compile(r'^T_CoolSupply -\d+min$'),
    ]
    drop = [c for c in df.columns if c == 'Room_T_slope_prev' or any(p.match(str(c)) for p in pats)]
    return df.drop(columns=drop, errors='ignore')

def create_heating_features_on_grid(full_room_df):
    out = add_unified_derivative_state_features(
        _legacy_create_heating_features_on_grid(full_room_df), 'heating'
    )
    return _drop_legacy_raw_history_columns(out)

def create_cooling_features_on_grid(full_room_df):
    out = add_unified_derivative_state_features(
        _legacy_create_cooling_features_on_grid(full_room_df), 'cooling'
    )
    return _drop_legacy_raw_history_columns(out)

def _new_model_export_feature_columns():
    cols = [
        'thermal_mode_sign', 'Phase_Target_T_C', 'Directed_Distance_to_Target_C',
        'Thermal_Source_T_C', 'Directed_Source_Room_DeltaT_C', 'Thermal_Input_Proxy',
        'Room_dTdt_now_bwd2_C_per_min', 'Room_d2Tdt2_now_bwd2_C_per_min2',
    ]
    cols += [f'Room_dTdt_central_-{lag}min_C_per_min' for lag in DERIVATIVE_HISTORY_LAGS_MINUTES]
    cols += [f'Room_d2Tdt2_central_-{lag}min_C_per_min2' for lag in DERIVATIVE_HISTORY_LAGS_MINUTES]
    cols += [f'ThermalSource_dTdt_central_-{lag}min_C_per_min' for lag in SOURCE_DERIVATIVE_HISTORY_LAGS_MINUTES]
    cols += [f'ThermalSource_d2Tdt2_central_-{lag}min_C_per_min2' for lag in SOURCE_DERIVATIVE_HISTORY_LAGS_MINUTES]
    return cols

def _remove_legacy_history_names(cols):
    pats = [
        re.compile(r'^Room_T -\d+min$'),
        re.compile(r'^T_Boiler -\d+min$'),
        re.compile(r'^T_CoolSupply -\d+min$'),
    ]
    return [c for c in cols if c != 'Room_T_slope_prev' and not any(p.match(str(c)) for p in pats)]

def phase_feature_columns(phase):
    cols = _remove_legacy_history_names(list(_legacy_phase_feature_columns(phase))) + _new_model_export_feature_columns()
    return list(dict.fromkeys(cols))

def remaining_duration_anchor_columns(phase):
    cols = _remove_legacy_history_names(list(_legacy_remaining_duration_anchor_columns(phase))) + _new_model_export_feature_columns() + [
        'remaining_trajectory_quality_score',
        'remaining_forward_progress_C', 'remaining_reverse_motion_C',
        'remaining_total_variation_C', 'remaining_quality_n_steps',
    ]
    return list(dict.fromkeys(cols))

def _remaining_quality_from_grid(room_grid, anchor_time, end_time, phase):
    """Calculate expected-direction net progress divided by total temperature
    variation from the prediction time to the endpoint. The score uses future
    observations as training information and is excluded from predictors.
    """
    phase = str(phase).lower().strip()
    sgn = 1.0 if phase == 'heating' else -1.0
    g = room_grid[['timestamp', 'Room_T']].copy()
    g['timestamp'] = pd.to_datetime(g['timestamp'], errors='coerce')
    g['Room_T'] = pd.to_numeric(g['Room_T'], errors='coerce')
    a = pd.to_datetime(anchor_time, errors='coerce')
    e = pd.to_datetime(end_time, errors='coerce')
    if pd.isna(a) or pd.isna(e) or e <= a:
        return 0.0, 0.0, 0.0, 0.0, 0

    # Use observed 5-min grid points whose centers lie within the remaining interval.
    z = g[(g['timestamp'] >= a) & (g['timestamp'] <= e)].dropna().sort_values('timestamp')
    if len(z) < 2:
        # If the continuous end lies between grid points, include the first point after
        # end only
        # to recover the final observed step, without using it as a model feature.
        before = g[g['timestamp'] >= a].dropna().sort_values('timestamp')
        if len(before) >= 2:
            z = before.iloc[:2].copy()
    if len(z) < 2:
        return 0.0, 0.0, 0.0, 0.0, 0

    directed_steps = sgn * np.diff(z['Room_T'].to_numpy(dtype=float))
    directed_steps = directed_steps[np.isfinite(directed_steps)]
    if len(directed_steps) == 0:
        return 0.0, 0.0, 0.0, 0.0, 0
    forward = float(np.clip(directed_steps, 0.0, None).sum())
    reverse = float(np.clip(-directed_steps, 0.0, None).sum())
    total_var = forward + reverse
    if total_var <= float(REMAINING_SCORE_EPS):
        score = 0.0
    else:
        score = float(np.clip((forward - reverse) / (total_var + float(REMAINING_SCORE_EPS)), 0.0, 1.0))
    return score, forward, reverse, total_var, int(len(directed_steps))

def extract_full_event_samples(room_grid, events, room_name, prefixes, phase):
    samples = _legacy_extract_full_event_samples(room_grid, events, room_name, prefixes, phase)
    if samples is None or len(samples) == 0:
        return samples

    grid = room_grid.copy()
    grid['timestamp'] = pd.to_datetime(grid['timestamp'], errors='coerce')
    rate_map = (
        grid.dropna(subset=['timestamp'])
        .drop_duplicates('timestamp', keep='first')
        .set_index('timestamp')['_Room_dTdt_central_label_C_per_min']
        if '_Room_dTdt_central_label_C_per_min' in grid.columns else pd.Series(dtype=float)
    )
    samples['target_Room_dTdt_central_C_per_min'] = pd.to_datetime(samples['target_timestamp'], errors='coerce').map(rate_map)
    sgn = 1.0 if str(phase).lower() == 'heating' else -1.0
    samples['target_directed_rate_C_per_min'] = sgn * pd.to_numeric(samples['target_Room_dTdt_central_C_per_min'], errors='coerce')
    samples['remaining_time_to_end_min'] = (
        pd.to_datetime(samples['rampup_end_time'], errors='coerce') - pd.to_datetime(samples['timestamp'], errors='coerce')
    ).dt.total_seconds() / 60.0
    samples['remaining_time_to_end_min'] = samples['remaining_time_to_end_min'].clip(lower=0.0)

    # The score is constant over all future-horizon rows issued from the same current
    # timestamp.
    q_cache = {}
    q, fwd, rev, tv, ns = [], [], [], [], []
    for _, row in samples.iterrows():
        key = (str(row.get('event_id')), pd.Timestamp(row['timestamp']))
        if key not in q_cache:
            end_for_score = row.get('rampup_end_grid_time', row.get('rampup_end_time'))
            q_cache[key] = _remaining_quality_from_grid(room_grid, row['timestamp'], end_for_score, phase)
        vals = q_cache[key]
        q.append(vals[0]); fwd.append(vals[1]); rev.append(vals[2]); tv.append(vals[3]); ns.append(vals[4])
    samples['remaining_trajectory_quality_score'] = q
    samples['remaining_forward_progress_C'] = fwd
    samples['remaining_reverse_motion_C'] = rev
    samples['remaining_total_variation_C'] = tv
    samples['remaining_quality_n_steps'] = ns
    return samples

def build_remaining_duration_anchors_from_events_and_grid(room_grid, events, room_name, room_prefix_list, phase, target_source):
    anchors = _legacy_build_remaining_duration_anchors(
        room_grid=room_grid, events=events, room_name=room_name,
        room_prefix_list=room_prefix_list, phase=phase, target_source=target_source,
    )
    if anchors is None or len(anchors) == 0:
        return anchors
    vals = []
    for _, row in anchors.iterrows():
        end_for_score = row.get('rampup_end_grid_time', row.get('rampup_end_time'))
        vals.append(_remaining_quality_from_grid(room_grid, row['anchor_timestamp'], end_for_score, phase))
    anchors['remaining_trajectory_quality_score'] = [v[0] for v in vals]
    anchors['remaining_forward_progress_C'] = [v[1] for v in vals]
    anchors['remaining_reverse_motion_C'] = [v[2] for v in vals]
    anchors['remaining_total_variation_C'] = [v[3] for v in vals]
    anchors['remaining_quality_n_steps'] = [v[4] for v in vals]
    return anchors

print('NEW FRAMEWORK ACTIVE')
print('  shared NN model representation: derivative-state; no room identity')
print('  central derivative history lags:', DERIVATIVE_HISTORY_LAGS_MINUTES)
print('  current derivative stencil: second-order backward (causal)')
print('  future derivative target stencil: central (label only)')
print('  remaining-quality score: directional net progress / total variation')


In [ ]:

# Room mapping and selected-room filtering

ROOMS_ALL = private_value('nb01.rooms_all')

selected_set = set(SELECTED_ROOM_PREFIXES)
ROOMS = {
    room: [p for p in prefixes if p in selected_set]
    for room, prefixes in ROOMS_ALL.items()
    if any(p in selected_set for p in prefixes)
}

found_prefixes = sorted({p for prefixes in ROOMS.values() for p in prefixes})
missing_prefixes = sorted(selected_set - set(found_prefixes))

print('Selected mapped rooms:', len(ROOMS))
for room, prefixes in ROOMS.items():
    print(f'  {room}: {prefixes}')
print('Missing requested prefixes:', missing_prefixes)

In [ ]:


# Load the raw data:
#          A) LocaliIN observation-based event-detection rows
#          B) the full-ramp feature-grid source data

# Parse private room observations as UTC, convert to local time, retain the first
# duplicate, replace invalid readings, forward-fill and round.
# Keep unrelated source-column missingness from removing event observations.
# Use the room control signal for event detection and restrict required-column filtering
# to the current room.

print('\n' + '=' * 100)
print('A) Building LocaliIN DrDadras-style event-detection dataframe')
print('=' * 100)

# Read and clean LocaliIN exactly in the reference-study style.
df_LocaliIN_dadras = read_csv_indexed_dadras_style(LOCALI_FILE, private_value('inputs.literal_005'))
if df_LocaliIN_dadras is None:
    raise FileNotFoundError(private_value('inputs.literal_015'))

rows_localiin_raw = len(df_LocaliIN_dadras)

# Apply numeric cleaning while preserving the timestamp index.
df_LocaliIN_dadras = df_LocaliIN_dadras.where(df_LocaliIN_dadras > -50, np.nan).ffill().round(decimals=2)

# This is the ONLY dataframe used for event boundary detection.
locali_detection = reset_index_timestamp(df_LocaliIN_dadras)

print('\nLocaliIN DrDadras-style detection diagnostics:')
print('Rows after timestamp parsing/duplicate removal:', rows_localiin_raw)
print('Rows used for event detection:', len(locali_detection))
print('Time range:', locali_detection['timestamp'].min(), 'to', locali_detection['timestamp'].max())

selected_cols_check = []
for p in SELECTED_ROOM_PREFIXES:
    selected_cols_check.append({
        'room': p,
        'has_T': f'{p}_T' in locali_detection.columns,
        'has_S': f'{p}_S' in locali_detection.columns,
        'has_V': f'{p}_V' in locali_detection.columns,
        'n_T_notna': int(pd.to_numeric(locali_detection.get(f'{p}_T', pd.Series(dtype=float)), errors='coerce').notna().sum()) if f'{p}_T' in locali_detection.columns else 0,
        'n_S_notna': int(pd.to_numeric(locali_detection.get(f'{p}_S', pd.Series(dtype=float)), errors='coerce').notna().sum()) if f'{p}_S' in locali_detection.columns else 0,
        'n_V_notna': int(pd.to_numeric(locali_detection.get(f'{p}_V', pd.Series(dtype=float)), errors='coerce').notna().sum()) if f'{p}_V' in locali_detection.columns else 0,
    })
selected_cols_check_df = pd.DataFrame(selected_cols_check)
try:
    display(selected_cols_check_df)
except Exception:
    print(selected_cols_check_df)

missing_required = selected_cols_check_df[~selected_cols_check_df[['has_T', 'has_V']].all(axis=1)]
if len(missing_required):
    print('WARNING: some selected rooms miss required T or V columns for cooling event detection:')
    print(missing_required)

# Use auxiliary streams for features only; event boundaries use the room observations.

print('\n' + '=' * 100)
print('B) Preparing our feature-grid source data')
print('=' * 100)

# Reuse the cleaned room observations for features, then reconstruct their regular grid
# after detecting boundaries.
locali_features = reset_index_timestamp(df_LocaliIN_dadras)

# AHUIN: reference-study style for common external/control features.
df_AHUIN_dadras = read_csv_indexed_dadras_style(AHU_FILE, private_value('inputs.literal_002'))
if df_AHUIN_dadras is not None:
    ahu_cols = keep_existing_columns(df_AHUIN_dadras, ['Stato', 'T_mand2', 'U_mand', 'T_est'])
    df_AHUIN_dadras = df_AHUIN_dadras[ahu_cols].copy()
    if 'Stato' in df_AHUIN_dadras.columns:
        df_AHUIN_dadras.loc[df_AHUIN_dadras['Stato'] == 3, 'Stato'] = 0
        df_AHUIN_dadras.loc[(df_AHUIN_dadras['Stato'] != 0) & (df_AHUIN_dadras['Stato'] != 1), 'Stato'] = np.nan
    df_AHUIN_dadras = df_AHUIN_dadras.where(df_AHUIN_dadras > -50, np.nan)
    ahu_features = reset_index_timestamp(df_AHUIN_dadras)
else:
    ahu_features = pd.DataFrame(columns=['timestamp'])

# Read auxiliary streams with the same UTC-to-local timestamp convention before
# alignment.
cal_features = (
    read_feature_source_dadras_time(CAL_FILE, private_value('inputs.literal_003'))
    if os.path.exists(CAL_FILE)
    else pd.DataFrame(columns=['timestamp'])
)

valve_features = (
    read_feature_source_dadras_time(VALVE_FILE, private_value('inputs.literal_004'))
    if os.path.exists(VALVE_FILE)
    else pd.DataFrame(columns=['timestamp'])
)

locali_features = clean_physical_sentinels(locali_features, 'locali_features_after_dadras_cleaning', verbose=False)
ahu_features = clean_physical_sentinels(ahu_features, 'ahu_features_after_dadras_cleaning', verbose=False)

source_time_alignment_df = audit_source_time_alignment({
    'locali_detection': locali_detection,
    'locali_features': locali_features,
    'ahu_features': ahu_features,
    'cal_features': cal_features,
    'valve_features': valve_features,
}, out_dir=OUT_DIR)

# Compatibility aliases used by the existing feature-building loop.
locali = locali_features
ahu = prefix_columns(ahu_features, 'AHU_')
cal = prefix_columns(cal_features, 'CAL_')
valve = valve_features.copy()

manual_cooling_targets = load_manual_cooling_targets(COOLING_TARGET_FILE)

all_room_temp_cols_detection = [c for c in locali_detection.columns if re.match(r'^P\d+_\d+_T$', c)]
all_room_temp_cols_features = [c for c in locali_features.columns if re.match(r'^P\d+_\d+_T$', c)]
all_room_temp_cols = all_room_temp_cols_features

print('\nDetected room temperature columns for event detection:', len(all_room_temp_cols_detection))
print('Example detection temperature columns:', all_room_temp_cols_detection[:10])
print('Detected room temperature columns for feature grid:', len(all_room_temp_cols_features))
print('Example feature temperature columns:', all_room_temp_cols_features[:10])

print('\nFeature source dataframes:')
print('locali_detection:', locali_detection.shape)
print('locali_features:', locali_features.shape)
print('ahu:', ahu.shape)
print('cal:', cal.shape)
print('valve:', valve.shape)

print('\nLoaded data successfully: LocaliIN Dadras-style event rows + our feature sources.')


In [ ]:

# Main extraction loop
#           Event boundaries from observation-based rows;
#           full-ramp samples from the feature grid

all_phase_outputs = {}

def append_csv_no_duplicate_header(df, path):
    if df is None or len(df) == 0:
        return
    os.makedirs(os.path.dirname(path), exist_ok=True)
    write_header = not os.path.exists(path)
    df.to_csv(path, mode='a', header=write_header, index=False)

def build_room_level_dataframe(base_df, room_name, room_prefix_list, all_temp_cols_for_average, phase, source_label):
    """Construct room temperature, setpoint, fan/control, offset and
    other-room-average fields from the selected source columns.
    """
    room_cols = get_room_columns(base_df, room_prefix_list)
    if len(room_cols) <= 1:
        raise ValueError(f'No matching room columns found in {source_label}.')

    room_df = base_df[room_cols].copy()

    T_cols = [f'{p}_T' for p in room_prefix_list if f'{p}_T' in room_df.columns]
    S_cols = [f'{p}_S' for p in room_prefix_list if f'{p}_S' in room_df.columns]
    V_cols = [f'{p}_V' for p in room_prefix_list if f'{p}_V' in room_df.columns]
    O_cols = [f'{p}_O' for p in room_prefix_list if f'{p}_O' in room_df.columns]

    # Event detection needs T and V for both modes. Heating target is fixed by
    # HEATING_ROOM_TARGET_C.
    # Room_S/O are diagnostics/export features only and must never remove valid event
    # rows.
    if len(T_cols) == 0 or len(V_cols) == 0:
        raise ValueError(
            f'Missing required room variables in {source_label}. '
            f'T={T_cols}, S={S_cols}, V={V_cols}, O={O_cols}'
        )

    room_df['Room_T'] = room_df[T_cols].apply(pd.to_numeric, errors='coerce').mean(axis=1)
    room_df['Room_S'] = room_df[S_cols].apply(pd.to_numeric, errors='coerce').mean(axis=1) if len(S_cols) else np.nan
    room_df['Room_V'] = room_df[V_cols].apply(pd.to_numeric, errors='coerce').mean(axis=1)
    room_df['Room_O'] = room_df[O_cols].apply(pd.to_numeric, errors='coerce').mean(axis=1) if len(O_cols) else np.nan

    if phase == 'heating':
        heating_target_value, heating_target_source = resolve_heating_target(room_name, room_prefix_list)
        room_df['Heating_Target_for_detection'] = float(heating_target_value)

    # Filter required detection fields within each room rather than across all building
    # columns.
    if 'dadras_style' in str(source_label).lower() and 'detection' in str(source_label).lower():
        required_detection_cols = ['timestamp', 'Room_T', 'Room_V']
        if phase == 'heating':
            required_detection_cols.append('Heating_Target_for_detection')
        before_room_drop = len(room_df)
        room_df = room_df.dropna(subset=required_detection_cols).copy()
        print(f'[{source_label}] Room-specific detection dropna: {before_room_drop} -> {len(room_df)} rows using {required_detection_cols}')

    other_room_temp_cols = [c for c in all_temp_cols_for_average if c not in T_cols]
    if len(other_room_temp_cols) > 0:
        room_df['T_ave_other_rooms'] = base_df[other_room_temp_cols].apply(pd.to_numeric, errors='coerce').mean(axis=1)
    else:
        room_df['T_ave_other_rooms'] = np.nan

    if phase == 'cooling':
        room_df['Cooling_Target_for_detection'], target_source = resolve_cooling_target(
            room_name, room_prefix_list, room_df['Room_S'], manual_cooling_targets
        )
    else:
        target_source = heating_target_source

    info = {
        'n_source_rows': len(base_df),
        'after_room_variable_creation_rows': len(room_df),
        'n_room_T_cols': len(T_cols),
        'n_other_rooms_in_building_average': len(other_room_temp_cols),
        'target_source_for_event_detection': target_source,
        'source_label': source_label,
    }
    return room_df, target_source, info

for phase in PHASES_TO_RUN:
    print('\n' + '#' * 100)
    print('PHASE:', phase.upper())
    print('#' * 100)

    phase_dir = os.path.join(OUT_DIR, phase)
    if CLEAN_OUTPUT_DIR_BEFORE_RUN and os.path.exists(phase_dir):
        print('Cleaning old phase output directory:', phase_dir)
        shutil.rmtree(phase_dir)
    os.makedirs(phase_dir, exist_ok=True)

    phase_sample_frames = []
    phase_duration_anchor_frames = []
    event_rows = []
    summary_rows = []
    sync_diagnostic_rows = []
    phase_duration_exclusion_rows = []

    for room_name, prefixes in ROOMS.items():
        room_prefix_list = normalize_prefix_list(prefixes)
        if not set(room_prefix_list).issubset(set(SELECTED_ROOM_PREFIXES)):
            print('SKIPPING non-selected mapping:', room_name, '| prefixes:', room_prefix_list)
            continue
        room_prefix_id = primary_room_prefix(room_prefix_list)

        print('\n' + '=' * 90)
        print('ROOM PREFIX:', room_prefix_id, '| label:', room_name, '| selected prefixes:', room_prefix_list)

        room_count_info = {
            'phase': phase,
            'room': room_prefix_id,
            'room_label': room_name,
            'prefixes': ', '.join(room_prefix_list),
            'event_detection_source': 'localiin_dadras_style_TSV_only',
            'feature_extraction_source': 'our_regular_5min_feature_grid',
            'raw_detection_rows': len(locali_detection),
            'raw_feature_locali_rows': len(locali_features),
        }

        try:
            
            # 1) Event boundary detection on observation-based prepared rows.
            
            detection_df, target_source, detection_info = build_room_level_dataframe(
                base_df=locali_detection,
                room_name=room_name,
                room_prefix_list=room_prefix_list,
                all_temp_cols_for_average=all_room_temp_cols_detection,
                phase=phase,
                source_label='locali_detection_dadras_style_TSV_only',
            )
            room_count_info.update({f'detection_{k}': v for k, v in detection_info.items()})
            room_count_info['target_source_for_event_detection'] = target_source

            detection_df = detection_df.sort_values('timestamp').drop_duplicates(subset=['timestamp'], keep='first').reset_index(drop=True)
            detection_df['grid_preparation_mode'] = 'localiin_dadras_style_event_detection_rows_no_independent_interpolation'

            preliminary_detection_events = []
            events = []
            cooling_duration_exclusions = []

            if phase == 'heating':
                # Apply the heating rules to cleaned original observations.
                detection_for_events = detection_df.copy()
                _, events = extract_heating_rampup_periods(detection_for_events)
                room_count_info['events_detected'] = len(events)
                print('Detected heating events from symmetric DrDadras-style rows:', len(events))
            elif phase == 'cooling':
                # Cooling remains exact observation-based detection to preserve the
                # event boundary rules.
                detection_for_events = detection_df.copy()
                _, events, cooling_duration_exclusions = extract_cooling_rampup_periods(
                    detection_for_events, max_duration_minutes=MAX_COOLING_RAMPUP_MINUTES
                )
                for exclusion in cooling_duration_exclusions:
                    phase_duration_exclusion_rows.append({
                        **exclusion,
                        'room': room_prefix_id,
                        'room_label': room_name,
                        'prefixes': ', '.join(room_prefix_list),
                    })
                room_count_info['events_excluded_above_phase_max_duration'] = len(cooling_duration_exclusions)
                room_count_info['events_detected'] = len(events)
                print(
                    'Detected cooling events from exact DrDadras-style rows:', len(events),
                    '| excluded above', MAX_COOLING_RAMPUP_MINUTES, 'min:', len(cooling_duration_exclusions)
                )
            else:
                raise ValueError(f'Unknown phase: {phase}')

            # 2) Build the feature grid from the feature source data.
            
            feature_room_df, _, feature_info = build_room_level_dataframe(
                base_df=locali_features,
                room_name=room_name,
                room_prefix_list=room_prefix_list,
                all_temp_cols_for_average=all_room_temp_cols_features,
                phase=phase,
                source_label='locali_features_dadras_cleaned_for_feature_grid',
            )
            room_count_info.update({f'feature_{k}': v for k, v in feature_info.items()})

            # Synchronize additional sources for the full feature set.
            valve_cols = get_valve_columns(valve, room_prefix_list)
            if len(valve_cols) > 1:
                valve_df = valve[valve_cols].copy()
                valve_df = valve_df.rename(columns={c: 'VALVE_' + c for c in valve_df.columns if c != 'timestamp'})
                feature_room_df = merge_asof_with_offset(feature_room_df, valve_df, source_label='VALVE')
                diag = summarize_offsets(feature_room_df, room_prefix_id, 'VALVE', phase)
                if diag is not None:
                    sync_diagnostic_rows.append(diag)

            feature_room_df = merge_asof_with_offset(feature_room_df, ahu, source_label='AHU')
            diag = summarize_offsets(feature_room_df, room_prefix_id, 'AHU', phase)
            if diag is not None:
                sync_diagnostic_rows.append(diag)

            feature_room_df = merge_asof_with_offset(feature_room_df, cal, source_label='CAL')
            diag = summarize_offsets(feature_room_df, room_prefix_id, 'CAL', phase)
            if diag is not None:
                sync_diagnostic_rows.append(diag)

            room_count_info['after_feature_synchronization_rows'] = len(feature_room_df)

            feature_grid = interpolate_to_exact_grid(feature_room_df, freq=GRID_FREQ)
            feature_grid = clean_physical_sentinels(feature_grid, f'{phase}_{room_prefix_id}_feature_grid_after_interpolation_before_feature_creation')
            feature_grid['grid_preparation_mode'] = 'our_regular_5min_interpolated_feature_grid_after_source_sentinel_cleaning'
            room_count_info['after_exact_5min_feature_grid_rows'] = len(feature_grid)

            if phase == 'heating':
                feature_grid = create_heating_features_on_grid(feature_grid)
            elif phase == 'cooling':
                feature_grid = create_cooling_features_on_grid(feature_grid)

            # Extract temperature targets using each mode's observed event boundaries.
            samples = extract_full_event_samples(feature_grid, events, room_name, room_prefix_list, phase)
            n_samples = len(samples)

            # Build duration prediction rows independently of future temperature-target
            # availability.
            duration_anchors = build_remaining_duration_anchors_from_events_and_grid(
                room_grid=feature_grid,
                events=events,
                room_name=room_name,
                room_prefix_list=room_prefix_list,
                phase=phase,
                target_source=target_source,
            )
            n_duration_anchors = len(duration_anchors)
            n_duration_anchor_events = int(duration_anchors[['room', 'event_id']].drop_duplicates().shape[0]) if n_duration_anchors and {'room', 'event_id'}.issubset(duration_anchors.columns) else 0
            phase_duration_anchor_frames.append(duration_anchors)
            room_count_info['duration_anchor_rows_generated'] = n_duration_anchors
            room_count_info['duration_anchor_events_covered'] = n_duration_anchor_events

            # Require temperature-target coverage for selected rooms with detected
            # events.
            if len(events) > 0 and n_samples == 0:
                debug_dir = os.path.join(phase_dir, 'debug_zero_LONG_rooms')
                os.makedirs(debug_dir, exist_ok=True)
                debug_path = os.path.join(debug_dir, f'{room_prefix_id}_{phase}_events_but_zero_LONG_rows.csv')
                pd.DataFrame(events).to_csv(debug_path, index=False)
                if globals().get('REQUIRE_SELECTED_ROOMS_WITH_EVENTS_IN_LONG', True):
                    raise RuntimeError(
                        f'{phase} {room_prefix_id}: {len(events)} events detected but zero LONG rows exported. '
                        f'Debug events saved: {debug_path}'
                    )

            phase_sample_frames.append(samples)

            room_count_info['rows_inside_detected_events'] = sum([len(ev.get('indices', [])) for ev in events]) if len(events) else 0
            room_count_info['sample_rows_generated'] = n_samples

            print('Sample rows from our feature grid:', n_samples)
            print('Remaining-duration anchor rows from event table + feature grid:', n_duration_anchors)
            if n_samples > 0:
                print('Max horizon:', int(samples['horizon_min'].max()), 'min')
                print('Post-end trend-continuation target rows:', int(samples['is_post_end_buffer_target'].sum()))

            # 4) Save event rows.
            
            # Event start/end/duration are unchanged. extraction_end_time below is only
            # the maximum exported long-format target time after applying the post-end
            # trend rule.
            _sample_extraction_end_by_event = {}
            _sample_post_continuation_by_event = {}
            if n_samples > 0 and {'event_id', 'extraction_end_time'}.issubset(samples.columns):
                _tmp_ext = samples.copy()
                _tmp_ext['extraction_end_time'] = pd.to_datetime(_tmp_ext['extraction_end_time'], errors='coerce')
                _sample_extraction_end_by_event = _tmp_ext.groupby('event_id')['extraction_end_time'].max().to_dict()
                if 'post_end_trend_continuation_min' in _tmp_ext.columns:
                    _sample_post_continuation_by_event = _tmp_ext.groupby('event_id')['post_end_trend_continuation_min'].max().to_dict()

            for ev in events:
                event_rows.append({
                    'phase': phase,
                    'room': room_prefix_id,
                    'room_label': room_name,
                    'prefixes': ', '.join(room_prefix_list),
                    'event_id': ev.get('event_id', np.nan),
                    'event_date': ev.get('event_date', pd.to_datetime(ev.get('start_time')).date() if ev.get('start_time', None) is not None else None),
                    'start_time': ev.get('start_time', None),
                    'end_time': ev.get('end_time', None),
                    'duration_min': ev.get('duration_min', np.nan),
                    'duration_target_min': ev.get('duration_target_min', ev.get('duration_min', np.nan)),
                    'duration_continuous_min': ev.get('duration_continuous_min', ev.get('duration_min', np.nan)),
                    'duration_grid_min': ev.get('duration_grid_min', np.nan),
                    'end_grid_time': ev.get('end_grid_time', None),
                    'end_crossing_time': ev.get('end_crossing_time', ev.get('end_time', None)),
                    'end_idx_grid': ev.get('end_idx_grid', np.nan),
                    'duration_estimation': ev.get('duration_estimation', 'dadras_style_discrete_timestamp_no_interpolation'),
                    'target_crossing_threshold_at_end': ev.get('target_crossing_threshold_at_end', np.nan),
                    'actual_end_time': ev.get('actual_end_time', ev.get('end_time', None)),
                    'actual_duration_min': ev.get('actual_duration_min', ev.get('duration_min', np.nan)),
                    'observed_end_time': ev.get('observed_end_time', ev.get('end_time', None)),
                    'observed_duration_min': ev.get('observed_duration_min', ev.get('duration_min', np.nan)),
                    'extraction_end_time': _sample_extraction_end_by_event.get(ev.get('event_id', np.nan), ev.get('end_grid_time', ev.get('end_time', None))),
                    'post_end_buffer_min': POST_END_BUFFER_MIN,
                    'post_end_export_rule': POST_END_EXPORT_MODE,
                    'post_end_trend_continuation_min': _sample_post_continuation_by_event.get(ev.get('event_id', np.nan), 0.0),
                    'n_rows_inside_event_detection_frame': ev.get('n_rows', np.nan),
                    'availability_source': ev.get('availability_source', None),
                    'target_source_for_event_detection': target_source,
                    'room_T_at_start': ev.get('room_T_at_start', np.nan),
                    'room_T_at_end': ev.get('room_T_at_end', np.nan),
                    'target_at_start': ev.get('target_at_start', np.nan),
                    'target_at_end': ev.get('target_at_end', np.nan),
                    'event_detection_source': (ev.get('event_detection_source', 'final_feature_grid_fixed_room_target_heating_TV') if phase == 'heating' else ev.get('event_detection_source', 'localiin_dadras_style_TSV_only')),
                    'feature_extraction_source': 'our_regular_5min_feature_grid',
                    'cooling_start_threshold_at_start': ev.get('cooling_start_threshold_at_start', np.nan),
                    'cooling_end_threshold_at_end': ev.get('cooling_end_threshold_at_end', np.nan),
                    'heating_start_threshold_at_start': ev.get('heating_start_threshold_at_start', np.nan),
                    'heating_end_threshold_at_end': ev.get('heating_end_threshold_at_end', np.nan),
                    'dadras_exact_T_minus_T_1lag_at_end_C': ev.get('dadras_exact_T_minus_T_1lag_at_end_C', np.nan),
                    'dadras_symmetric_T_minus_T_1lag_at_end_C': ev.get('dadras_symmetric_T_minus_T_1lag_at_end_C', np.nan),
                    'dadras_spirit_T_minus_T_1lag_at_end_C': ev.get('dadras_spirit_T_minus_T_1lag_at_end_C', np.nan),
                    'start_rule': ev.get('start_rule', None),
                    'end_rule': ev.get('end_rule', None),
                })

            # 5) Summary rows by horizon.
            
            if n_samples > 0:
                max_h = int(samples['horizon_min'].max())
                n_post_buffer_targets = int(samples['is_post_end_buffer_target'].sum())
                horizon_counts = samples.groupby('horizon_min').size().reset_index(name='rows')
                for _, hr in horizon_counts.iterrows():
                    summary_rows.append({
                        **room_count_info,
                        'horizon_min': int(hr['horizon_min']),
                        'rows_saved': int(hr['rows']),
                        'max_horizon_for_room_phase_min': max_h,
                        'post_buffer_target_rows_total_room_phase': n_post_buffer_targets,
                        'status': 'saved',
                    })
            else:
                summary_rows.append({
                    **room_count_info,
                    'horizon_min': np.nan,
                    'rows_saved': 0,
                    'max_horizon_for_room_phase_min': np.nan,
                    'post_buffer_target_rows_total_room_phase': 0,
                    'status': 'no_samples_after_dynamic_extraction',
                })

        except Exception as e:
            print('FAILED:', room_name, '| phase:', phase, '| error:', repr(e))
            summary_rows.append({
                **room_count_info,
                'horizon_min': np.nan,
                'rows_saved': 0,
                'max_horizon_for_room_phase_min': np.nan,
                'post_buffer_target_rows_total_room_phase': 0,
                'status': 'failed',
                'error': repr(e),
            })

    all_samples = pd.concat([df for df in phase_sample_frames if df is not None and len(df) > 0], ignore_index=True) if any((df is not None and len(df) > 0) for df in phase_sample_frames) else pd.DataFrame(columns=phase_feature_columns(phase))
    duration_anchors_df = pd.concat([df for df in phase_duration_anchor_frames if df is not None and len(df) > 0], ignore_index=True) if any((df is not None and len(df) > 0) for df in phase_duration_anchor_frames) else pd.DataFrame(columns=remaining_duration_anchor_columns(phase))
    events_df = pd.DataFrame(event_rows)
    summary_df = pd.DataFrame(summary_rows)
    sync_df = pd.DataFrame(sync_diagnostic_rows)

    paths = save_phase_outputs(phase, all_samples, events_df, summary_df, sync_df, duration_anchors_df=duration_anchors_df)
    duration_exclusion_columns = [
        'phase', 'room', 'room_label', 'prefixes', 'event_date', 'start_time', 'end_time',
        'duration_min', 'maximum_included_duration_min', 'exclusion_reason',
    ]
    duration_exclusion_df = pd.DataFrame(phase_duration_exclusion_rows)
    if duration_exclusion_df.empty:
        duration_exclusion_df = pd.DataFrame(columns=duration_exclusion_columns)
    else:
        duration_exclusion_df = duration_exclusion_df.reindex(columns=duration_exclusion_columns)
    duration_exclusion_path = os.path.join(phase_dir, f'{phase}_duration_exclusion_audit.csv')
    duration_exclusion_df.to_csv(duration_exclusion_path, index=False)
    paths['duration_exclusion_audit'] = duration_exclusion_path
    all_phase_outputs[phase] = paths

    print('\nPHASE FINISHED:', phase)
    print('Total samples:', len(all_samples))
    print('Events:', len(events_df))
    print('Remaining-duration anchors:', len(duration_anchors_df))
    print('Output paths:', json.dumps(paths, indent=2))

print('\nAll requested phases finished.')
print(json.dumps(all_phase_outputs, indent=2))

data_cleaning_audit_paths = save_data_cleaning_audits(OUT_DIR)
print(json.dumps(data_cleaning_audit_paths, indent=2))


In [ ]:


# Final endpoint consistency audits for saved heating events

def run_final_heating_physical_audits(out_dir):
    """Check saved heating events against the fixed-target endpoint conditions and
    summarize feature coverage. Original observation endpoints can fall between
    feature-grid times, so same-timestamp feature matches are diagnostic rather
    than mandatory.
    """
    heating_dir = os.path.join(out_dir, 'heating')
    events_path = os.path.join(heating_dir, 'heating_events_5min_grid.csv')
    samples_path = os.path.join(heating_dir, 'heating_full_rampup_5min_samples_LONG.csv')

    summary = {
        'events_path': events_path,
        'samples_path': samples_path,
        'heating_rule': 'Dadras-symmetric heating: fixed_room_target end condition Room_T(t_end) >= Heating_Target_for_detection - 0.4',
        'heating_end_below_setpoint_tolerance_C': float(globals().get('HEATING_FEATURE_GRID_VALIDATION_TOLERANCE_C', 1e-6)),
        'large_setpoint_change_threshold_C': 1.0,
        'actual_Room_S_change_is_diagnostic_only': True,
        'exact_LONG_endpoint_timestamp_required': False,
        'why_exact_LONG_endpoint_not_required': 'Dadras-style event boundaries can be off the 5-minute feature/LONG grid.',
    }

    if not os.path.exists(events_path):
        raise RuntimeError(f'Heating events file not found for final audit: {events_path}')

    he = pd.read_csv(events_path)
    summary['n_heating_events'] = int(len(he))

    for c in ['room_T_at_end', 'target_at_end', 'room_T_at_start', 'target_at_start']:
        if c not in he.columns:
            raise RuntimeError(f'Final heating audit requires column {c}, but it is missing from {events_path}')
        he[c] = pd.to_numeric(he[c], errors='coerce')

    tol = float(globals().get('HEATING_FEATURE_GRID_VALIDATION_TOLERANCE_C', 1e-6))

    # target_at_start and target_at_end are fixed room-specific heating targets,
    # not measured/commanded Room_S(t).
    he['heating_start_threshold_fixed_target_C'] = he['target_at_start'] - HEATING_START_BELOW_SETPOINT
    he['heating_start_margin_fixed_target_C'] = he['heating_start_threshold_fixed_target_C'] - he['room_T_at_start']
    he['heating_end_threshold_fixed_target_C'] = he['target_at_end'] - HEATING_END_BELOW_SETPOINT
    he['heating_end_margin_fixed_target_C'] = he['room_T_at_end'] - he['heating_end_threshold_fixed_target_C']

    he['heating_start_condition_ok_fixed_target'] = he['heating_start_margin_fixed_target_C'] >= -tol
    he['heating_end_condition_ok_fixed_target'] = he['heating_end_margin_fixed_target_C'] >= -tol

    threshold_audit_path = os.path.join(heating_dir, 'heating_final_fixed_target_end_threshold_audit.csv')
    he.to_csv(threshold_audit_path, index=False)

    bad_end = he[(~he['heating_end_condition_ok_fixed_target']) | he['heating_end_margin_fixed_target_C'].isna()].copy()
    summary['n_bad_heating_end_threshold_events'] = int(len(bad_end))
    summary['min_fixed_target_heating_end_margin_C'] = float(he['heating_end_margin_fixed_target_C'].min(skipna=True)) if len(he) else np.nan
    summary['min_fixed_target_heating_start_margin_C'] = float(he['heating_start_margin_fixed_target_C'].min(skipna=True)) if len(he) else np.nan
    summary['threshold_audit_path'] = threshold_audit_path

    print('\nFINAL HEATING FIXED-TARGET END-THRESHOLD AUDIT')
    print('Heating events:', len(he))
    print('Bad heating end-threshold events:', len(bad_end))
    print('Minimum fixed-target heating end margin [C]:', summary['min_fixed_target_heating_end_margin_C'])

    if len(bad_end) > 0:
        bad_path = os.path.join(heating_dir, 'BAD_heating_events_below_fixed_target_end_threshold.csv')
        bad_end.to_csv(bad_path, index=False)
        summary['bad_heating_end_threshold_path'] = bad_path
        cols = [
            'room', 'event_id', 'start_time', 'end_time', 'duration_min',
            'room_T_at_end', 'target_at_end',
            'heating_end_threshold_fixed_target_C', 'heating_end_margin_fixed_target_C',
            'event_detection_source'
        ]
        cols = [c for c in cols if c in bad_end.columns]
        print('Bad heating events saved to:', bad_path)
        try:
            display(bad_end[cols].sort_values('heating_end_margin_fixed_target_C').head(30))
        except Exception:
            print(bad_end[cols].sort_values('heating_end_margin_fixed_target_C').head(30))
        raise RuntimeError(f'Final heating audit failed: {len(bad_end)} saved events end below fixed heating target - {HEATING_END_BELOW_SETPOINT}.')

    # Check target coverage without requiring observation endpoints to coincide with
    # regular-grid timestamps.
    endpoint_path = None
    if os.path.exists(samples_path):
        usecols = lambda c: c in ['phase', 'room', 'event_id', 'timestamp', 'Room_T', 'Heating_Target_for_detection', 'Room_S']
        smp = pd.read_csv(samples_path, usecols=usecols)
        if len(smp):
            smp['timestamp'] = pd.to_datetime(smp['timestamp'], errors='coerce')
            smp['event_id'] = smp['event_id'].astype(str).str.strip()
            smp['room'] = smp['room'].astype(str).str.strip()
            smp['phase'] = smp['phase'].astype(str).str.lower().str.strip()
            smp['Room_T'] = pd.to_numeric(smp['Room_T'], errors='coerce')
            smp['Heating_Target_for_detection'] = pd.to_numeric(smp['Heating_Target_for_detection'], errors='coerce')
            uniq = smp[['phase', 'room', 'event_id', 'timestamp', 'Room_T', 'Heating_Target_for_detection']].drop_duplicates()

            ev_check = he.copy()
            ev_check['end_time'] = pd.to_datetime(ev_check['end_time'], errors='coerce')
            ev_check['event_id'] = ev_check['event_id'].astype(str).str.strip()
            ev_check['room'] = ev_check['room'].astype(str).str.strip()
            ev_check['phase'] = ev_check['phase'].astype(str).str.lower().str.strip()

            # Exact endpoint diagnostic only.
            end_rows = uniq.rename(columns={
                'timestamp': 'end_time',
                'Room_T': 'long_Room_T_at_end_same_timestamp',
                'Heating_Target_for_detection': 'long_heating_target_at_end_same_timestamp',
            })
            endpoint = ev_check.merge(
                end_rows,
                on=['phase', 'room', 'event_id', 'end_time'],
                how='left',
                validate='one_to_one',
            )
            endpoint['has_exact_LONG_endpoint_timestamp'] = endpoint['long_Room_T_at_end_same_timestamp'].notna()

            # Event-level long-format coverage and nearest/last available current rows.
            counts = (
                uniq.groupby(['phase', 'room', 'event_id'], dropna=False)
                .agg(
                    n_LONG_current_rows=('timestamp', 'nunique'),
                    first_LONG_current_timestamp=('timestamp', 'min'),
                    last_LONG_current_timestamp=('timestamp', 'max'),
                )
                .reset_index()
            )
            endpoint = endpoint.merge(counts, on=['phase', 'room', 'event_id'], how='left')
            endpoint['has_any_LONG_rows_for_event'] = endpoint['n_LONG_current_rows'].fillna(0).astype(int) > 0
            endpoint['minutes_end_after_last_LONG_current_row'] = (
                endpoint['end_time'] - endpoint['last_LONG_current_timestamp']
            ).dt.total_seconds() / 60.0

            # Attach the last available long-format current row as diagnostic only.
            last_idx = uniq.sort_values('timestamp').groupby(['phase', 'room', 'event_id'], dropna=False)['timestamp'].idxmax()
            last_rows = uniq.loc[last_idx, ['phase', 'room', 'event_id', 'timestamp', 'Room_T', 'Heating_Target_for_detection']].rename(columns={
                'timestamp': 'last_LONG_current_timestamp_from_row',
                'Room_T': 'last_LONG_current_Room_T',
                'Heating_Target_for_detection': 'last_LONG_current_heating_target',
            })
            endpoint = endpoint.merge(last_rows, on=['phase', 'room', 'event_id'], how='left')
            endpoint['last_LONG_current_fixed_target_threshold_C'] = endpoint['last_LONG_current_heating_target'] - HEATING_END_BELOW_SETPOINT
            endpoint['last_LONG_current_fixed_target_margin_C'] = endpoint['last_LONG_current_Room_T'] - endpoint['last_LONG_current_fixed_target_threshold_C']
            endpoint['event_table_minus_last_LONG_current_T_C'] = endpoint['room_T_at_end'] - endpoint['last_LONG_current_Room_T']

            endpoint_path = os.path.join(heating_dir, 'heating_LONG_event_coverage_and_endpoint_diagnostic_audit.csv')
            endpoint.to_csv(endpoint_path, index=False)

            missing_any = endpoint[~endpoint['has_any_LONG_rows_for_event']].copy()
            missing_exact = endpoint[~endpoint['has_exact_LONG_endpoint_timestamp']].copy()

            summary['endpoint_consistency_audit_path'] = endpoint_path
            summary['n_events_with_any_LONG_rows'] = int(endpoint['has_any_LONG_rows_for_event'].sum())
            summary['n_events_missing_all_LONG_rows'] = int(len(missing_any))
            summary['n_missing_exact_LONG_endpoint_timestamp_rows_diagnostic_only'] = int(len(missing_exact))
            summary['exact_LONG_endpoint_timestamp_fraction_diagnostic_only'] = float(endpoint['has_exact_LONG_endpoint_timestamp'].mean()) if len(endpoint) else np.nan
            summary['median_minutes_end_after_last_LONG_current_row'] = float(endpoint['minutes_end_after_last_LONG_current_row'].median(skipna=True)) if len(endpoint) else np.nan
            summary['max_minutes_end_after_last_LONG_current_row'] = float(endpoint['minutes_end_after_last_LONG_current_row'].max(skipna=True)) if len(endpoint) else np.nan

            print('\nHEATING LONG EVENT-COVERAGE / ENDPOINT DIAGNOSTIC AUDIT')
            print('Events with any LONG rows:', summary['n_events_with_any_LONG_rows'], '/', len(endpoint))
            print('Events missing all LONG rows:', summary['n_events_missing_all_LONG_rows'])
            print('Exact same-timestamp LONG endpoint rows missing [diagnostic only]:', summary['n_missing_exact_LONG_endpoint_timestamp_rows_diagnostic_only'])
            print('Median minutes from last LONG current row to event end:', summary['median_minutes_end_after_last_LONG_current_row'])

            if len(missing_any):
                p = os.path.join(heating_dir, 'DIAGNOSTIC_heating_events_missing_all_LONG_rows.csv')
                missing_any.to_csv(p, index=False)
                summary['diagnostic_missing_all_LONG_rows_path'] = p
                summary['missing_all_LONG_rows_is_diagnostic_only'] = True
                summary['why_missing_all_LONG_rows_not_hard_failure'] = (
                    'Temperature LONG rows require at least one valid 5-minute current/target sample. '
                    'Dadras-style event boundaries can be irregular and a very small number of events may have no valid temperature-forecast LONG rows. '
                    'This does not invalidate the event definition; duration-anchor coverage is checked separately.'
                )
                print('WARNING: events missing all LONG rows saved as diagnostic only:', p)

    # Actual Room_S-change audit from exported samples. This is diagnostic only.
    setpoint_audit_path = None
    large_setpoint_path = None
    n_large_setpoint_events = 0
    if os.path.exists(samples_path):
        smp = pd.read_csv(samples_path, usecols=lambda c: c in ['phase', 'room', 'event_id', 'timestamp', 'Room_S', 'target_context'])
        if len(smp) and {'phase', 'room', 'event_id', 'timestamp', 'Room_S'}.issubset(smp.columns):
            smp['Room_S'] = pd.to_numeric(smp['Room_S'], errors='coerce')
            uniq = smp[['phase', 'room', 'event_id', 'timestamp', 'Room_S']].drop_duplicates()
            sp = (
                uniq.groupby(['phase', 'room', 'event_id'], dropna=False)
                .agg(
                    n_unique_current_timestamps=('timestamp', 'nunique'),
                    actual_Room_S_min_C=('Room_S', 'min'),
                    actual_Room_S_max_C=('Room_S', 'max'),
                    actual_Room_S_mean_C=('Room_S', 'mean'),
                )
                .reset_index()
            )
            sp['actual_Room_S_range_C'] = sp['actual_Room_S_max_C'] - sp['actual_Room_S_min_C']
            sp['large_actual_Room_S_change_flag'] = sp['actual_Room_S_range_C'] > 1.0
            sp['diagnostic_only_not_removed'] = True

            setpoint_audit_path = os.path.join(heating_dir, 'heating_actual_Room_S_change_diagnostic_audit.csv')
            sp.to_csv(setpoint_audit_path, index=False)

            large = sp[sp['large_actual_Room_S_change_flag']].copy()
            n_large_setpoint_events = int(len(large))
            if len(large):
                large_setpoint_path = os.path.join(heating_dir, 'heating_events_with_large_actual_Room_S_change_DIAGNOSTIC_ONLY.csv')
                large.sort_values('actual_Room_S_range_C', ascending=False).to_csv(large_setpoint_path, index=False)

    summary['endpoint_consistency_audit_path'] = endpoint_path
    summary['actual_Room_S_change_audit_path'] = setpoint_audit_path
    summary['large_actual_Room_S_change_path'] = large_setpoint_path
    summary['n_large_actual_Room_S_change_events_gt_1C'] = n_large_setpoint_events

    print('\nHEATING ACTUAL Room_S CHANGE DIAGNOSTIC AUDIT')
    print('Large actual Room_S-change events > 1 C:', n_large_setpoint_events)
    print('These are diagnostic only; the valid rule is Room_T(t) vs fixed Heating_Target_for_detection.')
    if large_setpoint_path:
        print('Large actual Room_S-change audit saved to:', large_setpoint_path)

    summary_path = os.path.join(heating_dir, 'heating_final_fixed_target_physical_audit_summary.json')
    with open(summary_path, 'w') as f:
        json.dump(summary, f, indent=2, default=str)
    summary['summary_path'] = summary_path

    return summary

FINAL_HEATING_AUDIT_SUMMARY = run_final_heating_physical_audits(OUT_DIR)
print('\nFinal heating physical audit summary:')
print(json.dumps(FINAL_HEATING_AUDIT_SUMMARY, indent=2, default=str))


In [ ]:

# Diagnostics and validation checks, robust to empty outputs

from pandas.errors import EmptyDataError

def read_csv_safe_for_diagnostics(path, parse_dates=None):
    if path is None or not os.path.exists(path):
        print('Missing file:', path)
        return pd.DataFrame()
    if os.path.getsize(path) == 0:
        print('Zero-byte file:', path)
        return pd.DataFrame()
    try:
        return pd.read_csv(path, parse_dates=parse_dates)
    except EmptyDataError:
        print('Empty CSV with no columns:', path)
        return pd.DataFrame()
    except ValueError as e:
        print('Read warning:', e)
        try:
            return pd.read_csv(path)
        except Exception as e2:
            print('Could not read:', path, e2)
            return pd.DataFrame()

DATE_COLS_SAMPLES = ['timestamp', 'target_timestamp', 'rampup_start_time', 'rampup_end_time', 'rampup_end_grid_time', 'extraction_end_time']

for phase in PHASES_TO_RUN:
    print('\n' + '=' * 90)
    print('DIAGNOSTICS FOR:', phase.upper())
    phase_dir = os.path.join(OUT_DIR, phase)
    long_path = os.path.join(phase_dir, f'{phase}_full_rampup_5min_samples_LONG.csv')
    events_path = os.path.join(phase_dir, f'{phase}_events_5min_grid.csv')
    summary_path = os.path.join(phase_dir, f'{phase}_extraction_summary.csv')

    print('long samples path:', long_path)
    print('events path:', events_path)
    print('summary path:', summary_path)

    samples = read_csv_safe_for_diagnostics(long_path, parse_dates=DATE_COLS_SAMPLES)
    events = read_csv_safe_for_diagnostics(events_path)
    summary = read_csv_safe_for_diagnostics(summary_path)

    print('Samples:', len(samples), '| shape:', samples.shape)
    print('Events:', len(events), '| shape:', events.shape)
    print('Summary:', len(summary), '| shape:', summary.shape)

    if len(events) == 0:
        print(f'WARNING: no {phase} events were extracted. Check summary failure/status rows below.')
        if len(summary) > 0:
            display(summary.head(50))
        continue

    print('\nEvents by room:')
    if 'room' in events.columns:
        display(events.groupby('room').size().reset_index(name='events').sort_values('events', ascending=False))

    print('\nDuration summary:')
    duration_col = next((c for c in ['duration_min', 'actual_duration_min', 'observed_duration_min'] if c in events.columns), None)
    if duration_col is not None:
        display(pd.to_numeric(events[duration_col], errors='coerce').describe().to_frame(duration_col).T)

    if len(samples) == 0:
        print(f'WARNING: events exist but no {phase} samples were created.')
        if len(summary) > 0:
            print('\nSummary status counts:')
            display(summary['status'].value_counts(dropna=False).reset_index() if 'status' in summary.columns else summary.head())
        continue

    print('Rooms in samples:', samples['room'].nunique() if 'room' in samples.columns else 0)

    print('\nSentinel/extreme-value check:')
    try:
        assert_no_bad_sentinels(samples, f'{phase} diagnostic samples')
        print('No sentinel-like values <=', BAD_SENTINEL_THRESHOLD, 'found in saved samples.')
    except RuntimeError as e:
        print('ERROR:', e)
        raise

    all_missing = audit_all_missing_columns(samples, f'{phase}_saved_samples_diagnostic')
    if len(all_missing):
        print('All-missing columns in saved samples:')
        display(all_missing)

    if 'target_timestamp' in samples.columns and 'timestamp' in samples.columns and 'horizon_min' in samples.columns:
        samples['computed_gap_min'] = (samples['target_timestamp'] - samples['timestamp']).dt.total_seconds() / 60.0
        gap_abs_error = (samples['computed_gap_min'] - pd.to_numeric(samples['horizon_min'], errors='coerce')).abs()
        print('Max exact-gap error:', gap_abs_error.max())
    else:
        gap_abs_error = pd.Series([], dtype=float)
        print('Cannot compute exact-gap check.')

    if 'horizon_min' in samples.columns and len(samples):
        print('Unique max horizon:', int(pd.to_numeric(samples['horizon_min'], errors='coerce').max()))
    if 'is_post_end_buffer_target' in samples.columns:
        print('Post-end buffer target share (%):', 100 * pd.to_numeric(samples['is_post_end_buffer_target'], errors='coerce').mean())

    if all(c in samples.columns for c in ['timestamp', 'rampup_start_time', 'rampup_end_grid_time', 'target_timestamp', 'extraction_end_time']):
        violations = samples[
            (samples['timestamp'] < samples['rampup_start_time']) |
            (samples['timestamp'] > samples['rampup_end_grid_time']) |
            (samples['target_timestamp'] > samples['extraction_end_time']) |
            (gap_abs_error > 1e-9)
        ]
        print('Validation violations:', len(violations))
        if len(violations) > 0:
            display(violations.head())

    print('\nRows by horizon:')
    if 'horizon_min' in samples.columns:
        display(samples.groupby('horizon_min').size().reset_index(name='rows').head(50))

    print('\nRows by room:')
    if 'room' in samples.columns:
        display(samples.groupby('room').size().reset_index(name='rows').sort_values('rows', ascending=False))

    print('\nSummary status counts:')
    if len(summary) > 0 and 'status' in summary.columns:
        display(summary['status'].value_counts(dropna=False).reset_index())


In [ ]:

# Save metadata JSON for reproducibility and paper updates

metadata = {
    'cooling_event_extraction_mode': globals().get('COOLING_EVENT_EXTRACTION_MODE', 'dadras_exact'),
    'heating_event_extraction_mode': globals().get('HEATING_EVENT_EXTRACTION_MODE', 'dadras_symmetric_exact'),
    'base_dir': BASE_DIR,
    'data_preparation_mode': globals().get('DATA_PREPARATION_MODE', None),
    'event_detection_input': private_value('inputs.literal_021'),
    'feature_source_timestamp_handling': 'LocaliIN, AHUIN, CaldaiaPannelliIN, and LocaliValve are all read with the same UTC-to-Europe/Rome DrDadras-style timestamp conversion before synchronization.',
    'output_dir': OUT_DIR,
    'selected_room_prefixes': SELECTED_ROOM_PREFIXES,
    'selected_room_mapping': ROOMS,
    'phases_to_run': PHASES_TO_RUN,
    'grid_freq': GRID_FREQ,
    'step_min': STEP_MIN,
    'post_end_buffer_min': POST_END_BUFFER_MIN,
    'post_end_export_mode': POST_END_EXPORT_MODE,
    'post_end_slope_tol_C_per_step': POST_END_SLOPE_TOL_C_PER_STEP,
    'post_end_max_continuation_min': POST_END_MAX_CONTINUATION_MIN,
    'min_rampup_minutes': MIN_RAMPUP_MINUTES,
    'max_rampup_minutes_by_phase': {
        'heating': MAX_HEATING_RAMPUP_MINUTES,
        'cooling': MAX_COOLING_RAMPUP_MINUTES,
    },
    'final_refinements': {
        'target_context': 'inside_event/post_end_trend_continuation added to every sample',
        'horizon_support_category': 'added for cooling/heating horizon reliability',
        'sample_weights': 'not exported; any event/sample weighting is computed only inside the training/evaluation notebook if needed',
        'counterfactual': 'removed; official duration is actual observed discrete duration',
    },
    'new_logic_summary': {
        'time_grid': 'Heating and cooling boundaries are both detected on DrDadras-style LocaliIN rows. Cooling uses exact Dadras; heating uses the sign-reversed symmetric Dadras rule with fixed heating targets. Trajectories are exported on the exact 5-minute feature grid.',
        'target_construction': 'Targets are generated by exact 5-minute shifts. No nearest future timestamp or horizon tolerance is used.',
        'horizon_range': 'Horizon range is dynamic and event-specific. Event definitions are unchanged; post-end target export continues only while observed 5-minute Room_T trend keeps the phase sign (heating positive, cooling negative).',
        'event_start_end': 'Cooling start/end use exact Dadras; heating start/end use the symmetric sign-reversed Dadras rule with fixed room-specific heating targets.',
        'outputs': 'Heating and cooling are saved as two separate datasets from one unified codebase.',
    },
    'heating_event_rules': {
        'start': 'symmetric Dadras: first timestamp in heating morning window with Room_V > 0.4, Room_T(t) < Heating_Target - 0.5, and idx.hour < HEATING_LATEST_START_HOUR.',
        'end': 'symmetric Dadras: first timestamp after start with Room_T(t) - Room_T(t-5min) > 0 and Room_T(t) >= Heating_Target - 0.4.',
        'morning_window': [HEATING_MORNING_START, HEATING_MORNING_END],
        'start_below_setpoint': HEATING_START_BELOW_SETPOINT,
        'end_below_setpoint': HEATING_END_BELOW_SETPOINT,
        'fixed_room_specific_heating_target': True,
        'heating_room_target_c': HEATING_ROOM_TARGET_C,
        'actual_Room_S_change_is_diagnostic_only': True,
        'min_heating_supply_temp': HEATING_MIN_SUPPLY_TEMP,
    },
    'cooling_event_rules': {
        'start': 'Exact DrDadras: first timestamp in 05:00-09:30 with Room_V > 0.4, Room_T > Target + 0.5, and idx.hour < 8.',
        'end': 'Exact DrDadras: first timestamp after start with Room_T - Room_T_1lag < 0 and Room_T <= Target + 0.4.',
        'cooling_target_mode': TARGET_MODE,
        'fixed_cooling_target_c': FIXED_COOLING_TARGET_C,
        'morning_window': [COOLING_MORNING_START, COOLING_MORNING_END],
        'latest_start_time': COOLING_LATEST_START_TIME,
        'start_strictly_before_latest': COOLING_START_STRICTLY_BEFORE_LATEST,
        'end_search_end_time': COOLING_END_SEARCH_END_TIME,
        'latest_start_hour_deprecated': COOLING_LATEST_START_HOUR,
        'cooling_months': COOLING_MONTHS if USE_COOLING_SEASON_MONTH_FILTER else None,
    },
    'counterfactual_and_scoring': 'disabled; official event duration is observed/extracted duration only',
    'sample_weight_columns_exported': False,
    'heating_detection_fix': {
        'heating_event_detection_on_final_feature_grid': HEATING_EVENT_DETECTION_ON_FINAL_FEATURE_GRID,
        'signal_aware_grid_fill': SIGNAL_AWARE_GRID_FILL,
        'root_cause_audit_rows': len(globals().get('HEATING_DETECTION_GRID_AUDIT', [])),
    },
    'final_heating_physical_audit': globals().get('FINAL_HEATING_AUDIT_SUMMARY', {}),
    'data_cleaning': {
        'bad_sentinel_threshold': BAD_SENTINEL_THRESHOLD,
        'sentinel_cleaning_audit_rows': len(globals().get('SENTINEL_CLEANING_AUDIT', [])),
        'all_missing_feature_audit_rows': len(globals().get('ALL_MISSING_FEATURE_AUDIT', [])),
        'source_time_alignment_audit_rows': len(globals().get('SOURCE_TIME_ALIGNMENT_AUDIT', [])),
        'heating_detection_frame_vs_feature_grid_audit_rows': len(globals().get('HEATING_DETECTION_GRID_AUDIT', [])),
        'export_optional_cooling_source_features': EXPORT_OPTIONAL_COOLING_SOURCE_FEATURES,
        'export_optional_solar_weather_features': EXPORT_OPTIONAL_SOLAR_WEATHER_FEATURES,
        'cleaning_audit_paths': globals().get('data_cleaning_audit_paths', {}),
    },
    'paths': all_phase_outputs,
}

metadata_path = os.path.join(OUT_DIR, 'full_rampup_5min_extraction_metadata.json')
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2, default=str)

print('Metadata saved to:')
print(metadata_path)


In [ ]:

# Validate selected-room membership in saved outputs

print('Verifying that every saved CSV contains selected prefixes only...')
selected = set(SELECTED_ROOM_PREFIXES)
bad_files = []
checked_files = 0

for root, _, files in os.walk(OUT_DIR):
    for fn in files:
        if not fn.lower().endswith('.csv'):
            continue
        f = os.path.join(root, fn)
        try:
            # Read only columns that can identify rooms/prefixes, but read the full
            # file.
            header = pd.read_csv(f, nrows=0)
            usecols = [c for c in ['room', 'prefixes', 'selected_prefixes'] if c in header.columns]
            if not usecols:
                continue
            tmp = pd.read_csv(f, usecols=usecols)
            observed = set()
            for c in ['prefixes', 'selected_prefixes']:
                if c in tmp.columns:
                    for val in tmp[c].dropna().astype(str):
                        observed.update([p.strip() for p in val.replace(';', ',').split(',') if p.strip()])
            if 'room' in tmp.columns:
                observed.update(tmp['room'].dropna().astype(str).unique())
            bad = sorted(observed - selected)
            if bad:
                bad_files.append((f, bad))
            checked_files += 1
        except Exception as e:
            print('Could not verify file:', f, '|', repr(e))

if bad_files:
    print('FOUND NON-SELECTED ROOMS/PREFIXES:')
    for f, bad in bad_files[:20]:
        print(f, bad)
    raise ValueError('Some outputs contain non-selected room prefixes.')
else:
    print('OK — all checked CSV outputs contain selected prefixes only.')
    print('Checked CSV files:', checked_files)


In [ ]:

# Final dataset manifest for reproducibility

import os
import json
import pandas as pd
import numpy as np

manifest = {
    "notebook_name": "REAL_FINAL_DADRAS_EXACT_COOLING_SYMMETRIC_HEATING_FULL_COVERAGE_DURATION_ANCHORS",
    "output_dir": OUT_DIR,
    "event_boundary_detection": {
        "cooling": "exact DrDadras rule on DrDadras-style cleaned LocaliIN P*_T/P*_V columns",
        "heating": "symmetric DrDadras-style rule on DrDadras-style cleaned LocaliIN rows: Room_T is compared with fixed Heating_Target_for_detection, signs reversed from cooling",
    },
    "feature_dataset": "full-ramp 5-minute multi-horizon features; cooling uses exact DrDadras-style LocaliIN logic; heating uses symmetric sign-reversed DrDadras-style LocaliIN logic with fixed room-specific targets",
    "feature_source_timestamp_handling": "LocaliIN, AHUIN, CaldaiaPannelliIN, and LocaliValve are all read with the same UTC-to-Europe/Rome DrDadras-style timestamp conversion before synchronization.",
    "counterfactual_or_scoring": "not used",
    "official_duration_label": "actual observed discrete duration_min",
    "sample_weight_columns_exported": False,
    "sample_weight_policy": "No event_weight, room_event_weight, n_rows_in_event_samples, or n_events_in_room_phase columns are saved in extracted samples; weighting belongs only inside training/evaluation if used.",
    "full_coverage_fix": "Room_S and Offset are diagnostic/export columns and are not in the hard dropna subset; every selected room with detected events must have LONG rows and a by-room LONG file.",
    "phases": {},
    "heating_detection_fix": {
        "heating_event_detection_on_final_feature_grid": HEATING_EVENT_DETECTION_ON_FINAL_FEATURE_GRID,
        "fixed_room_specific_heating_target": True,
        "heating_room_target_c": HEATING_ROOM_TARGET_C,
        "actual_Room_S_change_is_diagnostic_only": True,
        "signal_aware_grid_fill": SIGNAL_AWARE_GRID_FILL,
        "root_cause_audit_rows": len(globals().get("HEATING_DETECTION_GRID_AUDIT", [])),
        "root_cause_audit": "heating_detection_frame_vs_feature_grid_audit.csv",
    },
    "final_heating_physical_audit": globals().get("FINAL_HEATING_AUDIT_SUMMARY", {}),
    "data_cleaning": {
        "bad_sentinel_threshold": BAD_SENTINEL_THRESHOLD,
        "sentinel_cleaning_audit_rows": len(globals().get("SENTINEL_CLEANING_AUDIT", [])),
        "all_missing_feature_audit_rows": len(globals().get("ALL_MISSING_FEATURE_AUDIT", [])),
        "source_time_alignment_audit_rows": len(globals().get("SOURCE_TIME_ALIGNMENT_AUDIT", [])),
        "heating_detection_frame_vs_feature_grid_audit_rows": len(globals().get("HEATING_DETECTION_GRID_AUDIT", [])),
        "export_optional_cooling_source_features": EXPORT_OPTIONAL_COOLING_SOURCE_FEATURES,
        "export_optional_solar_weather_features": EXPORT_OPTIONAL_SOLAR_WEATHER_FEATURES,
        "cleaning_audit_paths": globals().get("data_cleaning_audit_paths", {}),
    },
}

for phase in PHASES_TO_RUN:
    phase_info = {}
    outs = all_phase_outputs.get(phase, {}) if "all_phase_outputs" in globals() else {}
    for key, path in outs.items():
        phase_info[f"{key}_path"] = path
    events_path = outs.get("events")
    samples_path = outs.get("long_samples")
    if events_path and os.path.exists(events_path):
        ev = pd.read_csv(events_path)
        phase_info["n_events"] = int(len(ev))
        phase_info["n_rooms_events"] = int(ev["room"].nunique()) if "room" in ev.columns else None
        if "duration_min" in ev.columns and len(ev):
            phase_info["duration_min_summary"] = {k: float(v) for k, v in ev["duration_min"].describe().to_dict().items() if pd.notna(v)}
    if samples_path and os.path.exists(samples_path):
        smp = pd.read_csv(samples_path)
        phase_info["n_samples"] = int(len(smp))
        phase_info["n_feature_columns"] = int(len(smp.columns))
        phase_info["feature_columns"] = smp.columns.tolist()
        if "target_context" in smp.columns:
            phase_info["target_context_counts"] = {str(k): int(v) for k, v in smp["target_context"].value_counts(dropna=False).to_dict().items()}
        if "horizon_min" in smp.columns:
            phase_info["horizon_counts"] = {str(k): int(v) for k, v in smp["horizon_min"].value_counts().sort_index().to_dict().items()}
        if "horizon_support_category" in smp.columns:
            phase_info["horizon_support_counts"] = {str(k): int(v) for k, v in smp["horizon_support_category"].value_counts(dropna=False).to_dict().items()}
    manifest["phases"][phase] = phase_info

manifest_path = os.path.join(OUT_DIR, "dataset_manifest_REAL_FINAL_HEATING_ROOM_TARGET_NO_WEIGHTS.json")
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2, default=str)

print("Saved final dataset manifest:", manifest_path)
print(json.dumps({
    "output_dir": manifest["output_dir"],
    "phases": {p: {k: v for k, v in info.items() if k in ["n_events", "n_samples", "n_rooms_events"]} for p, info in manifest["phases"].items()},
    "counterfactual_or_scoring": manifest["counterfactual_or_scoring"],
}, indent=2))


## Dataset outputs and validation

The observed event boundaries define duration labels. Temperature targets may continue beyond the endpoint while consecutive five-minute changes retain the expected heating or cooling sign; this continuation does not extend the event itself.

Review event-rule, room-coverage, input-alignment and missing-value checks before using the exported datasets. Private room mappings and configuration are excluded from this public source package.


In [ ]:

# Heating T–t plots with fixed room-specific target

import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Path setup

if "OUT_DIR" not in globals():
    BASE_DIR = Path(str(private_workspace()))
    OUT_DIR = BASE_DIR / "full_rampup_5min_feature_dataset_REAL_FINAL_HEATING_ROOM_TARGET_NO_WEIGHTS"
else:
    OUT_DIR = Path(OUT_DIR)

HEATING_DIR = OUT_DIR / "heating"

samples_path = HEATING_DIR / "heating_full_rampup_5min_samples_LONG.csv"
events_path  = HEATING_DIR / "heating_events_5min_grid.csv"

if not samples_path.exists():
    raise FileNotFoundError(f"Missing heating samples file:\n{samples_path}")

if not events_path.exists():
    raise FileNotFoundError(f"Missing heating events file:\n{events_path}")

# User options

EVENT_KEY = None

ROOM = None
EVENT_ID = None

MAX_PLOTS = 12
PLOT_LARGEST_TARGET_MARGIN_FIRST = True
SAVE_PLOTS = False

PLOT_DIR = HEATING_DIR / "heating_T_t_fixed_target_plots"
if SAVE_PLOTS:
    PLOT_DIR.mkdir(parents=True, exist_ok=True)

HEATING_END_BELOW_SETPOINT = globals().get("HEATING_END_BELOW_SETPOINT", 0.4)
HEATING_START_BELOW_SETPOINT = globals().get("HEATING_START_BELOW_SETPOINT", 0.5)

# Helpers

def normalize_event_id(x):
    """Normalize event identifiers consistently across sample and event tables."""
    if pd.isna(x):
        return "nan"
    xs = str(x).strip()
    try:
        xf = float(xs)
        if np.isfinite(xf) and abs(xf - round(xf)) < 1e-9:
            return str(int(round(xf)))
    except Exception:
        pass
    return xs

def add_event_key(df):
    out = df.copy()
    out["phase"] = out["phase"].astype(str).str.strip().str.lower()
    out["room"] = out["room"].astype(str).str.strip()
    out["event_id_norm"] = out["event_id"].apply(normalize_event_id)
    out["event_key"] = out["phase"] + "|" + out["room"] + "|" + out["event_id_norm"
]
    return out

def reconstruct_same_timestamp_curve(event_samples):
    """Collapse repeated horizon rows into one current-temperature and fixed-target
    observation per timestamp for diagnostic plotting.
    """
    required = ["timestamp", "Room_T", "Heating_Target_for_detection"]
    missing = [c for c in required if c not in event_samples.columns]
    if missing:
        raise RuntimeError(f"Missing required columns in samples: {missing}")

    cols = ["timestamp", "Room_T", "Heating_Target_for_detection"] + (["Room_S"] if "Room_S" in event_samples.columns else [])
    if "Room_V" in event_samples.columns:
        cols.append("Room_V")

    cur = event_samples[cols].copy()
    cur["timestamp"] = pd.to_datetime(cur["timestamp"], errors="coerce")
    cur["Room_T"] = pd.to_numeric(cur["Room_T"], errors="coerce")
    cur["Heating_Target_for_detection"] = pd.to_numeric(cur["Heating_Target_for_detection"], errors="coerce")
    if "Room_S" in cur.columns:
        cur["Room_S"] = pd.to_numeric(cur["Room_S"], errors="coerce")

    if "Room_V" in cur.columns:
        cur["Room_V"] = pd.to_numeric(cur["Room_V"], errors="coerce")

    cur = cur.dropna(subset=["timestamp", "Room_T", "Heating_Target_for_detection"])

    # long-format data has repeated current timestamps because each current row has many
    # horizons.
    # We collapse duplicates at the same timestamp.
    agg_dict = {
        "Room_T": "mean",
        "Heating_Target_for_detection": "mean",
        **({"Room_S": "mean"} if "Room_S" in cur.columns else {}),
    }
    if "Room_V" in cur.columns:
        agg_dict["Room_V"] = "mean"

    cur = (
        cur.groupby("timestamp", as_index=False)
        .agg(agg_dict)
        .sort_values("timestamp")
        .reset_index(drop=True)
    )

    cur["heating_end_threshold"] = cur["Heating_Target_for_detection"] - HEATING_END_BELOW_SETPOINT
    cur["heating_start_threshold"] = cur["Heating_Target_for_detection"] - HEATING_START_BELOW_SETPOINT
    cur["end_margin_C"] = cur["Room_T"] - cur["heating_end_threshold"]

    return cur

def nearest_row_at_time(curve, t, tolerance_min=2.51):
    t = pd.to_datetime(t, errors="coerce")
    if pd.isna(t) or curve.empty:
        return None

    tmp = curve.copy()
    tmp["abs_dt_min"] = (tmp["timestamp"] - t).abs().dt.total_seconds().abs() / 60.0
    row = tmp.sort_values(["abs_dt_min", "timestamp"]).iloc[0]

    if row["abs_dt_min"] <= tolerance_min:
        return row

    return None

def plot_one_heating_event(samples, events, event_key):
    ev_rows = events[events["event_key"] == event_key].copy()
    if ev_rows.empty:
        raise RuntimeError(f"Event key not found in events table: {event_key}")

    ev = ev_rows.iloc[0]

    g = samples[samples["event_key"] == event_key].copy()
    if g.empty:
        raise RuntimeError(f"Event key not found in samples table: {event_key}")

    cur = reconstruct_same_timestamp_curve(g)

    start_time = pd.to_datetime(ev.get("start_time"), errors="coerce")
    end_time = pd.to_datetime(ev.get("end_time"), errors="coerce")

    start_row = nearest_row_at_time(cur, start_time)
    end_row = nearest_row_at_time(cur, end_time)

    fig, ax = plt.subplots(figsize=(11, 5.5))

    # Plot temperature and target values at matching timestamps.
    ax.plot(
        cur["timestamp"],
        cur["Room_T"],
        marker="o",
        linewidth=2,
        label="Room_T(t)"
    )

    ax.step(
        cur["timestamp"],
        cur["Heating_Target_for_detection"],
        where="post",
        linestyle="--",
        linewidth=2,
        label="Fixed heating target"
    )

    if "Room_S" in cur.columns:
        ax.step(
            cur["timestamp"],
            cur["Room_S"],
            where="post",
            linestyle="-.",
            linewidth=1.3,
            alpha=0.6,
            label="Actual Room_S(t), diagnostic only"
        )

    ax.step(
        cur["timestamp"],
        cur["heating_end_threshold"],
        where="post",
        linestyle=":",
        linewidth=2,
        label=f"Heating end threshold: target - {HEATING_END_BELOW_SETPOINT}"
    )

    # Event boundaries.
    if pd.notna(start_time):
        ax.axvline(start_time, linestyle=":", linewidth=1.5, label="Event start")

    if pd.notna(end_time):
        ax.axvline(end_time, linestyle="-.", linewidth=1.5, label="Event end")

    # Mark actual start/end points using matching-timestamp values.
    if start_row is not None:
        ax.scatter(
            [start_row["timestamp"]],
            [start_row["Room_T"]],
            s=90,
            marker="^",
            label="T(start), same timestamp"
        )

    if end_row is not None:
        ax.scatter(
            [end_row["timestamp"]],
            [end_row["Room_T"]],
            s=100,
            marker="v",
            label="T(end), same timestamp"
        )

        end_margin = float(end_row["end_margin_C"])
        end_ok = end_margin >= -1e-6

        txt = (
            f"End check at same timestamp:\n"
            f"T(end) = {end_row['Room_T']:.3f} °C\n"
            f"Target(end) = {end_row['Heating_Target_for_detection']:.3f} °C\n"
            f"Target-0.4 = {end_row['heating_end_threshold']:.3f} °C\n"
            f"margin = {end_margin:.3f} °C\n"
            f"OK = {end_ok}"
        )

        ax.text(
            0.01,
            0.98,
            txt,
            transform=ax.transAxes,
            va="top",
            ha="left",
            fontsize=9,
            bbox=dict(boxstyle="round", alpha=0.15)
        )

    # Optional fan signal on secondary axis.
    if "Room_V" in cur.columns:
        ax2 = ax.twinx()
        ax2.plot(
            cur["timestamp"],
            cur["Room_V"],
            marker="s",
            linestyle="--",
            alpha=0.45,
            label="Room_V"
        )
        ax2.set_ylabel("Room_V")
        h2, l2 = ax2.get_legend_handles_labels()
    else:
        h2, l2 = [], []

    room = ev.get("room", "unknown_room")
    event_id = ev.get("event_id_norm", ev.get("event_id", "unknown_event"))
    duration = ev.get("duration_min", np.nan)

    target_range = cur["Heating_Target_for_detection"].max() - cur["Heating_Target_for_detection"].min()
    room_s_range = (cur["Room_S"].max() - cur["Room_S"].min()) if "Room_S" in cur.columns else np.nan

    ax.set_title(
        f"Heating T–t with fixed room-specific target | {room} | event {event_id} | "
        f"duration={duration} min | target={cur["Heating_Target_for_detection"].iloc[0]:.1f} °C | Room_S range={room_s_range:.2f} °C"
    )
    ax.set_xlabel("Time")
    ax.set_ylabel("Temperature [°C]")
    ax.grid(True, alpha=0.3)

    h1, l1 = ax.get_legend_handles_labels()
    labels_seen = {}
    handles_final = []
    labels_final = []

    for h, l in zip(h1 + h2, l1 + l2):
        if l not in labels_seen:
            labels_seen[l] = True
            handles_final.append(h)
            labels_final.append(l)

    ax.legend(handles_final, labels_final, fontsize=8, loc="best")

    fig.autofmt_xdate()
    fig.tight_layout()

    if SAVE_PLOTS:
        safe_key = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(event_key))
        out_path = PLOT_DIR / f"{safe_key}_heating_T_t_fixed_target.png"
        fig.savefig(out_path, dpi=240)
        print("Saved:", out_path)

    plt.show()

    # Print exact same-timestamp endpoint check.
    print("=" * 90)
    print("EVENT:", event_key)
    print("Room:", room)
    print("Start time:", start_time)
    print("End time:", end_time)
    print("Fixed heating target [°C]:", float(cur["Heating_Target_for_detection"].iloc[0]))
    print("Actual Room_S range inside plotted sample [°C]:", room_s_range)

    if start_row is not None:
        print("\nStart same-timestamp values:")
        print("T(start):", float(start_row["Room_T"]))
        print("Heating target at start:", float(start_row["Heating_Target_for_detection"]))
        if "Room_S" in start_row.index:
            print("Actual Room_S(start):", float(start_row["Room_S"]))
        print("Start threshold target-0.5:", float(start_row["heating_start_threshold"]))
        print("Start condition T < target-0.5:", bool(start_row["Room_T"] < start_row["heating_start_threshold"]))

    if end_row is not None:
        print("\nEnd same-timestamp values:")
        print("T(end):", float(end_row["Room_T"]))
        print("Heating target at end:", float(end_row["Heating_Target_for_detection"]))
        if "Room_S" in end_row.index:
            print("Actual Room_S(end):", float(end_row["Room_S"]))
        print("End threshold target-0.4:", float(end_row["heating_end_threshold"]))
        print("End margin T - (target-0.4):", float(end_row["end_margin_C"]))
        print("End condition T >= target-0.4:", bool(end_row["end_margin_C"] >= -1e-6))

    return cur

# Load files

samples = pd.read_csv(samples_path)
events = pd.read_csv(events_path)

samples = add_event_key(samples)
events = add_event_key(events)

for c in ["timestamp", "target_timestamp", "rampup_start_time", "rampup_end_time"]:
    if c in samples.columns:
        samples[c] = pd.to_datetime(samples[c], errors="coerce")

for c in ["start_time", "end_time", "end_grid_time"]:
    if c in events.columns:
        events[c] = pd.to_datetime(events[c], errors="coerce")

# Choose event(s)

if EVENT_KEY is not None:
    selected_event_keys = [EVENT_KEY]

elif ROOM is not None and EVENT_ID is not None:
    selected_event_keys = [f"heating|{ROOM}|{normalize_event_id(EVENT_ID)}"]

else:
    # Default: plot events with the lowest fixed-target end margin first.
    unique_ts = samples[["event_key", "timestamp", "Room_T", "Heating_Target_for_detection"]].drop_duplicates().copy()
    unique_ts["timestamp"] = pd.to_datetime(unique_ts["timestamp"], errors="coerce")
    unique_ts["Room_T"] = pd.to_numeric(unique_ts["Room_T"], errors="coerce")
    unique_ts["Heating_Target_for_detection"] = pd.to_numeric(unique_ts["Heating_Target_for_detection"], errors="coerce")
    unique_ts = unique_ts.dropna(subset=["event_key", "timestamp", "Room_T", "Heating_Target_for_detection"])
    unique_ts = unique_ts.sort_values(["event_key", "timestamp"])

    rank = (
        unique_ts.groupby("event_key")
        .agg(
            room_T_at_last=("Room_T", "last"),
            target_at_last=("Heating_Target_for_detection", "last"),
        )
        .reset_index()
    )
    rank["end_margin_C"] = rank["room_T_at_last"] - (rank["target_at_last"] - HEATING_END_BELOW_SETPOINT)

    if PLOT_LARGEST_TARGET_MARGIN_FIRST:
        rank = rank.sort_values("end_margin_C", ascending=True)
    else:
        rank = rank.sort_values("event_key")

    selected_event_keys = rank["event_key"].head(MAX_PLOTS).tolist()

print("Selected events:")
for k in selected_event_keys:
    print(" ", k)

# Plot

plotted_curves = {}

for event_key in selected_event_keys:
    plotted_curves[event_key] = plot_one_heating_event(samples, events, event_key)

## Chronological split and dataset summaries

Events are grouped into Monday-start weeks and assigned chronologically to training, validation and test partitions. The training and validation week counts are obtained by rounding down their specified proportions; the remaining weeks form the test partition. All samples from an event inherit its assignment.

The exported summaries use one observation per event. Room-level figures use anonymous labels; identity mappings remain in the private workspace. File and split fingerprints support consistency checks in subsequent notebooks.


In [ ]:
# Create the event split once. Downstream notebooks verify and reuse the saved
# assignments.
from pathlib import Path
import os
import re

SHARED_SPLIT_BLOCK = "week"
SHARED_TRAIN_FRAC = 0.70
SHARED_VALID_FRAC = 0.15
SHARED_TEST_FRAC = 0.15

PAPER_TABLE_DIR = Path(OUT_DIR) / "paper_tables"
PAPER_FIG_DIR = Path(OUT_DIR) / "paper_figures"
PAPER_TABLE_DIR.mkdir(parents=True, exist_ok=True)
PAPER_FIG_DIR.mkdir(parents=True, exist_ok=True)

def _normalize_event_id_token(value):
    if pd.isna(value):
        raise RuntimeError("event_id is missing; a stable shared split cannot be created.")
    text = str(value).strip()
    if re.fullmatch(r"[-+]?\d+\.0+", text):
        text = str(int(float(text)))
    return text

def _shared_event_uid_frame(df, phase):
    out = df.copy()
    if "event_id" not in out.columns or "room" not in out.columns:
        raise RuntimeError(f"{phase}: event table requires room and event_id for the shared split map.")
    out["phase"] = phase
    out["shared_event_uid"] = (
        out["phase"].astype(str) + "|" + out["room"].astype(str) + "|" +
        out["event_id"].map(_normalize_event_id_token)
    )
    time_col = "start_time" if "start_time" in out.columns else "rampup_start_time"
    if time_col not in out.columns:
        raise RuntimeError(f"{phase}: no event start-time column was found.")
    out["event_sort_time"] = pd.to_datetime(out[time_col], errors="coerce")
    if out["event_sort_time"].isna().any():
        raise RuntimeError(f"{phase}: event start times contain invalid values.")
    out["split_block"] = out["event_sort_time"].dt.to_period("W-SUN").dt.start_time.dt.strftime("%Y-%m-%d")
    return out

event_frames = []
anchor_frames = []
for phase in ["heating", "cooling"]:
    outputs = all_phase_outputs.get(phase, {})
    event_path = outputs.get("events")
    anchor_path = outputs.get("remaining_duration_anchors")
    if not event_path or not Path(event_path).exists():
        raise RuntimeError(f"{phase}: authoritative event table is missing: {event_path}")
    if not anchor_path or not Path(anchor_path).exists():
        raise RuntimeError(f"{phase}: remaining-duration anchor table is missing: {anchor_path}")
    events = _shared_event_uid_frame(pd.read_csv(event_path), phase)
    anchors = pd.read_csv(anchor_path)
    anchors["phase"] = phase
    event_frames.append(events)
    anchor_frames.append(anchors)

all_events_for_split = pd.concat(event_frames, ignore_index=True)
if all_events_for_split["shared_event_uid"].duplicated().any():
    dup = all_events_for_split[all_events_for_split["shared_event_uid"].duplicated(keep=False)]
    dup.to_csv(PAPER_TABLE_DIR / "BAD_duplicate_shared_event_uid.csv", index=False)
    raise RuntimeError("Duplicate shared_event_uid values found; see BAD_duplicate_shared_event_uid.csv")

split_parts = []
boundary_rows = []
for phase, phase_events in all_events_for_split.groupby("phase", sort=False):
    blocks = (
        phase_events[["split_block", "event_sort_time"]]
        .groupby("split_block")["event_sort_time"].min().sort_values().index.tolist()
    )
    if len(blocks) < 3:
        raise RuntimeError(f"{phase}: at least three weekly blocks are required; found {len(blocks)}")
    n_train = max(1, int(np.floor(len(blocks) * SHARED_TRAIN_FRAC)))
    n_valid = max(1, int(np.floor(len(blocks) * SHARED_VALID_FRAC)))
    if n_train + n_valid >= len(blocks):
        n_train, n_valid = max(1, len(blocks) - 2), 1
    block_to_split = {
        **{b: "train" for b in blocks[:n_train]},
        **{b: "valid" for b in blocks[n_train:n_train + n_valid]},
        **{b: "test" for b in blocks[n_train + n_valid:]},
    }
    part = phase_events.copy()
    part["split"] = part["split_block"].map(block_to_split)
    split_parts.append(part)
    for split in ["train", "valid", "test"]:
        sb = part.loc[part["split"].eq(split), "split_block"]
        boundary_rows.append({
            "phase": phase, "split": split,
            "first_week": sb.min() if len(sb) else None,
            "last_week": sb.max() if len(sb) else None,
            "n_weeks": int(sb.nunique()),
        })

split_map = pd.concat(split_parts, ignore_index=True)
split_map = split_map[[
    "shared_event_uid", "phase", "room", "event_id", "event_sort_time", "split_block", "split"
]].sort_values(["phase", "event_sort_time", "room", "event_id"]).reset_index(drop=True)

if split_map["split"].isna().any() or set(split_map["split"].unique()) != {"train", "valid", "test"}:
    raise RuntimeError("Shared split map is incomplete.")

shared_split_path = Path(OUT_DIR) / "shared_event_split_map.csv"
split_map.to_csv(shared_split_path, index=False)
split_map.to_csv(PAPER_TABLE_DIR / "shared_event_split_map.csv", index=False)

split_summary = (
    split_map.groupby(["phase", "split"], as_index=False)
    .agg(events=("shared_event_uid", "nunique"), rooms=("room", "nunique"), weeks=("split_block", "nunique"),
         first_event=("event_sort_time", "min"), last_event=("event_sort_time", "max"))
)
split_summary["split"] = pd.Categorical(split_summary["split"], ["train", "valid", "test"], ordered=True)
split_summary = split_summary.sort_values(["phase", "split"])
split_summary.to_csv(Path(OUT_DIR) / "shared_split_summary.csv", index=False)
split_summary.to_csv(PAPER_TABLE_DIR / "dataset_split_support.csv", index=False)
pd.DataFrame(boundary_rows).to_csv(Path(OUT_DIR) / "shared_split_boundaries.csv", index=False)

# Stable anonymized room labels for all manuscript outputs.
rooms = sorted(split_map["room"].dropna().astype(str).unique())
room_alias = pd.DataFrame({"room": rooms, "room_alias": [f"R{i:02d}" for i in range(1, len(rooms) + 1)]})
room_alias.to_csv(Path(OUT_DIR) / "paper_room_alias_map.csv", index=False)

all_anchors_for_paper = pd.concat(anchor_frames, ignore_index=True)
dataset_rows = []
room_rows = []
for phase, events in all_events_for_split.groupby("phase", sort=False):
    duration_col = next((c for c in ["duration_min", "rampup_duration_min", "true_total_duration_min"] if c in events.columns), None)
    if duration_col is None:
        raise RuntimeError(f"{phase}: no event-duration column found for paper summaries.")
    duration = pd.to_numeric(events[duration_col], errors="coerce")
    anchors = all_anchors_for_paper[all_anchors_for_paper["phase"].eq(phase)]
    dataset_rows.append({
        "phase": phase,
        "rooms": int(events["room"].nunique()),
        "events": int(events["shared_event_uid"].nunique()),
        "duration_anchors": int(len(anchors)),
        "duration_mean_min": float(duration.mean()),
        "duration_median_min": float(duration.median()),
        "duration_IQR_min": float(duration.quantile(0.75) - duration.quantile(0.25)),
        "duration_min_min": float(duration.min()),
        "duration_max_min": float(duration.max()),
    })
    tmp = events.assign(event_duration_min=duration).merge(room_alias, on="room", how="left")
    room_rows.append(
        tmp.groupby(["phase", "room_alias"], as_index=False)
        .agg(events=("shared_event_uid", "nunique"), median_duration_min=("event_duration_min", "median"),
             IQR_duration_min=("event_duration_min", lambda x: x.quantile(.75) - x.quantile(.25)))
    )

dataset_summary = pd.DataFrame(dataset_rows)
dataset_summary.to_csv(PAPER_TABLE_DIR / "dataset_phase_summary.csv", index=False)
room_support = pd.concat(room_rows, ignore_index=True)
room_support.to_csv(PAPER_TABLE_DIR / "dataset_anonymized_room_support.csv", index=False)

# Large, publication-ready event-duration distributions (one event = one observation).
plt.rcParams.update({"font.size": 12, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 11})
fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.3), constrained_layout=True)
phase_colors = {"heating": "#C44E52", "cooling": "#4C72B0"}
for ax, phase in zip(axes, ["heating", "cooling"]):
    events = all_events_for_split[all_events_for_split["phase"].eq(phase)].copy()
    duration_col = next(c for c in ["duration_min", "rampup_duration_min", "true_total_duration_min"] if c in events.columns)
    vals = pd.to_numeric(events[duration_col], errors="coerce").dropna()
    bins = min(30, max(8, int(np.sqrt(len(vals)))))
    ax.hist(vals, bins=bins, color=phase_colors[phase], alpha=.82, edgecolor="white")
    ax.axvline(vals.median(), color="black", linestyle="--", linewidth=2, label=f"median = {vals.median():.1f} min")
    ax.set(title=phase.capitalize(), xlabel="Event ramp-up duration (min)", ylabel="Number of events")
    ax.grid(axis="y", alpha=.22)
    ax.legend(frameon=False)
fig.savefig(PAPER_FIG_DIR / "event_duration_distributions.png", dpi=300, bbox_inches="tight")
fig.savefig(PAPER_FIG_DIR / "event_duration_distributions.pdf", bbox_inches="tight")
plt.show()

# Room support is shown with anonymous labels; internal room identifiers stay in the
# private map only.
support_plot = room_support.pivot(index="room_alias", columns="phase", values="events").fillna(0)
support_plot = support_plot.reindex(columns=[c for c in ["heating", "cooling"] if c in support_plot.columns])
ax = support_plot.plot(kind="bar", figsize=(13.5, 5.2), width=.78, color=[phase_colors.get(c, "gray") for c in support_plot.columns])
ax.set(xlabel="Anonymized room", ylabel="Number of detected events")
ax.grid(axis="y", alpha=.22)
ax.legend(title="Phase", frameon=False)
plt.tight_layout()
plt.savefig(PAPER_FIG_DIR / "event_support_by_anonymized_room.png", dpi=300, bbox_inches="tight")
plt.savefig(PAPER_FIG_DIR / "event_support_by_anonymized_room.pdf", bbox_inches="tight")
plt.show()

# Exact, machine-readable event definitions for the methods section.
event_rule_rows = []
for phase, events in all_events_for_split.groupby("phase", sort=False):
    for rule_col in ["start_rule", "end_rule"]:
        if rule_col in events.columns:
            for rule in sorted(events[rule_col].dropna().astype(str).unique()):
                event_rule_rows.append({"phase": phase, "rule_type": rule_col, "rule": rule})
event_definition_table = pd.DataFrame(event_rule_rows)
event_definition_table.to_csv(PAPER_TABLE_DIR / "event_definition_rules.csv", index=False)

# Fingerprint generated files to detect inconsistent extraction artifacts.
import hashlib
hash_rows = []
for phase, outputs in all_phase_outputs.items():
    for output_name, output_path in outputs.items():
        # ``outputs`` also contains integer counters such as
        # n_horizon_files and n_by_room_files.  Only path-like values belong
        # in the file-provenance table.
        if not isinstance(output_path, (str, os.PathLike)):
            continue
        path = Path(output_path)
        if not path.is_file():
            continue
        digest = hashlib.sha256()
        with open(path, "rb") as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b""):
                digest.update(chunk)
        hash_rows.append({
            "phase": phase, "output_name": output_name, "file_name": path.name,
            "size_bytes": int(path.stat().st_size), "sha256": digest.hexdigest(),
        })
dataset_file_hashes = pd.DataFrame(hash_rows)
dataset_file_hashes.to_csv(Path(OUT_DIR) / "dataset_output_file_hashes.csv", index=False)
dataset_file_hashes.to_csv(PAPER_TABLE_DIR / "dataset_output_file_hashes.csv", index=False)

print("Authoritative shared split map:", shared_split_path)
display(dataset_summary.round(2))
display(split_summary)


In [ ]:

# Feature manifest and column validation

framework_manifest = {
    'framework': NEW_FRAMEWORK_NAME,
    'output_dir': OUT_DIR,
    'event_definitions_changed': False,
    'authoritative_shared_split': str(Path(OUT_DIR) / 'shared_event_split_map.csv'),
    'model_facing_state': {
        'absolute_current_temperature_retained': True,
        'raw_room_temperature_lag_history_exported': False,
        'raw_room_temperature_lag_history_used_by_new_nn': False,
        'central_derivative_history_lags_min': DERIVATIVE_HISTORY_LAGS_MINUTES,
        'current_derivative_scheme': 'second_order_backward_causal',
        'past_derivative_scheme': 'central_at_past_centers_only',
        'future_rate_label_scheme': 'central_label_only',
        'nn_output_target': 'directed_rate = thermal_mode_sign * dT/dt',
        'thermal_mode_sign': {'heating': +1, 'cooling': -1},
        'room_identity_feature': None,
    },
    'remaining_duration_quality': {
        'score': 'clip((forward_progress - reverse_motion)/(forward_progress + reverse_motion), 0, 1)',
        'used_as_model_input': False,
        'intended_use': 'training-only weight for normalized remaining-duration loss',
        'test_metrics_weighted': False,
    },
}
framework_manifest_path = Path(OUT_DIR) / 'NEW_FRAMEWORK_extraction_manifest.json'
with open(framework_manifest_path, 'w') as f:
    json.dump(framework_manifest, f, indent=2, default=str)

# Verify required columns in saved phase-level files.
required_new = [
    'thermal_mode_sign', 'Phase_Target_T_C', 'Directed_Distance_to_Target_C',
    'Room_dTdt_now_bwd2_C_per_min', 'Room_d2Tdt2_now_bwd2_C_per_min2',
    'target_directed_rate_C_per_min', 'remaining_trajectory_quality_score',
]
for phase in ['heating', 'cooling']:
    p = Path(OUT_DIR) / phase / f'{phase}_full_rampup_5min_samples_LONG.csv'
    a = Path(OUT_DIR) / phase / f'{phase}_remaining_duration_anchors_5min.csv'
    if not p.exists() or not a.exists():
        raise RuntimeError(f'{phase}: expected final files are missing.')
    d = pd.read_csv(p, nrows=20)
    missing = [c for c in required_new if c not in d.columns]
    if missing:
        raise RuntimeError(f'{phase}: new LONG framework columns missing: {missing}')
    an = pd.read_csv(a, nrows=20)
    if 'remaining_trajectory_quality_score' not in an.columns:
        raise RuntimeError(f'{phase}: remaining-duration anchor score is missing.')

print('New extraction framework manifest:', framework_manifest_path)
print('Notebook 01 completed successfully.')


In [ ]:

# Split signature and consistency checks

# Fingerprint the saved split without changing its assignments.
# Downstream notebooks use the fingerprint to detect inconsistent inputs.

import hashlib

_split_path = Path(OUT_DIR) / 'shared_event_split_map.csv'
if not _split_path.exists():
    raise FileNotFoundError(_split_path)

_split_sig_df = pd.read_csv(_split_path)
_required_split_cols = ['phase','room','event_id','split','split_block']
_missing = [c for c in _required_split_cols if c not in _split_sig_df.columns]
if _missing:
    raise RuntimeError(f'Authoritative split file missing columns: {_missing}')

_split_sig_df['room'] = _split_sig_df['room'].astype(str).str.strip()
def _norm_sig_event_id(x):
    if pd.isna(x):
        raise RuntimeError('Missing event_id in authoritative split.')
    s=str(x).strip()
    if re.fullmatch(r'[-+]?\d+\.0+',s):
        s=str(int(float(s)))
    return s

_split_sig_df['event_id'] = _split_sig_df['event_id'].map(_norm_sig_event_id)
_split_sig_df = (
    _split_sig_df[_required_split_cols]
    .drop_duplicates()
    .sort_values(['phase','room','event_id'])
    .reset_index(drop=True)
)

_canonical = _split_sig_df.to_csv(index=False, lineterminator='\n')
_split_sha256 = hashlib.sha256(_canonical.encode('utf-8')).hexdigest()

_split_counts = (
    _split_sig_df.groupby(['phase','split'],as_index=False)
    .size().rename(columns={'size':'events'})
)

_split_signature = {
    'sha256': _split_sha256,
    'canonical_columns': _required_split_cols,
    'n_unique_events': int(len(_split_sig_df)),
    'counts_by_phase_split': _split_counts.to_dict(orient='records'),
    'policy': 'one authoritative chronological event split for ALL downstream models; never altered for baseline support',
}

with open(Path(OUT_DIR)/'shared_event_split_signature.json','w') as f:
    json.dump(_split_signature,f,indent=2)

print('Authoritative shared split SHA256:', _split_sha256)
display(_split_counts)
